# الفرز الآلي لصور كاميرات الحياة البرية
## المرحلة الأولى: البيانات والتصنيف والتقييم

المشروع الخامس — AI251P Applied Computer Vision

يدرس المشروع تصنيف الإطارات إلى فارغة أو محتوية على حيوان، وتصنيف الأنواع
باستخدام القصاصات البشرية من CCT20. نقارن CNN من الصفر مع ResNet-50
بتجميد الشبكة الأساسية وتدريبها الكامل، مع فصل مواقع الكاميرات بين المجموعات.

يتكون العرض من أربعة أقسام: البيانات، CNN، التعلم بالنقل، والمقارنة والتقرير.
تعرض النتائج بعد تنفيذ الخلايا على البيانات الفعلية. تتطلب مراجعة التسميات
وملاحظات الفريق إدخالًا بشريًا، وتظل حالة التقرير غير مكتملة حتى توفر أدلتها.
التشغيل الكامل يحتاج GPU ومكتبات requirements.in وزوج torch/torchvision متوافق.
في Colab تثبت خلية البيئة المكتبات، ثم تنفذ الأقسام بالتسلسل في الجلسة نفسها.
ملف Python المصاحب يستخدم خلايا # %% ويمكن فتحه في محرر يدعم Python Interactive؛
التدقيق التفاعلي يتم في الدفتر، ولا يتجاوز تشغيل السكربت متطلبات التدقيق.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ImageHash', 'pandas', 'scipy', 'matplotlib', 'Pillow', 'PyYAML', 'ipywidgets', 'scikit-learn', 'opencv-python-headless', 'tqdm', 'wandb', 'transformers==4.49.0'])

In [ ]:
from pathlib import Path
import json
presentation_root = Path.cwd() if Path.cwd().name == 'CameraTrap_Phase1_Team' else Path.cwd()/'CameraTrap_Phase1_Team'
presentation_sources = presentation_root/'notebooks'
presentation_sources.mkdir(parents=True, exist_ok=True)
presentation_cells = json.loads('[{"cell_type": "markdown", "source": "# الفرز الآلي لصور كاميرات الحياة البرية\\n## المرحلة الأولى: البيانات والتصنيف والتقييم\\n\\nالمشروع الخامس — AI251P Applied Computer Vision\\n\\nيدرس المشروع تصنيف الإطارات إلى فارغة أو محتوية على حيوان، وتصنيف الأنواع\\nباستخدام القصاصات البشرية من CCT20. نقارن CNN من الصفر مع ResNet-50\\nبتجميد الشبكة الأساسية وتدريبها الكامل، مع فصل مواقع الكاميرات بين المجموعات.\\n\\nيتكون العرض من أربعة أقسام: البيانات، CNN، التعلم بالنقل، والمقارنة والتقرير.\\nتعرض النتائج بعد تنفيذ الخلايا على البيانات الفعلية. تتطلب مراجعة التسميات\\nوملاحظات الفريق إدخالًا بشريًا، وتظل حالة التقرير غير مكتملة حتى توفر أدلتها.\\nالتشغيل الكامل يحتاج GPU ومكتبات requirements.in وزوج torch/torchvision متوافق.\\nفي Colab تثبت خلية البيئة المكتبات، ثم تنفذ الأقسام بالتسلسل في الجلسة نفسها.\\nملف Python المصاحب يستخدم خلايا # %% ويمكن فتحه في محرر يدعم Python Interactive؛\\nالتدقيق التفاعلي يتم في الدفتر، ولا يتجاوز تشغيل السكربت متطلبات التدقيق.\\n", "metadata": {}}, {"cell_type": "code", "source": "import subprocess, sys\\nsubprocess.check_call([sys.executable, \'-m\', \'pip\', \'install\', \'ImageHash\', \'pandas\', \'scipy\', \'matplotlib\', \'Pillow\', \'PyYAML\', \'ipywidgets\', \'scikit-learn\', \'opencv-python-headless\', \'tqdm\', \'wandb\', \'transformers==4.49.0\'])", "metadata": {}}, {"cell_type": "markdown", "source": "# 1. البيانات والتدقيق — M1", "metadata": {}}, {"cell_type": "code", "source": "from pathlib import Path\\nimport os, sys, subprocess\\nimport matplotlib.pyplot as plt\\nfrom IPython.display import display, Markdown\\n\\"\\"\\"Download official CCT20 metadata; no predicted boxes are used as labels.\\"\\"\\"\\nimport collections\\nimport hashlib\\nimport html\\nimport io\\nimport json\\nfrom pathlib import Path\\nimport random\\nimport tarfile\\nimport time\\nimport urllib.parse\\nimport urllib.request\\nimport numpy as np\\nimport pandas as pd\\nfrom PIL import Image, ImageDraw\\nimport imagehash\\n\\n\\nMETA_URL = \'https://storage.googleapis.com/public-datasets-lila/caltechcameratraps/eccv_18_annotations.tar.gz\'\\nIMAGE_BASES = [\\n    \'https://storage.googleapis.com/public-datasets-lila/caltech-unzipped/cct_images/\',\\n    \'https://lilawildlife.blob.core.windows.net/lila-wildlife/caltech-unzipped/cct_images/\']\\nNON_ANIMALS = {\'empty\', \'car\', \'vehicle\', \'person\', \'human\', \'people\'}\\n\\n\\n\\nSEED = 42\\nMAX_IMAGES = 6000\\nAUDIT_IMAGES = 100\\nPROJECT = Path.cwd() if Path.cwd().name == \'CameraTrap_Phase1_Team\' else Path.cwd()/\'CameraTrap_Phase1_Team\'\\nPROJECT.mkdir(exist_ok=True)\\nos.chdir(PROJECT)\\nDATA = Path(\'data/cct20_v1\')\\nDATA.mkdir(parents=True, exist_ok=True)\\nPath(\'runs/data\').mkdir(parents=True, exist_ok=True)\\nPath(\'handoff\').mkdir(exist_ok=True)\\n\\ndef read_json(path):\\n    return json.loads(Path(path).read_text())\\ndef write_json(path, value):\\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False))\\ndef digest(path):\\n    h = hashlib.sha256()\\n    with open(path, \'rb\') as f:\\n        for block in iter(lambda: f.read(1024*1024), b\'\'): h.update(block)\\n    return h.hexdigest()\\n\\ncfg = {\'seed\': SEED, \'max_images\': MAX_IMAGES, \'max_image_side\': 1024,\\n       \'audit_events\': AUDIT_IMAGES, \'data_dir\': str(DATA), \'output_dir\': \'runs/data\'}\\nassert not Path(\'handoff/dataset_release.json\').exists(), \'هذا الإصدار صدر بالفعل. استخدم نتائجه أو ابدأ مجلدًا جديدًا؛ لا تعدله بعد تدريب زملائك.\'\\nprint(\'Project folder:\', PROJECT)\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 2. قراءة التسميات", "metadata": {}}, {"cell_type": "code", "source": "def fetch(url, path):\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    if path.exists() and path.stat().st_size:\\n        return\\n    tmp = path.with_suffix(path.suffix+\'.part\')\\n    for attempt in range(3):\\n        try:\\n            with urllib.request.urlopen(url, timeout=45) as r, tmp.open(\'wb\') as f:\\n                while chunk := r.read(1024*1024):\\n                    f.write(chunk)\\n            tmp.replace(path)\\n            return\\n        except Exception:\\n            tmp.unlink(missing_ok=True)\\n            if attempt == 2:\\n                raise\\n            time.sleep(attempt+1)\\n\\ndef read_metadata(root):\\n    archive = root/\'metadata\'/\'eccv_18_annotations.tar.gz\'\\n    fetch(META_URL, archive)\\n    images, annotations, categories = {}, {}, {}\\n    with tarfile.open(archive) as tar:\\n        for member in tar.getmembers():\\n            if not member.name.endswith(\'.json\') or not member.isfile():\\n                continue\\n            # Read members in memory: never extract untrusted paths.\\n            d = json.load(tar.extractfile(member))\\n            for c in d[\'categories\']:\\n                if c[\'id\'] in categories and categories[c[\'id\']] != c[\'name\']:\\n                    raise ValueError(\'Conflicting category maps\')\\n                categories[c[\'id\']] = c[\'name\']\\n            for im in d[\'images\']:\\n                if im[\'id\'] in images and images[im[\'id\']] != im:\\n                    raise ValueError(\'Conflicting image metadata\')\\n                images[im[\'id\']] = im\\n            for a in d[\'annotations\']:\\n                key = (a[\'image_id\'], a[\'category_id\'], tuple(a.get(\'bbox\', [])))\\n                annotations[key] = a\\n    by_image = collections.defaultdict(list)\\n    for a in annotations.values():\\n        by_image[a[\'image_id\']].append(a)\\n    rows, excluded = [], []\\n    for ident, im in sorted(images.items()):\\n        aa = by_image[ident]\\n        labels = sorted({categories[a[\'category_id\']] for a in aa})\\n        if not labels or any(c in NON_ANIMALS-{\'empty\'} for c in labels):\\n            excluded.append({\'id\': ident, \'reason\': \'non-animal-category-or-unlabelled\', \'labels\': labels})\\n            continue\\n        animal_labels = [s for s in labels if s not in NON_ANIMALS]\\n        if \'empty\' in labels and animal_labels:\\n            excluded.append({\'id\': ident, \'reason\': \'conflicting-empty-label\'})\\n            continue\\n        boxes = []\\n        incomplete = False\\n        for a in aa:\\n            name = categories[a[\'category_id\']]\\n            if name in NON_ANIMALS:\\n                continue\\n            if \'bbox\' not in a:\\n                incomplete = True\\n                continue\\n            x, y, w, h = map(float, a[\'bbox\'])\\n            W, H = im[\'width\'], im[\'height\']\\n            x1, y1 = max(0.,x/W), max(0.,y/H)\\n            x2, y2 = min(1.,(x+w)/W), min(1.,(y+h)/H)\\n            if x2 <= x1 or y2 <= y1:\\n                incomplete = True\\n                continue\\n            boxes.append({\'species\': name, \'xyxy\': [x1,y1,x2,y2], \'annotation_source\': \'human_CCT20\'})\\n        rows.append(dict(id=ident, file_name=Path(im[\'file_name\']).name,\\n                         camera=str(im[\'location\']), event=str(im.get(\'seq_id\') or ident),\\n                         timestamp=im.get(\'date_captured\',im.get(\'datetime\')),\\n                         original_width=im[\'width\'], original_height=im[\'height\'],\\n                         seq_num_frames=im.get(\'seq_num_frames\'), labels=animal_labels,\\n                         animal=int(bool(animal_labels)), boxes=boxes,\\n                         incomplete_boxes=incomplete))\\n    summary = dict(source=META_URL, sha256=digest(archive), total_images=len(images),\\n                   unique_annotations=len(annotations), human_boxes=sum(\'bbox\' in a for a in annotations.values()),\\n                   cameras=len({str(i[\'location\']) for i in images.values()}),\\n                   categories=categories, eligible_frames=len(rows), excluded_frames=len(excluded),\\n                   empty_frames=sum(r[\'animal\']==0 for r in rows),\\n                   note=\'CCT20 is curated and is not the full 243k-image dataset. Percent empty is recomputed.\')\\n    write_json(root/\'metadata_summary.json\', summary)\\n    write_json(root/\'metadata_exclusions.json\', excluded)\\n    return rows\\n\\nrows = read_metadata(DATA)\\nprint(\\"Eligible frames:\\", len(rows))\\ndisplay(pd.DataFrame(rows)[[\\"id\\", \\"camera\\", \\"event\\", \\"animal\\", \\"labels\\"]].head())", "metadata": {}}, {"cell_type": "markdown", "source": "## 3. Stratified Group Split", "metadata": {}}, {"cell_type": "code", "source": "from sklearn.model_selection import StratifiedGroupKFold\\n\\ndef assign_sites(rows, seed):\\n    # A multi-species frame gets one joint stratum, not multiple independently split copies.\\n    y=np.array([\'|\'.join(sorted(r[\'labels\'])) or \'empty\' for r in rows])\\n    groups=np.array([r[\'camera\'] for r in rows])\\n    def first_fold(indices, folds):\\n        splitter=StratifiedGroupKFold(n_splits=folds,shuffle=True,random_state=seed)\\n        a,b=next(splitter.split(np.zeros(len(indices)),y[indices],groups[indices]))\\n        return indices[a],indices[b]\\n    remaining,test=first_fold(np.arange(len(rows)),5)\\n    train,validation=first_fold(remaining,4)\\n    select,calibrate=first_fold(validation,2)\\n    site_map={}\\n    for name,ids in [(\'train\',train),(\'val_select\',select),(\'val_calibrate\',calibrate),(\'test\',test)]:\\n        for camera in set(groups[ids]):\\n            assert camera not in site_map\\n            site_map[camera]=name\\n    return site_map\\n\\ndef assert_disjoint(rows):\\n    for field in [\'id\', \'camera\', \'event\', \'pixel_sha256\']:\\n        seen = {}\\n        for r in rows:\\n            if field not in r:\\n                continue\\n            key = r[field]\\n            if key in seen and seen[key] != r[\'split\']:\\n                raise AssertionError(f\'{field} leakage across splits\')\\n            seen[key] = r[\'split\']\\n\\nsite_map=assign_sites(rows,SEED)\\nfor r in rows:r[\'split\']=site_map[r[\'camera\']]\\nassert_disjoint(rows)\\nwrite_json(DATA/\'site_split.json\',site_map)\\ndisplay(pd.DataFrame(rows).groupby(\'split\').agg(images=(\'id\',\'count\'),cameras=(\'camera\',\'nunique\')))\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 4. اختيار العينة مع الحفاظ على bursts", "metadata": {}}, {"cell_type": "code", "source": "def sample_events(rows, limit, seed):\\n    \\"\\"\\"Sample whole bursts within each location, proportionally to eligible frame counts.\\"\\"\\"\\n    if not limit or limit >= len(rows):\\n        return rows\\n    rng = random.Random(seed)\\n    selected = []\\n    for camera in sorted({r[\'camera\'] for r in rows}):\\n        rr = [r for r in rows if r[\'camera\']==camera]\\n        groups = collections.defaultdict(list)\\n        for r in rr:\\n            groups[r[\'event\']].append(r)\\n        events = sorted(groups)\\n        rng.shuffle(events)\\n        budget = max(3, round(limit*len(rr)/len(rows)))\\n        chosen = []\\n        for event in events:\\n            chosen.extend(groups[event])\\n            if len(chosen) >= budget:\\n                break\\n        selected.extend(chosen)\\n    return selected\\n\\nchosen = sample_events(rows, MAX_IMAGES, SEED)\\nwrite_json(DATA/\'selected_ids.json\', [r[\'id\'] for r in chosen])\\nprint(\'Selected images:\', len(chosen))\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 5. تنزيل الصور وفحص الملفات", "metadata": {}}, {"cell_type": "code", "source": "def download_one(row, root, max_side):\\n    path = root/\'images\'/(row[\'id\']+\'.jpg\')\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    if not path.exists():\\n        err = None\\n        for base in IMAGE_BASES:\\n            try:\\n                with urllib.request.urlopen(base+urllib.parse.quote(row[\'file_name\']),timeout=40) as r:\\n                    raw = r.read()\\n                im = Image.open(io.BytesIO(raw)).convert(\'RGB\')\\n                im.thumbnail((max_side,max_side), Image.Resampling.LANCZOS)\\n                tmp = path.with_suffix(\'.tmp\')\\n                im.save(tmp, format=\'JPEG\',quality=95)\\n                tmp.replace(path)\\n                break\\n            except Exception as e:\\n                err = str(e)\\n        else:\\n            return None, {\'id\':row[\'id\'],\'event\':row[\'event\'],\'reason\':err}\\n    try:\\n        with Image.open(path) as src:\\n            im = src.convert(\'RGB\'); im.load()\\n            small = np.array(im.resize((64,64)),dtype=float)\\n            row = dict(row, path=str(path.resolve()),width=im.width,height=im.height,\\n                       processed_file_md5=hashlib.md5(path.read_bytes()).hexdigest(),\\n                       pixel_sha256=hashlib.sha256(im.tobytes()).hexdigest(),\\n                       phash=str(imagehash.phash(im)),\\n                       grayscale_proxy=bool(np.mean(np.max(small,2)-np.min(small,2))<6))\\n            return row, None\\n    except Exception as e:\\n        return None, {\'id\':row[\'id\'],\'event\':row[\'event\'],\'reason\':str(e)}\\n\\nfrom tqdm.auto import tqdm\\ngood, failures = [], []\\nfor row in tqdm(chosen):\\n    image_row, error = download_one(row, DATA, cfg[\'max_image_side\'])\\n    if image_row is not None:\\n        image_row[\'path\'] = str(DATA/\'images\'/(image_row[\'id\']+\'.jpg\'))  # portable relative path\\n        good.append(image_row)\\n    if error is not None: failures.append(error)\\nwrite_json(DATA/\'download_failures.json\', failures)\\nprint(\'Readable:\', len(good), \'| Failed:\', len(failures))\\nassert len(failures) <= 0.02*len(chosen), \'فشل أكثر من 2%؛ أعد خلية التنزيل قبل المتابعة\'\\nfailed_events = {e[\'event\'] for e in failures}\\ngood = sorted([r for r in good if r[\'event\'] not in failed_events], key=lambda r:r[\'id\'])\\n", "metadata": {}}, {"cell_type": "markdown", "source": "### فحص الصور والألوان والجودة", "metadata": {}}, {"cell_type": "code", "source": "import cv2\\ninventory=[]\\nfor r in good:\\n    im=Image.open(r[\'path\']);rgb=np.asarray(im.convert(\'RGB\'))\\n    small=cv2.resize(rgb,(224,224),interpolation=cv2.INTER_AREA)\\n    blur=float(cv2.Laplacian(cv2.cvtColor(small,cv2.COLOR_RGB2GRAY),cv2.CV_64F).var())\\n    inventory.append({\'id\':r[\'id\'],\'split\':r[\'split\'],\'mode\':im.mode,\'width\':im.width,\'height\':im.height,\\n      \'brightness\':float(small.mean()),\'blur_score\':blur,\'path\':r[\'path\']})\\ninventory=pd.DataFrame(inventory);inventory.to_csv(DATA/\'inventory.csv\',index=False)\\ndisplay(inventory.describe(include=\'all\'))\\nlowest=inventory.query(\\"split == \'train\'\\").nsmallest(8,\'blur_score\').copy()\\nlowest[\'reviewer\']=\'\';lowest[\'decision\']=\'\';lowest[\'notes\']=\'\'\\nif not (DATA/\'blur_review.csv\').exists():lowest.to_csv(DATA/\'blur_review.csv\',index=False)\\nfor r in lowest.to_dict(\'records\'):\\n    print(r[\'id\'],\'blur:\',round(r[\'blur_score\'],2));im=Image.open(r[\'path\']);im.thumbnail((420,280));display(im)\\nprint(\'أكمل blur_review.csv: reviewer = اسمك، decision = keep أو flag، notes = السبب. flag للتوثيق فقط؛ لا حذف تلقائي.\')\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 6. إزالة التكرار وفحص التسرب", "metadata": {}}, {"cell_type": "code", "source": "def near_duplicate_quarantine(rows, distance=2):\\n    # 64-bit pHash: four chunks guarantee a candidate bucket if Hamming distance <=2.\\n    buckets = collections.defaultdict(list)\\n    bad, pairs = set(), []\\n    for r in rows:\\n        h = int(r[\'phash\'],16)\\n        candidates = {}\\n        for part in range(4):\\n            for old in buckets[(part,(h>>(part*16)) & 65535)]:\\n                candidates[old[\'id\']] = old\\n        for old in candidates.values():\\n            if old[\'split\']==r[\'split\']:\\n                continue\\n            dist = (h ^ int(old[\'phash\'],16)).bit_count()\\n            if dist <= distance:\\n                bad.update([r[\'event\'],old[\'event\']])\\n                pairs.append({\'id_a\':old[\'id\'],\'id_b\':r[\'id\'],\'distance\':dist})\\n        for part in range(4):\\n            buckets[(part,(h>>(part*16)) & 65535)].append(r)\\n    return [r for r in rows if r[\'event\'] not in bad], pairs, sorted(bad)", "metadata": {}}, {"cell_type": "code", "source": "hashes=collections.defaultdict(list)\\nfor r in good:hashes[r[\'pixel_sha256\']].append(r)\\nexcluded_events=set(); duplicate_ids=[];conflicting_duplicate_groups=[]\\nfor group in hashes.values():\\n    if len(group)>1:\\n        if len({tuple(r[\'labels\']) for r in group})>1:\\n            conflicting_duplicate_groups.append([r[\'id\'] for r in group])\\n            excluded_events.update(r[\'event\'] for r in group)\\n        elif len({r[\'split\'] for r in group})>1:\\n            excluded_events.update(r[\'event\'] for r in group)\\n        else:duplicate_ids.extend(r[\'id\'] for r in group[1:])\\ncleaned=[r for r in good if r[\'event\'] not in excluded_events and r[\'id\'] not in set(duplicate_ids)]\\ncleaned,pairs,near_events=near_duplicate_quarantine(cleaned)\\nassert_disjoint(cleaned)\\nfor split in [\'train\',\'val_select\',\'val_calibrate\',\'test\']:\\n    if {r[\'animal\'] for r in cleaned if r[\'split\']==split}!={0,1}:\\n        raise ValueError(f\'{split} lacks empty/animal examples after sampling. Increase max_images; do not adjust test based on results.\')\\nlabels=sorted({b[\'species\'] for r in cleaned if r[\'split\']==\'train\' for b in r[\'boxes\']})\\nif len(labels)<2:raise ValueError(\'Need at least two training species\')\\nwrite_json(DATA/\'species.json\',labels)\\nwrite_json(DATA/\'manifest.json\',cleaned)\\nwrite_json(DATA/\'preparation_config.json\',cfg)\\nwrite_json(DATA/\'cleaning_report.json\',dict(selected=len(chosen),downloaded=len(good),kept=len(cleaned),\\n    failures=len(failures),excluded_failed_events=len(failed_events),duplicate_ids=duplicate_ids,\\n    exact_cross_split_events=sorted(excluded_events),near_duplicate_pairs=pairs,near_quarantined_events=near_events,\\n    conflicting_duplicate_label_groups=conflicting_duplicate_groups,\\n    near_policy=\'Conservative pHash <=2 across partitions: both events quarantined; not asserted to be true duplicates.\',\\n    corrupted_or_failed_files=failures,grayscale_proxy=\'RGB channel difference <6; proxy only, not verified infrared ground truth\'))\\n\\nprint(\'Clean frames:\', len(cleaned), \'| Training species:\', labels)", "metadata": {}}, {"cell_type": "markdown", "source": "## 7. عرض الصور وتوزيع الفئات", "metadata": {}}, {"cell_type": "code", "source": "def gallery(path, rows, draw_boxes=False, titles=None):\\n    path=Path(path);thumbs=path.parent/(path.stem+\'_images\');thumbs.mkdir(exist_ok=True)\\n    cards=[]\\n    for i,r in enumerate(rows):\\n        im=Image.open(r[\'path\']).convert(\'RGB\')\\n        if draw_boxes:\\n            draw=ImageDraw.Draw(im)\\n            for b in r[\'boxes\']:\\n                x1,y1,x2,y2=b[\'xyxy\'];box=[x1*im.width,y1*im.height,x2*im.width,y2*im.height]\\n                draw.rectangle(box,outline=\'lime\',width=3);draw.text((box[0],box[1]),b[\'species\'],fill=\'yellow\')\\n        im.thumbnail((480,360));name=f\'{i:04d}.jpg\';im.save(thumbs/name)\\n        title=titles[i] if titles else f\'{r[\\"id\\"]} | {r[\\"split\\"]} | {r[\\"labels\\"] or [\\"empty\\"]}\'\\n        cards.append(f\'<article><img src=\\"{thumbs.name}/{name}\\"><p>{html.escape(str(title))}</p></article>\')\\n    path.write_text(\'<!doctype html><meta charset=\\"utf-8\\"><title>Local review gallery</title><style>body{font:14px sans-serif;background:#132127;color:white}main{display:grid;grid-template-columns:repeat(auto-fit,minmax(340px,1fr));gap:16px}article{background:#22343b;padding:10px}img{max-width:100%}p{overflow-wrap:anywhere}</style><h1>Private local review gallery</h1><p>Review before sharing; exclude or blur any identifiable people. Do not redistribute dataset images.</p><main>\'+\'\'.join(cards)+\'</main>\',encoding=\'utf-8\')\\n\\ndef audit_templates(cfg, rows):\\n    root=Path(cfg[\'data_dir\'])\\n    # A random event sample with one random frame per event. Do not claim a random-frame error rate.\\n    rng=random.Random(cfg[\'seed\'])\\n    events=collections.defaultdict(list)\\n    for r in rows:\\n        if r[\'split\']==\'train\':events[r[\'event\']].append(r)\\n    selected=rng.sample(sorted(events),min(cfg[\'audit_events\'],len(events)))\\n    audit=[]\\n    for event in selected:\\n        r=rng.choice(events[event]);audit.append(dict(id=r[\'id\'],event=event,split=r[\'split\'],\\n          expected_species=\';\'.join(r[\'labels\']) or \'empty\',label_correct=\'\',boxes_correct=\'\',\\n          identifiable_person=\'\',reviewer=\'\',notes=\'\'))\\n    pd.DataFrame(audit).to_csv(root/\'label_audit.csv\',index=False)\\n    gallery(root/\'audit_gallery.html\',[next(r for r in rows if r[\'id\']==a[\'id\']) for a in audit],draw_boxes=True)\\n    # Site-wide label balance and crop support, without exposing exact coordinates.\\n    count_rows=[dict(split=r[\'split\'],label=s) for r in rows for s in (r[\'labels\'] or [\'empty\'])]\\n    pd.crosstab(pd.DataFrame(count_rows).label,pd.DataFrame(count_rows).split).to_csv(root/\'class_counts.csv\')\\n    crop_rows=[dict(split=r[\'split\'],label=b[\'species\']) for r in rows for b in r[\'boxes\']]\\n    pd.crosstab(pd.DataFrame(crop_rows).label,pd.DataFrame(crop_rows).split).to_csv(root/\'crop_counts.csv\')\\n    gallery(root/\'annotation_gallery.html\',rng.sample(rows,min(48,len(rows))),draw_boxes=True)\\n\\nif not (DATA/\'label_audit.csv\').exists():\\n    audit_templates(cfg, cleaned)\\ncounts = pd.read_csv(DATA/\'class_counts.csv\', index_col=0)\\ndisplay(counts)\\ncounts.plot.bar(stacked=True, figsize=(12,4))\\nplt.ylabel(\'Label occurrences\'); plt.tight_layout(); plt.show()\\nfor row in [r for r in cleaned if r[\'split\']==\'train\'][:4]:\\n    print(row[\'id\'], row[\'labels\'] or [\'empty\'])\\n    im=Image.open(row[\'path\']); im.thumbnail((600,400)); display(im)\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 8. تدقيق بشري", "metadata": {}}, {"cell_type": "code", "source": "def audit_widget(cfg):\\n    import ipywidgets as w\\n    from IPython.display import display,clear_output\\n    from PIL import Image,ImageDraw\\n    root=Path(cfg[\'data_dir\']);path=root/\'label_audit.csv\'\\n    df=pd.read_csv(path).fillna(\'\');rows={r[\'id\']:r for r in read_json(root/\'manifest.json\')}\\n    index=w.IntSlider(value=0,min=0,max=len(df)-1,description=\'Sample\')\\n    correct=w.Dropdown(options=[\'\',\'yes\',\'no\'],description=\'Label correct\')\\n    boxes=w.Dropdown(options=[\'\',\'yes\',\'no\',\'not applicable\'],description=\'Boxes correct\')\\n    person=w.Dropdown(options=[\'\',\'yes\',\'no\'],description=\'Person visible\')\\n    reviewer=w.Text(description=\'Reviewer\');notes=w.Textarea(description=\'Notes\')\\n    save=w.Button(description=\'Save and next\',button_style=\'success\');output=w.Output();status=w.Output()\\n    def show(change=None):\\n        row=df.iloc[index.value];r=rows[row.id]\\n        correct.value=str(row.label_correct);boxes.value=str(row.boxes_correct);person.value=str(row.identifiable_person)\\n        if row.reviewer:reviewer.value=str(row.reviewer)\\n        notes.value=str(row.notes)\\n        with output:\\n            clear_output(wait=True);print(\'Expected:\',row.expected_species,\'| sample:\',row.id)\\n            im=Image.open(r[\'path\']).convert(\'RGB\');draw=ImageDraw.Draw(im)\\n            for b in r[\'boxes\']:\\n                x1,y1,x2,y2=b[\'xyxy\'];draw.rectangle([x1*im.width,y1*im.height,x2*im.width,y2*im.height],outline=\'lime\',width=3)\\n            im.thumbnail((900,600));display(im)\\n    def persist(_):\\n        with status:\\n            clear_output()\\n            if not reviewer.value.strip() or not correct.value or not person.value or not boxes.value:\\n                print(\'Fill every decision and the reviewer name before saving.\');return\\n        i=index.value\\n        for key,value in [(\'label_correct\',correct.value),(\'boxes_correct\',boxes.value),(\'identifiable_person\',person.value),\\n                          (\'reviewer\',reviewer.value.strip()),(\'notes\',notes.value)]:df.loc[i,key]=value\\n        df.to_csv(path,index=False)\\n        with status:print(\'Saved. Corrections/privacy flags need resolution before final training/report.\')\\n        if i<len(df)-1:index.value=i+1\\n    index.observe(show,names=\'value\');save.on_click(persist)\\n    display(w.VBox([index,output,correct,boxes,person,reviewer,notes,save,status]));show()\\n\\naudit_widget(cfg)", "metadata": {}}, {"cell_type": "markdown", "source": "## 9. معالجة مشاكل التدقيق وحساب حد الخطأ", "metadata": {}}, {"cell_type": "code", "source": "from scipy.stats import beta\\n\\ndef audit_report(cfg):\\n    root=Path(cfg[\'data_dir\']);df=pd.read_csv(root/\'label_audit.csv\').fillna(\'\')\\n    valid=(df.label_correct.astype(str).str.lower().isin([\'yes\',\'no\']) &\\n           df.boxes_correct.astype(str).str.lower().isin([\'yes\',\'no\',\'not applicable\']) &\\n           df.identifiable_person.astype(str).str.lower().isin([\'yes\',\'no\']) &\\n           df.reviewer.astype(str).str.strip().ne(\'\'))\\n    reviewed=df[valid];n=len(reviewed);errors=int((reviewed.label_correct.astype(str).str.lower()==\'no\').sum())\\n    rows=read_json(root/\'manifest.json\');present_events={r[\'event\'] for r in rows}\\n    flagged=reviewed[(reviewed.label_correct.astype(str).str.lower()==\'no\') |\\n                     (reviewed.boxes_correct.astype(str).str.lower()==\'no\') |\\n                     (reviewed.identifiable_person.astype(str).str.lower()==\'yes\')]\\n    upper=float(beta.ppf(.95,errors+1,n-errors)) if n and errors<n else (1. if n else None)\\n    result=dict(status=\'complete\' if n==len(df) else \'pending\',requested=len(df),reviewed=n,label_errors=errors,\\n      estimated_error_rate=errors/n if n else None,one_sided_95_upper_bound=upper,\\n      sampling_unit=\'random train event, then one random eligible frame; not all dataset frames\',\\n      limitations=\'Binomial interval assumes independent sampled events. Same-camera correlation may make it optimistic.\',\\n      no_review_is_not_zero_errors=True,\\n      privacy_flags=int(((reviewed.identifiable_person.astype(str).str.lower()==\'yes\') & reviewed.event.isin(present_events)).sum()),\\n      unresolved_flagged_events=len(set(flagged.event)&present_events))\\n    write_json(root/\'audit_report.json\',result);return result\\n\\ndef apply_audit(cfg):\\n    \\"\\"\\"Conservatively quarantine train events with human-confirmed problems; no guessed relabelling.\\"\\"\\"\\n    root=Path(cfg[\'data_dir\']);out=Path(cfg[\'output_dir\'])\\n    df=pd.read_csv(root/\'label_audit.csv\').fillna(\'\')\\n    reviewed=df[df.reviewer.astype(str).str.strip().ne(\'\')]\\n    flagged=reviewed[(reviewed.label_correct.astype(str).str.lower()==\'no\') |\\n                     (reviewed.boxes_correct.astype(str).str.lower()==\'no\') |\\n                     (reviewed.identifiable_person.astype(str).str.lower()==\'yes\')]\\n    bad=set(flagged.event);rows=read_json(root/\'manifest.json\')\\n    if not (bad & {r[\'event\'] for r in rows}):\\n        print(\'No unresolved human-flagged events to quarantine.\');return\\n    if list(out.glob(\'*/*/best.pt\')):\\n        raise RuntimeError(\'Checkpoints already exist. Set a NEW output_dir in config before applying training-data corrections.\')\\n    if any(r[\'split\']!=\'train\' and r[\'event\'] in bad for r in rows):\\n        raise ValueError(\'Only train audit changes are allowed; held-out protocol is immutable.\')\\n    cleaned=[r for r in rows if r[\'event\'] not in bad]\\n    labels=sorted({b[\'species\'] for r in cleaned if r[\'split\']==\'train\' for b in r[\'boxes\']})\\n    if len(labels)<2 or {r[\'animal\'] for r in cleaned if r[\'split\']==\'train\'}!={0,1}:\\n        raise ValueError(\'Audit exclusions remove required training support; collect more training events.\')\\n    write_json(root/\'audit_quarantine.json\',dict(events=sorted(bad),frames_removed=len(rows)-len(cleaned),\\n        policy=\'Human-flagged events excluded. No automatic corrected species labels.\'))\\n    write_json(root/\'manifest.json\',cleaned);write_json(root/\'species.json\',labels);assert_disjoint(cleaned)\\n    for name,values in [(\'class_counts\',[dict(split=r[\'split\'],label=s) for r in cleaned for s in (r[\'labels\'] or [\'empty\'])]),\\n                        (\'crop_counts\',[dict(split=r[\'split\'],label=b[\'species\']) for r in cleaned for b in r[\'boxes\']])]:\\n        frame=pd.DataFrame(values);pd.crosstab(frame.label,frame.split).to_csv(root/(name+\'.csv\'))\\n    print(\'Quarantined\',len(rows)-len(cleaned),\'training frames across\',len(bad),\'flagged events.\')\\n\\naudit = audit_report(cfg)\\nassert audit[\'status\']==\'complete\', \'أكمل نموذج التدقيق أولاً\'\\napply_audit(cfg)\\naudit = audit_report(cfg)\\nassert audit[\'unresolved_flagged_events\']==0\\nrows = read_json(DATA/\'manifest.json\')\\nprint(audit)\\n", "metadata": {}}, {"cell_type": "markdown", "source": "### التحقق النهائي من Class Stratification", "metadata": {}}, {"cell_type": "code", "source": "splits=[\'train\',\'val_select\',\'val_calibrate\',\'test\']\\nall_labels=sorted({label for r in rows for label in (r[\'labels\'] or [\'empty\'])})\\nbalance=[]\\nfor label in all_labels:\\n    overall=sum(label in (r[\'labels\'] or [\'empty\']) for r in rows)/len(rows)\\n    for split in splits:\\n        subset=[r for r in rows if r[\'split\']==split]\\n        count=sum(label in (r[\'labels\'] or [\'empty\']) for r in subset)\\n        rate=count/len(subset) if subset else 0\\n        balance.append({\'label\':label,\'split\':split,\'count\':count,\'frames\':len(subset),\\n                        \'percent\':100*rate,\'overall_percent\':100*overall,\'difference_pp\':100*(rate-overall)})\\nbalance_report=pd.DataFrame(balance);balance_report.to_csv(DATA/\'class_balance_proportions.csv\',index=False)\\ndisplay(balance_report)\\nbalance_report.pivot(index=\'label\',columns=\'split\',values=\'percent\').plot.bar(figsize=(12,5))\\nplt.ylabel(\'Frames containing label (%)\');plt.tight_layout();plt.show()\\ncounts=pd.DataFrame([{\'species\':b[\'species\'],\'split\':r[\'split\']} for r in rows for b in r[\'boxes\']])\\nsupport=pd.crosstab(counts.species,counts.split).reindex(columns=splits,fill_value=0)\\nsupport.to_csv(DATA/\'species_crop_support.csv\');display(support)\\nprint(\'Missing class/split combinations:\');display(balance_report.query(\'count == 0\')[[\'label\',\'split\']])\\nassert_disjoint(rows)\\nprint(\'PASS: cameras, events and exact pixel duplicates are disjoint.\')\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 10. Preprocessor + Dataset + DataLoader", "metadata": {}}, {"cell_type": "code", "source": "import cv2, numpy as np, torch\\nfrom PIL import Image\\nfrom torchvision import transforms\\nfrom torch.utils.data import DataLoader\\nclass Preprocessor:\\n    \\"\\"\\"Lab 1 cell 72: prepare uint8 RGB, augment, finalize and CHW tensor.\\"\\"\\"\\n    def __init__(self,size):\\n        self.size=size\\n        self.mean=np.array([.485,.456,.406],np.float32)\\n        self.std=np.array([.229,.224,.225],np.float32)\\n    def prepare(self,bgr):\\n        rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB);h,w=rgb.shape[:2]\\n        interp=cv2.INTER_AREA if h>self.size or w>self.size else cv2.INTER_LINEAR\\n        return cv2.resize(rgb,(self.size,self.size),interpolation=interp)\\n    def finalize(self,rgb):return (rgb.astype(np.float32)/255.-self.mean)/self.std\\n\\ndef load_rgb_sample(row):\\n    im=cv2.imread(row[\'path\'])\\n    if im is None:raise ValueError(\'Unreadable image: \'+row[\'path\'])\\n    if row.get(\'crop\') is not None:\\n        x1,y1,x2,y2=row[\'crop\'];h,w=im.shape[:2];dx=(x2-x1)*.05;dy=(y2-y1)*.05\\n        left,top=max(0,int((x1-dx)*w)),max(0,int((y1-dy)*h))\\n        right,bottom=min(w,max(left+1,int((x2+dx)*w))),min(h,max(top+1,int((y2+dy)*h)))\\n        im=im[top:bottom,left:right].copy()\\n    return im\\n\\nclass FrameDataset(torch.utils.data.Dataset):\\n    \\"\\"\\"Lab 1 cell 104 adapted to frame and human-box crop tasks.\\"\\"\\"\\n    def __init__(self,samples,size,train=False):\\n        self.samples=samples;self.pre=Preprocessor(size)\\n        self.tf=transforms.Compose([transforms.RandomHorizontalFlip(),\\n                  transforms.ColorJitter(brightness=.15,contrast=.15)]) if train else None\\n    def __len__(self):return len(self.samples)\\n    def __getitem__(self,i):\\n        row=self.samples[i];img=self.pre.prepare(load_rgb_sample(row))\\n        if self.tf:img=np.array(self.tf(Image.fromarray(img)))\\n        x=torch.from_numpy(np.ascontiguousarray(self.pre.finalize(img))).permute(2,0,1).float()\\n        return x,int(row[\'label\']),i\\n\\ntrain_rows=[dict(r,label=r[\'animal\'],crop=None) for r in rows if r[\'split\']==\'train\']\\ntrain_dataset=FrameDataset(train_rows,224,train=True)\\neval_dataset=FrameDataset(train_rows,224,train=False)\\ntrain_loader=DataLoader(train_dataset,batch_size=8,shuffle=True,generator=torch.Generator().manual_seed(SEED))\\nimages,targets,_=next(iter(train_loader))\\nprint(\'N,C,H,W:\',images.shape,\'| labels:\',targets.tolist())\\nfig,axes=plt.subplots(2,4,figsize=(12,6))\\nfor x,y,ax in zip(images,targets,axes.flat):\\n    rgb=x.permute(1,2,0).numpy()*[.229,.224,.225]+[.485,.456,.406]\\n    ax.imshow(np.clip(rgb,0,1));ax.set_title([\'empty\',\'animal\'][int(y)]);ax.axis(\'off\')\\nplt.tight_layout();plt.show()\\n", "metadata": {}}, {"cell_type": "markdown", "source": "### قبل وبعد", "metadata": {}}, {"cell_type": "code", "source": "sample=next(r for r in rows if r[\'split\']==\'train\' and r[\'animal\'])\\nbgr=cv2.imread(sample[\'path\']);pre=Preprocessor(224)\\nrgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB);prepared=pre.prepare(bgr)\\naugmented=np.array(train_dataset.tf(Image.fromarray(prepared)))\\nfig,axes=plt.subplots(1,3,figsize=(12,4))\\nfor ax,img,title in zip(axes,[rgb,prepared,augmented],[\'Downloaded RGB\',\'Resize 224\',\'Train augmentation\']):\\n    ax.imshow(img);ax.set_title(title);ax.axis(\'off\')\\nplt.show()\\nfor ch,color in enumerate([\'red\',\'green\',\'blue\']):\\n    plt.hist(prepared[:,:,ch].ravel(),bins=32,alpha=.35,color=color,label=color)\\nplt.legend();plt.title(\'Train sample channel histogram\');plt.show()\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 11. مثال قصاصة لتصنيف النوع", "metadata": {}}, {"cell_type": "code", "source": "row = next(r for r in rows if r[\'split\']==\'train\' and r[\'boxes\'])\\nbox = row[\'boxes\'][0]\\nim = Image.open(row[\'path\']).convert(\'RGB\')\\nx1,y1,x2,y2 = box[\'xyxy\']\\ndx,dy=(x2-x1)*.05,(y2-y1)*.05\\ncrop=im.crop((max(0,int((x1-dx)*im.width)),max(0,int((y1-dy)*im.height)),\\n              min(im.width,int((x2+dx)*im.width)),min(im.height,int((y2+dy)*im.height))))\\nprint(\'Species:\', box[\'species\']); display(crop)\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 12. Dataset Card — توثيق البيانات", "metadata": {}}, {"cell_type": "code", "source": "TEAM_NOTES = \\"\\"  # اكتب ملاحظاتك الفعلية عن البيانات\\ncounts = pd.read_csv(DATA/\'class_counts.csv\', index_col=0)\\ncleaning = read_json(DATA/\'cleaning_report.json\')\\nsummary = pd.DataFrame(rows).groupby(\'split\').agg(frames=(\'id\',\'count\'),cameras=(\'camera\',\'nunique\'))\\ncard = f\\"\\"\\"# Dataset Card — CCT20 / Phase 1\\nSource: https://lila.science/datasets/caltech-camera-traps\\nCitation: Beery, Van Horn and Perona, Recognition in Terra Incognita, ECCV 2018.\\nLicence: CDLA-Permissive; follow course policy: do not redistribute images or commit them to Git.\\n\\n## Tasks\\nFull-frame empty/animal classification; species classification on human-box crops.\\nNo predicted boxes are used as ground truth. Positive frames without boxes remain in triage only.\\nHuman/vehicle-labelled, conflicting or unlabelled frames are excluded by metadata.\\n\\n## Acquisition and cleaning\\nSelected: {cleaning[\'selected\']}; retained after human audit: {len(rows)}.\\nDownload failures: {cleaning[\'failures\']}; exact duplicate IDs removed: {len(cleaning[\'duplicate_ids\'])}.\\nNear-duplicate pairs across splits: {len(cleaning[\'near_duplicate_pairs\'])}.\\nImage maximum side: 1024; model input: 224 x 224; seed: {SEED}.\\nCorrupt/failed-event and duplicate exclusions are recorded in cleaning_report.json.\\n\\n## Split\\nStratifiedGroupKFold by camera, with joint image-label strata. Camera counts and class proportions below are measured, not assumed.\\nWhole bursts stay within one split; exact duplicate and event leakage checks pass.\\nGroup constraints approximate class stratification; this is not the official CCT benchmark split.\\n{summary.to_string()}\\n\\n## Long-tailed class distribution (frame label occurrences)\\n{counts.to_string()}\\nSpecies crop counts: crop_counts.csv. Rare species may have little or zero support in held-out locations.\\nTraining vocabulary is built from train crops only; unseen held-out species are reported by Member 2.\\n\\n## Final stratification check\\n{balance_report.to_string(index=False)}\\nRates count label presence per frame; multi-label class percentages need not sum to 100.\\nSee class_balance_proportions.csv and species_crop_support.csv. Camera constraints can prevent exact balance.\\n\\n## Label audit\\nReviewed: {audit[\'reviewed\']} / {audit[\'requested\']}; observed label errors: {audit[\'label_errors\']}.\\nOne-sided 95% upper error bound: {audit[\'one_sided_95_upper_bound\']}.\\nSampling: one frame per randomly sampled train event. Same-camera dependence may make this bound optimistic.\\nFlagged training events are quarantined; original audit decisions remain recorded.\\n\\n## Preprocessing\\nRGB, resize 224, ImageNet normalization. Random flip and brightness/contrast changes on train only.\\nThe identical OpenCV Preprocessor and augmentation code is embedded in Members 1, 2 and 3.\\n\\n## Limitations and privacy\\nCurated US wildlife data; no claim of Saudi generalisation. The empty fraction must be measured on this subset.\\nGrayscale is an infrared proxy, not verified nighttime. Blur is a review flag; no image is discarded solely for low Laplacian variance. No GPS publication; keep photos and review galleries private.\\nTest accuracy confidence intervals belong to Member 2 after training, not this untrained data notebook.\\n\\n## Team observations\\n{TEAM_NOTES or \'Pending: add actual observations before submission.\'}\\n\\"\\"\\"\\nPath(\'runs/data/DATASET_CARD.md\').write_text(card)\\ndisplay(Markdown(card))\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 13. تثبيت إصدار البيانات والإعداد المشترك", "metadata": {}}, {"cell_type": "code", "source": "blur_check=pd.read_csv(DATA/\'blur_review.csv\').fillna(\'\')\\nassert len(blur_check) and blur_check[\'reviewer\'].str.strip().ne(\'\').all() and blur_check[\'decision\'].isin([\'keep\',\'flag\']).all() and blur_check[\'notes\'].str.strip().ne(\'\').all(), \'أكمل blur_review.csv: reviewer وdecision = keep أو flag؛ flag للتوثيق ولا يحذف الصورة\'\\nassert TEAM_NOTES.strip(), \'أضف ملاحظاتك الفعلية في Dataset Card وأعد تشغيل خليتها\'\\ntraining_defaults = {\'project\': \'camera-trap-phase1\', \'backbone\': \'microsoft/resnet-50\', \'seed\': 42, \'profile\': \'full\', \'data_dir\': \'data/cct20_v1\', \'output_dir\': \'runs/data\', \'max_images\': 6000, \'download_workers\': 12, \'max_image_side\': 1024, \'image_size\': 224, \'batch_size\': 32, \'workers\': 2, \'epochs\': 12, \'patience\': 4, \'warmup_epochs\': 2, \'lr_scratch\': 0.001, \'lr_frozen\': 0.001, \'lr_full\': 0.0001, \'weight_decay\': 0.0001, \'label_smoothing\': 0.05, \'class_weighted_loss\': True, \'wandb_mode\': \'offline\', \'target_animal_recall\': 0.99, \'species_review_confidence\': 0.6, \'bootstrap_repeats\': 1000, \'audit_events\': 100, \'acceptance\': {\'animal_recall_min\': 0.99, \'empty_removal_min\': 0.2, \'species_macro_f1_min\': 0.65, \'species_ece_max\': 0.1}}\\ntraining_defaults.update(cfg)\\ntraining_defaults.update(contract_version=2,preprocessing_sha256=\'829fef7a32c1157ffe4da29da42026105afc0266ed647e562127bcc416697937\',warmup_epochs=0,weight_decay=0.0,label_smoothing=0.0,class_weighted_loss=False,optimizer=\'Adam\',wandb_mode=\'offline\')\\nwrite_json(\'handoff/preprocessing.json\',{\'sha256\':\'829fef7a32c1157ffe4da29da42026105afc0266ed647e562127bcc416697937\',\'input_size\':224,\'library\':\'OpenCV + torchvision\',\'train_only\':[\'horizontal_flip\',\'brightness_contrast\']})\\nwrite_json(\'handoff/data_config.json\', training_defaults)\\nwrite_json(DATA/\'preparation_config.json\', training_defaults)\\nPath(\'handoff/requirements_member1.txt\').write_text(subprocess.check_output([sys.executable,\'-m\',\'pip\',\'freeze\'],text=True))\\nwrite_json(\'handoff/runtime_member1.json\', {\'python\':sys.version, \'platform\':sys.platform})\\nfiles=[DATA/name for name in [\'manifest.json\',\'species.json\',\'site_split.json\',\'audit_report.json\',\\n \'label_audit.csv\',\'preparation_config.json\',\'cleaning_report.json\',\'class_counts.csv\',\'crop_counts.csv\',\'metadata_summary.json\',\'class_balance_proportions.csv\',\'species_crop_support.csv\',\'inventory.csv\',\'blur_review.csv\']]\\nfiles += [Path(\'handoff/data_config.json\'),Path(\'runs/data/DATASET_CARD.md\'),Path(\'handoff/preprocessing.json\')]\\nwrite_json(\'handoff/dataset_release.json\', {\'schema\':2,\'scope\':\'Phase 1 M1-M3\',\\n \'data_dir\':str(DATA),\'audit_complete\':True,\'files\':{str(f):digest(f) for f in files},\\n \'owner\':\'member1\',\'source\':\'CCT20 human annotations\',\'dataset_card\':\'runs/data/DATASET_CARD.md\'})\\nprint(\'Member 1 outputs are ready in:\', PROJECT)\\n", "metadata": {}}, {"cell_type": "markdown", "source": "# 2. الشبكة المدربة من الصفر — M2", "metadata": {}}, {"cell_type": "code", "source": "from pathlib import Path\\nimport os,json,hashlib,sys,subprocess,platform,random\\nimport numpy as np,pandas as pd\\nfrom IPython.display import display,Markdown\\nROOT=Path.cwd() if Path.cwd().name==\'CameraTrap_Phase1_Team\' else Path.cwd()/\'CameraTrap_Phase1_Team\'\\nROOT.mkdir(exist_ok=True);os.chdir(ROOT)\\nfor name in [\'handoff\',\'runs\',\'review\',\'submission\',\'docs\']:Path(name).mkdir(exist_ok=True)\\ndef read_json(p):return json.loads(Path(p).read_text())\\ndef write_json(p,v):\\n    Path(p).parent.mkdir(parents=True,exist_ok=True)\\n    Path(p).write_text(json.dumps(v,indent=2,ensure_ascii=False,allow_nan=False))\\ndef digest(p):\\n    h=hashlib.sha256()\\n    with open(p,\'rb\') as f:\\n        for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\\n    return h.hexdigest()\\ndef environment(member):\\n    Path(f\'handoff/requirements_member{member}.txt\').write_text(subprocess.check_output([sys.executable,\'-m\',\'pip\',\'freeze\'],text=True))\\n    write_json(f\'handoff/runtime_member{member}.json\',{\'python\':sys.version,\'platform\':platform.platform()})\\nprint(\'Workspace:\',ROOT)\\n", "metadata": {}}, {"cell_type": "code", "source": "release=read_json(\'handoff/dataset_release.json\')\\nassert release[\'schema\']==2 and release[\'audit_complete\'], \'استخدم إصدار Member 1 الجديد بعد اكتمال التدقيق\'\\nfor p,h in release[\'files\'].items():assert Path(p).is_file() and digest(p)==h, f\'Changed/missing: {p}\'\\nDATA=Path(release[\'data_dir\']);rows=read_json(DATA/\'manifest.json\')\\nassert rows and all(Path(r[\'path\']).exists() for r in rows)\\nDATA_HASH=digest(DATA/\'manifest.json\')\\nCFG=read_json(\'handoff/data_config.json\')\\nassert CFG[\'preprocessing_sha256\']==\'829fef7a32c1157ffe4da29da42026105afc0266ed647e562127bcc416697937\'\\nprint(\'Frames:\',len(rows),\'| split sizes:\',pd.Series([r[\'split\'] for r in rows]).value_counts().to_dict())\\n", "metadata": {}}, {"cell_type": "code", "source": "import cv2, numpy as np, torch\\nfrom PIL import Image\\nfrom torchvision import transforms\\nfrom torch.utils.data import DataLoader\\nclass Preprocessor:\\n    \\"\\"\\"Lab 1 cell 72: prepare uint8 RGB, augment, finalize and CHW tensor.\\"\\"\\"\\n    def __init__(self,size):\\n        self.size=size\\n        self.mean=np.array([.485,.456,.406],np.float32)\\n        self.std=np.array([.229,.224,.225],np.float32)\\n    def prepare(self,bgr):\\n        rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB);h,w=rgb.shape[:2]\\n        interp=cv2.INTER_AREA if h>self.size or w>self.size else cv2.INTER_LINEAR\\n        return cv2.resize(rgb,(self.size,self.size),interpolation=interp)\\n    def finalize(self,rgb):return (rgb.astype(np.float32)/255.-self.mean)/self.std\\n\\ndef load_rgb_sample(row):\\n    im=cv2.imread(row[\'path\'])\\n    if im is None:raise ValueError(\'Unreadable image: \'+row[\'path\'])\\n    if row.get(\'crop\') is not None:\\n        x1,y1,x2,y2=row[\'crop\'];h,w=im.shape[:2];dx=(x2-x1)*.05;dy=(y2-y1)*.05\\n        left,top=max(0,int((x1-dx)*w)),max(0,int((y1-dy)*h))\\n        right,bottom=min(w,max(left+1,int((x2+dx)*w))),min(h,max(top+1,int((y2+dy)*h)))\\n        im=im[top:bottom,left:right].copy()\\n    return im\\n\\nclass FrameDataset(torch.utils.data.Dataset):\\n    \\"\\"\\"Lab 1 cell 104 adapted to frame and human-box crop tasks.\\"\\"\\"\\n    def __init__(self,samples,size,train=False):\\n        self.samples=samples;self.pre=Preprocessor(size)\\n        self.tf=transforms.Compose([transforms.RandomHorizontalFlip(),\\n                  transforms.ColorJitter(brightness=.15,contrast=.15)]) if train else None\\n    def __len__(self):return len(self.samples)\\n    def __getitem__(self,i):\\n        row=self.samples[i];img=self.pre.prepare(load_rgb_sample(row))\\n        if self.tf:img=np.array(self.tf(Image.fromarray(img)))\\n        x=torch.from_numpy(np.ascontiguousarray(self.pre.finalize(img))).permute(2,0,1).float()\\n        return x,int(row[\'label\']),i", "metadata": {}}, {"cell_type": "code", "source": "def seed_everything(seed):\\n    random.seed(seed)\\n    np.random.seed(seed)\\n    import torch\\n    torch.manual_seed(seed)\\n    torch.cuda.manual_seed_all(seed)\\n    torch.backends.cudnn.benchmark = False\\n    torch.backends.cudnn.deterministic = True\\n\\ndef artifact_context(cfg, task):\\n    return dict(training_implementation=2, task=task, profile=cfg[\'profile\'], image_size=cfg[\'image_size\'],\\n                seed=cfg[\'seed\'], manifest_sha256=digest(Path(cfg[\'data_dir\'])/\'manifest.json\'),\\n                split_policy=\'camera-disjoint train / val_select / val_calibrate / test\',\\n                preprocessing_sha256=cfg[\'preprocessing_sha256\'],\\n                training_config={k:cfg[k] for k in [\'optimizer\',\'backbone\',\'batch_size\',\'epochs\',\'patience\',\'warmup_epochs\',\\n                    \'lr_scratch\',\'lr_frozen\',\'lr_full\',\'weight_decay\',\'label_smoothing\',\'class_weighted_loss\']})\\n\\ndef samples_for_task(rows,task,classes):\\n    samples=[]\\n    for r in rows:\\n        if task==\'triage\':\\n            samples.append(dict(r,label=r[\'animal\'],sample_id=r[\'id\'],crop=None))\\n        else:\\n            for i,b in enumerate(r[\'boxes\']):\\n                samples.append(dict(r,label=classes.index(b[\'species\']) if b[\'species\'] in classes else -1,\\n                                    sample_id=f\'{r[\\"id\\"]}_{i}\',crop=b[\'xyxy\'],species=b[\'species\']))\\n    return samples", "metadata": {}}, {"cell_type": "markdown", "source": "## 2. إعدادات مشتركة وتجارب W&B", "metadata": {}}, {"cell_type": "code", "source": "DEVICE=\'cuda\' if torch.cuda.is_available() else \'cpu\'\\nprint(\'Device:\',DEVICE)\\nif DEVICE==\'cpu\':print(\'CPU بطيء للتدريب الكامل؛ اختر GPU في Colab\')\\nWANDB_MODE=\'offline\'  # online لحساب W&B؛ لا تكتب المفتاح في الكود\\nimport wandb\\nif WANDB_MODE==\'online\':wandb.login()\\nREGIMES=[\'scratch\']\\nOUT=Path(\'runs/member2\');OUT.mkdir(parents=True,exist_ok=True)\\nCFG[\'output_dir\']=str(OUT)\\nassert CFG[\'optimizer\']==\'Adam\'\\nwrite_json(OUT/\'config.json\',CFG)\\nenvironment(2)\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 3. النموذج وforward", "metadata": {}}, {"cell_type": "code", "source": "from torch import nn\\n\\nclass ConvNN(nn.Module):\\n    \\"\\"\\"Lab 2 cell 39 architecture; debug helpers removed, dimensions explicit.\\"\\"\\"\\n    def __init__(self,num_classes,size=224,num_filters=16,num_neurons=100):\\n        super().__init__()\\n        def block(a,b):return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU())\\n        self.block_1=block(3,num_filters)\\n        self.block_2=block(num_filters,2*num_filters)\\n        self.block_3=block(2*num_filters,4*num_filters)\\n        self.maxpool=nn.MaxPool2d(kernel_size=3,stride=2)\\n        self.dropout=nn.Dropout(.2);self.relu=nn.ReLU();self.flat=nn.Flatten()\\n        h=size\\n        for _ in range(3):h=(h-3)//2+1\\n        self.fc1=nn.Linear(4*num_filters*h*h,num_neurons)\\n        self.fc2=nn.Linear(num_neurons,num_neurons)\\n        self.output=nn.Linear(num_neurons,num_classes)\\n    def forward(self,x):\\n        x=self.maxpool(self.block_1(x));x=self.maxpool(self.block_2(x));x=self.maxpool(self.block_3(x))\\n        x=self.flat(self.dropout(x));x=self.relu(self.fc1(x));x=self.relu(self.fc2(x))\\n        return self.output(x)\\n\\nclass TransferNet(nn.Module):\\n    \\"\\"\\"Lab 3 cells 25/34: the same HF ResNet-50 and replaced classifier.\\"\\"\\"\\n    def __init__(self,classes,pretrained=True):\\n        super().__init__()\\n        from transformers import AutoModelForImageClassification,ResNetConfig,ResNetForImageClassification\\n        self.net=(AutoModelForImageClassification.from_pretrained(\'microsoft/resnet-50\') if pretrained\\n                  else ResNetForImageClassification(ResNetConfig()))\\n        self.net.classifier=nn.Sequential(nn.Flatten(start_dim=1),nn.Linear(self.net.config.hidden_sizes[-1],classes))\\n        self.net.config.num_labels=classes\\n    def forward(self,x):return self.net(x).logits\\n    @property\\n    def classifier(self):return self.net.classifier\\n\\ndef make_model(regime,classes,pretrained=True,size=224):\\n    return ConvNN(classes,size) if regime==\'scratch\' else TransferNet(classes,pretrained)\\n\\ndef freeze_backbone(model,freeze):\\n    for p in model.net.base_model.parameters():p.requires_grad=not freeze\\n    for p in model.classifier.parameters():p.requires_grad=True", "metadata": {}}, {"cell_type": "code", "source": "classes=[\'empty\',\'animal\']\\nexample_model=make_model(REGIMES[0],len(classes),size=CFG[\'image_size\'])\\nif REGIMES[0]!=\'scratch\':freeze_backbone(example_model,True)\\nprint(example_model)\\nprint(\'Trainable parameters:\',sum(p.numel() for p in example_model.parameters() if p.requires_grad))\\nexample_model.eval()\\nwith torch.no_grad():print(\'Forward shape:\',example_model(torch.zeros(2,3,CFG[\'image_size\'],CFG[\'image_size\'])).shape)\\ndel example_model\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 4. المقاييس والمعايرة", "metadata": {}}, {"cell_type": "code", "source": "import numpy as np\\nfrom scipy.special import softmax, logsumexp\\nfrom scipy.optimize import minimize_scalar\\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score\\n\\n\\ndef ece(y, probabilities, bins=15):\\n    y=np.asarray(y);p=np.asarray(probabilities)\\n    if len(y)==0:return None\\n    conf=p.max(1);correct=p.argmax(1)==y\\n    indices=np.minimum((conf*bins).astype(int),bins-1)\\n    return float(sum(np.mean(indices==i)*abs(correct[indices==i].mean()-conf[indices==i].mean())\\n                     for i in range(bins) if np.any(indices==i)))\\n\\n\\ndef fit_temperature(logits,y):\\n    logits=np.asarray(logits);y=np.asarray(y,dtype=int)\\n    if not len(y):raise ValueError(\'No known-class calibration examples\')\\n    def nll(log_t):\\n        z=logits/np.exp(log_t)\\n        return float(np.mean(logsumexp(z,axis=1)-z[np.arange(len(y)),y]))\\n    result=minimize_scalar(nll,bounds=(np.log(0.25),np.log(10)),method=\'bounded\')\\n    return float(np.exp(result.x))\\n\\n\\ndef wilson(k,n,z=1.959963984540054):\\n    if n==0:return [None,None]\\n    p=k/n;d=1+z*z/n\\n    centre=(p+z*z/(2*n))/d\\n    half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d\\n    return [float(max(0,centre-half)),float(min(1,centre+half))]\\n\\n\\ndef triage_metrics(y, animal_scores, threshold):\\n    y=np.asarray(y);pred=np.asarray(animal_scores)>=threshold\\n    pos=y==1;neg=y==0\\n    tp=int(np.sum(pred & pos));fn=int(np.sum(~pred & pos));tn=int(np.sum(~pred & neg));fp=int(np.sum(pred & neg))\\n    return dict(threshold=float(threshold),animal_recall=float(tp/pos.sum()) if pos.any() else None,\\n                empty_removal_rate=float(tn/neg.sum()) if neg.any() else None,\\n                animal_frames_lost=fn,animal_frames=int(pos.sum()),empty_frames=int(neg.sum()),\\n                removed_frames=tn+fn,tp=tp,tn=tn,fp=fp,fn=fn,\\n                animal_recall_wilson95_independence_assumption=wilson(tp,int(pos.sum())))\\n\\n\\ndef choose_threshold(y, scores, target=0.99):\\n    if set(np.asarray(y).tolist())!={0,1}:raise ValueError(\'Calibration needs empty and animal labels\')\\n    candidates=np.unique(np.r_[0.,scores,1.])\\n    feasible=[triage_metrics(y,scores,t) for t in candidates]\\n    feasible=[r for r in feasible if r[\'animal_recall\']>=target]\\n    best=max(feasible,key=lambda r:(r[\'empty_removal_rate\'],r[\'threshold\']))\\n    best[\'selection_split\']=\'val_calibrate\'\\n    best[\'recall_target_is_empirical_not_guaranteed\']=True\\n    return best\\n\\n\\ndef classification_metrics(y,probs,classes):\\n    y=np.asarray(y);probs=np.asarray(probs);pred=probs.argmax(1);n=len(classes)\\n    support=np.bincount(y,minlength=n);supported=np.flatnonzero(support)\\n    report=classification_report(y,pred,labels=list(range(n)),target_names=classes,output_dict=True,zero_division=0)\\n    k=min(3,n)\\n    return dict(n=len(y),accuracy=float(accuracy_score(y,pred)),\\n       macro_f1_all_training_classes=float(f1_score(y,pred,labels=list(range(n)),average=\'macro\',zero_division=0)),\\n       macro_f1_supported_classes=float(f1_score(y,pred,labels=supported,average=\'macro\',zero_division=0)),\\n       top_k=k,top_k_accuracy=float(np.mean([label in indices for label,indices in zip(y,np.argsort(probs,axis=1)[:,-k:])])),\\n       ece=ece(y,probs),per_class=report,confusion_matrix=confusion_matrix(y,pred,labels=list(range(n))).tolist(),\\n       classes_absent_from_this_split=[classes[i] for i in range(n) if support[i]==0])\\n\\n\\ndef camera_bootstrap(y,probs,cameras,repeats=1000,seed=42,threshold=None):\\n    \\"\\"\\"Resample whole test cameras; preserve dependence among frames/crops within camera.\\"\\"\\"\\n    y=np.asarray(y);p=np.asarray(probs);c=np.asarray(cameras);sites=np.unique(c)\\n    if len(sites)<2:return {\'status\':\'not_estimable_fewer_than_two_cameras\'}\\n    rng=np.random.default_rng(seed);acc=[];f1=[];rec=[];empty=[]\\n    for _ in range(repeats):\\n        idx=np.concatenate([np.flatnonzero(c==site) for site in rng.choice(sites,len(sites),replace=True)])\\n        yy=y[idx];pp=p[idx]\\n        acc.append(float(np.mean(pp.argmax(1)==yy)))\\n        f1.append(float(f1_score(yy,pp.argmax(1),labels=range(p.shape[1]),average=\'macro\',zero_division=0)))\\n        if threshold is not None:\\n            m=triage_metrics(yy,pp[:,1],threshold)\\n            if m[\'animal_recall\'] is not None:rec.append(m[\'animal_recall\'])\\n            if m[\'empty_removal_rate\'] is not None:empty.append(m[\'empty_removal_rate\'])\\n    ci=lambda v:np.quantile(v,[.025,.975]).tolist() if v else [None,None]\\n    out=dict(unit=\'camera\',cameras=len(sites),repeats=repeats,accuracy95=ci(acc),macro_f1_95=ci(f1),\\n             caveat=\'Intervals are descriptive and unstable when there are few held-out cameras.\')\\n    if threshold is not None:out.update(animal_recall95=ci(rec),empty_removal95=ci(empty))\\n    return out\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 5. حلقة التدريب — forward → loss → backward → step", "metadata": {}}, {"cell_type": "code", "source": "from scipy.special import softmax\\nimport time,wandb\\n\\ndef loader(samples,train=False):\\n    gen=torch.Generator().manual_seed(CFG[\'seed\'])\\n    return DataLoader(FrameDataset(samples,CFG[\'image_size\'],train),batch_size=CFG[\'batch_size\'],\\n                      shuffle=train,num_workers=0,generator=gen)\\n\\ndef predict(model,data):\\n    model.eval();all_logits=[];all_y=[]\\n    with torch.no_grad():\\n        for images,labels,_ in data:\\n            all_logits.append(model(images.to(DEVICE)).cpu().numpy());all_y.extend(labels.tolist())\\n    assert all_logits, \'No known-class samples in this split\'\\n    return np.concatenate(all_logits),np.array(all_y,dtype=int)\\n\\ndef train_model(task,regime):\\n    seed_everything(CFG[\'seed\'])\\n    classes=[\'empty\',\'animal\'] if task==\'triage\' else read_json(DATA/\'species.json\')\\n    samples=samples_for_task(rows,task,classes)\\n    train=[r for r in samples if r[\'split\']==\'train\' and r[\'label\']>=0]\\n    val=[r for r in samples if r[\'split\']==\'val_select\' and r[\'label\']>=0]\\n    calibrate=[r for r in samples if r[\'split\']==\'val_calibrate\' and r[\'label\']>=0]\\n    assert train and val and calibrate\\n    folder=OUT/task/regime;folder.mkdir(parents=True,exist_ok=True)\\n    context=artifact_context(CFG,task)\\n    if (folder/\'summary.json\').exists():\\n        saved=read_json(folder/\'summary.json\');assert saved[\'context\']==context\\n        assert saved[\'checkpoint_sha256\']==digest(folder/\'best.pt\')\\n        print(\'Reusing completed run:\',task,regime);return saved\\n    assert not (folder/\'best.pt\').exists(), \'تدريب جزئي موجود؛ استخدم إصدار تجربة جديدًا لتجنب استبداله\'\\n    model=make_model(regime,len(classes),size=CFG[\'image_size\']).to(DEVICE)\\n    if regime!=\'scratch\':freeze_backbone(model,regime==\'frozen\')\\n    assert CFG[\'optimizer\']==\'Adam\', \'Only Adam is implemented\'\\n    weights=None\\n    if CFG[\'class_weighted_loss\']:\\n        counts=np.bincount([r[\'label\'] for r in train],minlength=len(classes))\\n        assert (counts>0).all(), \'Every training class must have samples\'\\n        weights=torch.tensor(len(train)/(len(classes)*counts),dtype=torch.float32,device=DEVICE)\\n    criterion=nn.CrossEntropyLoss(weight=weights,label_smoothing=CFG[\'label_smoothing\'])\\n    optimizer=torch.optim.Adam([p for p in model.parameters() if p.requires_grad],lr=CFG[\'lr_\'+regime],weight_decay=CFG[\'weight_decay\'])\\n    run=wandb.init(project=CFG[\'project\'],name=f\'{task}-{regime}\',group=DATA_HASH[:12],\\n                   mode=WANDB_MODE,dir=str(folder.resolve()),config=dict(CFG,task=task,regime=regime),reinit=True)\\n    best=-1;stale=0;history=[]\\n    for epoch in range(CFG[\'epochs\']):\\n        warmup=max(0,int(CFG[\'warmup_epochs\']))\\n        for group in optimizer.param_groups:\\n            group[\'lr\']=CFG[\'lr_\'+regime]*min(1.,(epoch+1)/max(1,warmup))\\n        model.train()\\n        if regime==\'frozen\':model.eval();model.classifier.train() # Freeze BatchNorm state too.\\n        total_loss=0;correct=0;total=0\\n        for images,labels,_ in loader(train,True):\\n            images,labels=images.to(DEVICE),labels.to(DEVICE)\\n            optimizer.zero_grad()\\n            logits=model(images)                     # forward\\n            loss=criterion(logits,labels)            # compare with true labels\\n            loss.backward()                         # gradients\\n            optimizer.step()                        # update weights\\n            total_loss+=loss.item()*len(labels)\\n            correct+=int((logits.argmax(1)==labels).sum());total+=len(labels)\\n        logits,y=predict(model,loader(val))\\n        m=classification_metrics(y,softmax(logits,axis=1),classes)\\n        val_loss=float(nn.functional.cross_entropy(torch.tensor(logits),torch.tensor(y)))\\n        row={\'epoch\':epoch+1,\'learning_rate\':optimizer.param_groups[0][\'lr\'],\'train_loss\':total_loss/total,\'train_accuracy\':correct/total,\\n             \'val_loss\':val_loss,\'val_accuracy\':m[\'accuracy\'],\'val_macro_f1\':m[\'macro_f1_all_training_classes\']}\\n        history.append(row);run.log(row);print(task,regime,row)\\n        if row[\'val_macro_f1\']>best:\\n            best=row[\'val_macro_f1\'];stale=0\\n            torch.save({\'state_dict\':model.state_dict(),\'regime\':regime,\'task\':task,\'classes\':classes,\\n              \'image_size\':CFG[\'image_size\'],\'epoch\':epoch+1,\'config\':CFG,\'context\':context},folder/\'best.pt\')\\n        else:stale+=1\\n        if stale>=CFG[\'patience\']:break\\n    pd.DataFrame(history).to_csv(folder/\'history.csv\',index=False)\\n    checkpoint=torch.load(folder/\'best.pt\',map_location=DEVICE,weights_only=False)\\n    model.load_state_dict(checkpoint[\'state_dict\'])\\n    z,y=predict(model,loader(calibrate));temperature=fit_temperature(z,y)\\n    before=softmax(z,axis=1);after=softmax(z/temperature,axis=1)\\n    policy={\'temperature\':temperature,\'calibration_split\':\'val_calibrate\',\\n            \'species_review_confidence\':CFG[\'species_review_confidence\']}\\n    if task==\'triage\':\\n        policy.update(choose_threshold(y,after[:,1],CFG[\'target_animal_recall\']))\\n        sweep=[triage_metrics(y,after[:,1],t) for t in np.linspace(0,1,101)]\\n        pd.DataFrame(sweep).to_csv(folder/\'validation_threshold_sweep.csv\',index=False)\\n    write_json(folder/\'policy.json\',policy)\\n    result={\'task\':task,\'regime\':regime,\'context\':context,\'selection_score\':best,\\n      \'best_epoch\':checkpoint[\'epoch\'],\'classes\':classes,\'checkpoint_sha256\':digest(folder/\'best.pt\'),\\n      \'wandb_run_id\':run.id,\'wandb_url\':run.url if WANDB_MODE==\'online\' else None,\\n      \'wandb_mode\':WANDB_MODE,\'calibration_ece_before\':ece(y,before),\'calibration_ece_after\':ece(y,after),\\n      \'train_n\':len(train),\'val_select_n\':len(val),\'val_calibrate_n\':len(calibrate)}\\n    run.summary.update({\'best_val_macro_f1\':best,\'calibration_ece_before\':ece(y,before),\'calibration_ece_after\':ece(y,after)})\\n    run.finish();write_json(folder/\'summary.json\',result)\\n    return result\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 6. تدريب فارغة/حيوان", "metadata": {}}, {"cell_type": "code", "source": "triage_runs=[train_model(\'triage\',regime) for regime in REGIMES]\\ndisplay(pd.DataFrame(triage_runs)[[\'task\',\'regime\',\'selection_score\',\'best_epoch\']])", "metadata": {}}, {"cell_type": "markdown", "source": "## 7. تدريب الأنواع على القصاصات البشرية", "metadata": {}}, {"cell_type": "code", "source": "species_runs=[train_model(\'species\',regime) for regime in REGIMES]\\ndisplay(pd.DataFrame(species_runs)[[\'task\',\'regime\',\'selection_score\',\'best_epoch\']])", "metadata": {}}, {"cell_type": "markdown", "source": "## 8. تثبيت الاختيار قبل Test", "metadata": {}}, {"cell_type": "code", "source": "selection={task:max([r for r in triage_runs+species_runs if r[\'task\']==task],key=lambda r:r[\'selection_score\'])[\'regime\'] for task in [\'triage\',\'species\']}\\nwrite_json(OUT/\'selection.json\',{\'selected\':selection,\'criterion\':\'val_select macro-F1\',\'dataset_hash\':DATA_HASH})\\nprint(selection)\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 9. التقييم النهائي وعرض النتائج", "metadata": {}}, {"cell_type": "code", "source": "import matplotlib.pyplot as plt\\nfrom sklearn.metrics import ConfusionMatrixDisplay\\n\\ndef evaluate_model(task,regime):\\n    folder=OUT/task/regime;summary=read_json(folder/\'summary.json\')\\n    signature={\'data\':DATA_HASH,\'checkpoint\':digest(folder/\'best.pt\'),\'policy\':digest(folder/\'policy.json\')}\\n    if (folder/\'test_metrics.json\').exists():\\n        saved=read_json(folder/\'test_metrics.json\');assert saved[\'signature\']==signature\\n        return saved\\n    checkpoint=torch.load(folder/\'best.pt\',map_location=DEVICE,weights_only=False)\\n    assert checkpoint[\'context\']==artifact_context(CFG,task)\\n    classes=checkpoint[\'classes\'];policy=read_json(folder/\'policy.json\')\\n    model=make_model(regime,len(classes),pretrained=False,size=CFG[\'image_size\']).to(DEVICE)\\n    model.load_state_dict(checkpoint[\'state_dict\'])\\n    all_test=[r for r in samples_for_task(rows,task,classes) if r[\'split\']==\'test\']\\n    known=[r for r in all_test if r[\'label\']>=0];unknown=[r for r in all_test if r[\'label\']<0]\\n    logits,y=predict(model,loader(known));raw=softmax(logits,axis=1);p=softmax(logits/policy[\'temperature\'],axis=1)\\n    m=classification_metrics(y,p,classes)\\n    m.update(task=task,regime=regime,split=\'test\',image_size=CFG[\'image_size\'],policy=policy,\\n       signature=signature,context=checkpoint[\'context\'],uncalibrated_ece=ece(y,raw),\\n       known_count=len(known),unknown_count=len(unknown),coverage=len(known)/len(all_test),\\n       unknown_species=sorted({r[\'species\'] for r in unknown}),\\n       accuracy_counting_unknown_as_errors=float((p.argmax(1)==y).sum()/len(all_test)),\\n       accuracy_wilson95_independence_assumption=wilson(int((p.argmax(1)==y).sum()),len(y)),\\n       camera_bootstrap=camera_bootstrap(y,p,[r[\'camera\'] for r in known],CFG[\'bootstrap_repeats\'],CFG[\'seed\'],\\n                                      policy[\'threshold\'] if task==\'triage\' else None))\\n    if task==\'triage\':m[\'operating_point\']=triage_metrics(y,p[:,1],policy[\'threshold\'])\\n    m[\'per_camera\']={camera:classification_metrics(y[idx],p[idx],classes)\\n      for camera in sorted({r[\'camera\'] for r in known})\\n      for idx in [[i for i,r in enumerate(known) if r[\'camera\']==camera]]}\\n    np.savez_compressed(folder/\'test_probabilities.npz\',y=y,probabilities=p,raw=raw)\\n    pred=p.argmax(1)\\n    pd.DataFrame([{\'sample_id\':r[\'sample_id\'],\'camera\':r[\'camera\'],\'true\':classes[y[i]],\\n      \'prediction\':classes[pred[i]],\'confidence\':float(p[i].max())} for i,r in enumerate(known)]).to_csv(folder/\'test_predictions.csv\',index=False)\\n    # Triage gallery uses the actual removal threshold, not argmax.\\n    decisions=(p[:,1]>=policy[\'threshold\']).astype(int) if task==\'triage\' else pred\\n    errors=np.flatnonzero(decisions!=y)\\n    chosen=sorted(errors,key=lambda i:float(p[i].max()),reverse=True)[:12]\\n    if chosen:\\n        fig,axes=plt.subplots((len(chosen)+3)//4,4,figsize=(12,3*((len(chosen)+3)//4)))\\n        for ax in np.asarray(axes).flat:ax.axis(\'off\')\\n        for ax,i in zip(np.asarray(axes).flat,chosen):\\n            rgb=cv2.cvtColor(load_rgb_sample(known[i]),cv2.COLOR_BGR2RGB)\\n            ax.imshow(rgb);ax.set_title(f\\"{known[i][\'sample_id\'][:12]}\\\\n{classes[y[i]]} -> {classes[decisions[i]]}\\",fontsize=8)\\n        fig.suptitle(f\'{task}/{regime}: test errors, input {CFG[\\"image_size\\"]}, threshold \'+str(policy.get(\'threshold\',\'argmax\')))\\n        fig.tight_layout();fig.savefig(folder/\'private_errors.jpg\');plt.close(fig)\\n    run=wandb.init(project=CFG[\'project\'],name=f\'{task}-{regime}-test\',group=DATA_HASH[:12],\\n      mode=WANDB_MODE,dir=str(folder.resolve()),config=dict(CFG,task=task,regime=regime,split=\'locked_test\'),reinit=True)\\n    run.log({\'test/accuracy\':m[\'accuracy\'],\'test/macro_f1\':m[\'macro_f1_all_training_classes\'],\\n             \'test/ece\':m[\'ece\'],\'test/top_k_accuracy\':m[\'top_k_accuracy\']})\\n    run.log({\'test/per_class\':wandb.Table(dataframe=pd.DataFrame(m[\'per_class\']).T.reset_index())})\\n    m[\'wandb_test_run_id\']=run.id;m[\'wandb_test_url\']=run.url if WANDB_MODE==\'online\' else None\\n    run.finish();write_json(folder/\'test_metrics.json\',m)\\n    return m\\n\\ndef show_results(task,regime):\\n    folder=OUT/task/regime;m=read_json(folder/\'test_metrics.json\')\\n    display(pd.DataFrame(m[\'per_class\']).T)\\n    print(\'Accuracy:\',m[\'accuracy\'],\'Top-k:\',m[\'top_k_accuracy\'],\'ECE:\',m[\'ece\'])\\n    print(\'Camera bootstrap:\',m[\'camera_bootstrap\'],\'Unknown crops:\',m[\'unknown_count\'])\\n    h=pd.read_csv(folder/\'history.csv\');h.plot(x=\'epoch\',y=[\'train_loss\',\'val_loss\'],title=f\'{task}/{regime}: train/val_select loss, {CFG[\\"image_size\\"]}px, threshold N/A\');plt.tight_layout();plt.savefig(folder/\'learning_loss.png\');plt.show()\\n    h.plot(x=\'epoch\',y=[\'train_accuracy\',\'val_accuracy\'],title=f\'{task}/{regime}: train/val_select accuracy, {CFG[\\"image_size\\"]}px, argmax\');plt.tight_layout();plt.savefig(folder/\'learning_accuracy.png\');plt.show()\\n    classes=read_json(folder/\'summary.json\')[\'classes\']\\n    ConfusionMatrixDisplay(np.array(m[\'confusion_matrix\']),display_labels=classes).plot(xticks_rotation=90)\\n    plt.title(f\'{task}/{regime}: test, {CFG[\\"image_size\\"]}px, argmax\');plt.tight_layout();plt.savefig(folder/\'confusion.png\');plt.show()\\n    z=np.load(folder/\'test_probabilities.npz\');y=z[\'y\']\\n    plt.plot([0,1],[0,1],\'--\',color=\'gray\')\\n    for key,label in [(\'raw\',\'Before calibration\'),(\'probabilities\',\'After calibration\')]:\\n        p=z[key];conf=p.max(1);correct=p.argmax(1)==y;bins=np.minimum((conf*10).astype(int),9)\\n        points=[(conf[bins==b].mean(),correct[bins==b].mean()) for b in range(10) if np.any(bins==b)]\\n        plt.plot(*zip(*points),\'o-\',label=label)\\n    plt.xlabel(\'Confidence\');plt.ylabel(\'Accuracy\');plt.legend();plt.title(f\'{task}/{regime}: test reliability, {CFG[\\"image_size\\"]}px, argmax\');plt.tight_layout();plt.savefig(folder/\'reliability.png\');plt.show()\\n    if (folder/\'private_errors.jpg\').exists():display(Image.open(folder/\'private_errors.jpg\'))\\n    if task==\'triage\':print(\'Operating point:\',m[\'operating_point\'])\\n", "metadata": {}}, {"cell_type": "code", "source": "for task in [\'triage\',\'species\']:\\n    for regime in REGIMES:\\n        evaluate_model(task,regime)\\n        show_results(task,regime)", "metadata": {}}, {"cell_type": "markdown", "source": "## 10. تحليل الأخطاء وتوثيق التجارب", "metadata": {}}, {"cell_type": "code", "source": "analysis=OUT/\'ERROR_ANALYSIS.md\'\\nif not analysis.exists():analysis.write_text(\'# Error analysis\\\\n\\\\nTODO: actual sample IDs, rare-class recall/support, false empty removals, calibration, plausible causes, and data improvement plan.\\\\n\')\\nSYNC_WANDB=False\\nif SYNC_WANDB:\\n    wandb.login()\\n    for p in OUT.glob(\'*/*/wandb/offline-run-*\'):\\n        subprocess.run([sys.executable,\'-m\',\'wandb\',\'sync\',str(p)],check=True)\\n", "metadata": {}}, {"cell_type": "code", "source": "required=[OUT/\'config.json\',OUT/\'selection.json\']\\nfor task in [\'triage\',\'species\']:\\n    for regime in REGIMES:\\n        required += [OUT/task/regime/n for n in [\'best.pt\',\'summary.json\',\'history.csv\',\'policy.json\',\'test_metrics.json\',\'test_probabilities.npz\',\'test_predictions.csv\']]\\nwrite_json(\'handoff/member2_release.json\',{\'schema\':2,\'dataset_hash\':DATA_HASH,\'output_dir\':str(OUT),\'member\':2,\'files\':{str(p):digest(p) for p in required}})\\nprint(\'Ready: member2_release.json\')\\n", "metadata": {}}, {"cell_type": "markdown", "source": "# 3. التعلم بالنقل — M3", "metadata": {}}, {"cell_type": "code", "source": "from pathlib import Path\\nimport os,json,hashlib,sys,subprocess,platform,random\\nimport numpy as np,pandas as pd\\nfrom IPython.display import display,Markdown\\nROOT=Path.cwd() if Path.cwd().name==\'CameraTrap_Phase1_Team\' else Path.cwd()/\'CameraTrap_Phase1_Team\'\\nROOT.mkdir(exist_ok=True);os.chdir(ROOT)\\nfor name in [\'handoff\',\'runs\',\'review\',\'submission\',\'docs\']:Path(name).mkdir(exist_ok=True)\\ndef read_json(p):return json.loads(Path(p).read_text())\\ndef write_json(p,v):\\n    Path(p).parent.mkdir(parents=True,exist_ok=True)\\n    Path(p).write_text(json.dumps(v,indent=2,ensure_ascii=False,allow_nan=False))\\ndef digest(p):\\n    h=hashlib.sha256()\\n    with open(p,\'rb\') as f:\\n        for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\\n    return h.hexdigest()\\ndef environment(member):\\n    Path(f\'handoff/requirements_member{member}.txt\').write_text(subprocess.check_output([sys.executable,\'-m\',\'pip\',\'freeze\'],text=True))\\n    write_json(f\'handoff/runtime_member{member}.json\',{\'python\':sys.version,\'platform\':platform.platform()})\\nprint(\'Workspace:\',ROOT)\\n", "metadata": {}}, {"cell_type": "code", "source": "release=read_json(\'handoff/dataset_release.json\')\\nassert release[\'schema\']==2 and release[\'audit_complete\'], \'استخدم إصدار Member 1 الجديد بعد اكتمال التدقيق\'\\nfor p,h in release[\'files\'].items():assert Path(p).is_file() and digest(p)==h, f\'Changed/missing: {p}\'\\nDATA=Path(release[\'data_dir\']);rows=read_json(DATA/\'manifest.json\')\\nassert rows and all(Path(r[\'path\']).exists() for r in rows)\\nDATA_HASH=digest(DATA/\'manifest.json\')\\nCFG=read_json(\'handoff/data_config.json\')\\nassert CFG[\'preprocessing_sha256\']==\'829fef7a32c1157ffe4da29da42026105afc0266ed647e562127bcc416697937\'\\nprint(\'Frames:\',len(rows),\'| split sizes:\',pd.Series([r[\'split\'] for r in rows]).value_counts().to_dict())\\n", "metadata": {}}, {"cell_type": "code", "source": "import cv2, numpy as np, torch\\nfrom PIL import Image\\nfrom torchvision import transforms\\nfrom torch.utils.data import DataLoader\\nclass Preprocessor:\\n    \\"\\"\\"Lab 1 cell 72: prepare uint8 RGB, augment, finalize and CHW tensor.\\"\\"\\"\\n    def __init__(self,size):\\n        self.size=size\\n        self.mean=np.array([.485,.456,.406],np.float32)\\n        self.std=np.array([.229,.224,.225],np.float32)\\n    def prepare(self,bgr):\\n        rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB);h,w=rgb.shape[:2]\\n        interp=cv2.INTER_AREA if h>self.size or w>self.size else cv2.INTER_LINEAR\\n        return cv2.resize(rgb,(self.size,self.size),interpolation=interp)\\n    def finalize(self,rgb):return (rgb.astype(np.float32)/255.-self.mean)/self.std\\n\\ndef load_rgb_sample(row):\\n    im=cv2.imread(row[\'path\'])\\n    if im is None:raise ValueError(\'Unreadable image: \'+row[\'path\'])\\n    if row.get(\'crop\') is not None:\\n        x1,y1,x2,y2=row[\'crop\'];h,w=im.shape[:2];dx=(x2-x1)*.05;dy=(y2-y1)*.05\\n        left,top=max(0,int((x1-dx)*w)),max(0,int((y1-dy)*h))\\n        right,bottom=min(w,max(left+1,int((x2+dx)*w))),min(h,max(top+1,int((y2+dy)*h)))\\n        im=im[top:bottom,left:right].copy()\\n    return im\\n\\nclass FrameDataset(torch.utils.data.Dataset):\\n    \\"\\"\\"Lab 1 cell 104 adapted to frame and human-box crop tasks.\\"\\"\\"\\n    def __init__(self,samples,size,train=False):\\n        self.samples=samples;self.pre=Preprocessor(size)\\n        self.tf=transforms.Compose([transforms.RandomHorizontalFlip(),\\n                  transforms.ColorJitter(brightness=.15,contrast=.15)]) if train else None\\n    def __len__(self):return len(self.samples)\\n    def __getitem__(self,i):\\n        row=self.samples[i];img=self.pre.prepare(load_rgb_sample(row))\\n        if self.tf:img=np.array(self.tf(Image.fromarray(img)))\\n        x=torch.from_numpy(np.ascontiguousarray(self.pre.finalize(img))).permute(2,0,1).float()\\n        return x,int(row[\'label\']),i", "metadata": {}}, {"cell_type": "code", "source": "def seed_everything(seed):\\n    random.seed(seed)\\n    np.random.seed(seed)\\n    import torch\\n    torch.manual_seed(seed)\\n    torch.cuda.manual_seed_all(seed)\\n    torch.backends.cudnn.benchmark = False\\n    torch.backends.cudnn.deterministic = True\\n\\ndef artifact_context(cfg, task):\\n    return dict(training_implementation=2, task=task, profile=cfg[\'profile\'], image_size=cfg[\'image_size\'],\\n                seed=cfg[\'seed\'], manifest_sha256=digest(Path(cfg[\'data_dir\'])/\'manifest.json\'),\\n                split_policy=\'camera-disjoint train / val_select / val_calibrate / test\',\\n                preprocessing_sha256=cfg[\'preprocessing_sha256\'],\\n                training_config={k:cfg[k] for k in [\'optimizer\',\'backbone\',\'batch_size\',\'epochs\',\'patience\',\'warmup_epochs\',\\n                    \'lr_scratch\',\'lr_frozen\',\'lr_full\',\'weight_decay\',\'label_smoothing\',\'class_weighted_loss\']})\\n\\ndef samples_for_task(rows,task,classes):\\n    samples=[]\\n    for r in rows:\\n        if task==\'triage\':\\n            samples.append(dict(r,label=r[\'animal\'],sample_id=r[\'id\'],crop=None))\\n        else:\\n            for i,b in enumerate(r[\'boxes\']):\\n                samples.append(dict(r,label=classes.index(b[\'species\']) if b[\'species\'] in classes else -1,\\n                                    sample_id=f\'{r[\\"id\\"]}_{i}\',crop=b[\'xyxy\'],species=b[\'species\']))\\n    return samples", "metadata": {}}, {"cell_type": "markdown", "source": "## 2. إعدادات مشتركة وتجارب W&B", "metadata": {}}, {"cell_type": "code", "source": "DEVICE=\'cuda\' if torch.cuda.is_available() else \'cpu\'\\nprint(\'Device:\',DEVICE)\\nif DEVICE==\'cpu\':print(\'CPU بطيء للتدريب الكامل؛ اختر GPU في Colab\')\\nWANDB_MODE=\'offline\'  # online لحساب W&B؛ لا تكتب المفتاح في الكود\\nimport wandb\\nif WANDB_MODE==\'online\':wandb.login()\\nREGIMES=[\'frozen\', \'full\']\\nOUT=Path(\'runs/member3\');OUT.mkdir(parents=True,exist_ok=True)\\nCFG[\'output_dir\']=str(OUT)\\nassert CFG[\'optimizer\']==\'Adam\'\\nwrite_json(OUT/\'config.json\',CFG)\\nenvironment(3)\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 3. النموذج وforward", "metadata": {}}, {"cell_type": "code", "source": "from torch import nn\\n\\nclass ConvNN(nn.Module):\\n    \\"\\"\\"Lab 2 cell 39 architecture; debug helpers removed, dimensions explicit.\\"\\"\\"\\n    def __init__(self,num_classes,size=224,num_filters=16,num_neurons=100):\\n        super().__init__()\\n        def block(a,b):return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU())\\n        self.block_1=block(3,num_filters)\\n        self.block_2=block(num_filters,2*num_filters)\\n        self.block_3=block(2*num_filters,4*num_filters)\\n        self.maxpool=nn.MaxPool2d(kernel_size=3,stride=2)\\n        self.dropout=nn.Dropout(.2);self.relu=nn.ReLU();self.flat=nn.Flatten()\\n        h=size\\n        for _ in range(3):h=(h-3)//2+1\\n        self.fc1=nn.Linear(4*num_filters*h*h,num_neurons)\\n        self.fc2=nn.Linear(num_neurons,num_neurons)\\n        self.output=nn.Linear(num_neurons,num_classes)\\n    def forward(self,x):\\n        x=self.maxpool(self.block_1(x));x=self.maxpool(self.block_2(x));x=self.maxpool(self.block_3(x))\\n        x=self.flat(self.dropout(x));x=self.relu(self.fc1(x));x=self.relu(self.fc2(x))\\n        return self.output(x)\\n\\nclass TransferNet(nn.Module):\\n    \\"\\"\\"Lab 3 cells 25/34: the same HF ResNet-50 and replaced classifier.\\"\\"\\"\\n    def __init__(self,classes,pretrained=True):\\n        super().__init__()\\n        from transformers import AutoModelForImageClassification,ResNetConfig,ResNetForImageClassification\\n        self.net=(AutoModelForImageClassification.from_pretrained(\'microsoft/resnet-50\') if pretrained\\n                  else ResNetForImageClassification(ResNetConfig()))\\n        self.net.classifier=nn.Sequential(nn.Flatten(start_dim=1),nn.Linear(self.net.config.hidden_sizes[-1],classes))\\n        self.net.config.num_labels=classes\\n    def forward(self,x):return self.net(x).logits\\n    @property\\n    def classifier(self):return self.net.classifier\\n\\ndef make_model(regime,classes,pretrained=True,size=224):\\n    return ConvNN(classes,size) if regime==\'scratch\' else TransferNet(classes,pretrained)\\n\\ndef freeze_backbone(model,freeze):\\n    for p in model.net.base_model.parameters():p.requires_grad=not freeze\\n    for p in model.classifier.parameters():p.requires_grad=True", "metadata": {}}, {"cell_type": "code", "source": "classes=[\'empty\',\'animal\']\\nexample_model=make_model(REGIMES[0],len(classes),size=CFG[\'image_size\'])\\nif REGIMES[0]!=\'scratch\':freeze_backbone(example_model,True)\\nprint(example_model)\\nprint(\'Trainable parameters:\',sum(p.numel() for p in example_model.parameters() if p.requires_grad))\\nexample_model.eval()\\nwith torch.no_grad():print(\'Forward shape:\',example_model(torch.zeros(2,3,CFG[\'image_size\'],CFG[\'image_size\'])).shape)\\ndel example_model\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 4. المقاييس والمعايرة", "metadata": {}}, {"cell_type": "code", "source": "import numpy as np\\nfrom scipy.special import softmax, logsumexp\\nfrom scipy.optimize import minimize_scalar\\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score\\n\\n\\ndef ece(y, probabilities, bins=15):\\n    y=np.asarray(y);p=np.asarray(probabilities)\\n    if len(y)==0:return None\\n    conf=p.max(1);correct=p.argmax(1)==y\\n    indices=np.minimum((conf*bins).astype(int),bins-1)\\n    return float(sum(np.mean(indices==i)*abs(correct[indices==i].mean()-conf[indices==i].mean())\\n                     for i in range(bins) if np.any(indices==i)))\\n\\n\\ndef fit_temperature(logits,y):\\n    logits=np.asarray(logits);y=np.asarray(y,dtype=int)\\n    if not len(y):raise ValueError(\'No known-class calibration examples\')\\n    def nll(log_t):\\n        z=logits/np.exp(log_t)\\n        return float(np.mean(logsumexp(z,axis=1)-z[np.arange(len(y)),y]))\\n    result=minimize_scalar(nll,bounds=(np.log(0.25),np.log(10)),method=\'bounded\')\\n    return float(np.exp(result.x))\\n\\n\\ndef wilson(k,n,z=1.959963984540054):\\n    if n==0:return [None,None]\\n    p=k/n;d=1+z*z/n\\n    centre=(p+z*z/(2*n))/d\\n    half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d\\n    return [float(max(0,centre-half)),float(min(1,centre+half))]\\n\\n\\ndef triage_metrics(y, animal_scores, threshold):\\n    y=np.asarray(y);pred=np.asarray(animal_scores)>=threshold\\n    pos=y==1;neg=y==0\\n    tp=int(np.sum(pred & pos));fn=int(np.sum(~pred & pos));tn=int(np.sum(~pred & neg));fp=int(np.sum(pred & neg))\\n    return dict(threshold=float(threshold),animal_recall=float(tp/pos.sum()) if pos.any() else None,\\n                empty_removal_rate=float(tn/neg.sum()) if neg.any() else None,\\n                animal_frames_lost=fn,animal_frames=int(pos.sum()),empty_frames=int(neg.sum()),\\n                removed_frames=tn+fn,tp=tp,tn=tn,fp=fp,fn=fn,\\n                animal_recall_wilson95_independence_assumption=wilson(tp,int(pos.sum())))\\n\\n\\ndef choose_threshold(y, scores, target=0.99):\\n    if set(np.asarray(y).tolist())!={0,1}:raise ValueError(\'Calibration needs empty and animal labels\')\\n    candidates=np.unique(np.r_[0.,scores,1.])\\n    feasible=[triage_metrics(y,scores,t) for t in candidates]\\n    feasible=[r for r in feasible if r[\'animal_recall\']>=target]\\n    best=max(feasible,key=lambda r:(r[\'empty_removal_rate\'],r[\'threshold\']))\\n    best[\'selection_split\']=\'val_calibrate\'\\n    best[\'recall_target_is_empirical_not_guaranteed\']=True\\n    return best\\n\\n\\ndef classification_metrics(y,probs,classes):\\n    y=np.asarray(y);probs=np.asarray(probs);pred=probs.argmax(1);n=len(classes)\\n    support=np.bincount(y,minlength=n);supported=np.flatnonzero(support)\\n    report=classification_report(y,pred,labels=list(range(n)),target_names=classes,output_dict=True,zero_division=0)\\n    k=min(3,n)\\n    return dict(n=len(y),accuracy=float(accuracy_score(y,pred)),\\n       macro_f1_all_training_classes=float(f1_score(y,pred,labels=list(range(n)),average=\'macro\',zero_division=0)),\\n       macro_f1_supported_classes=float(f1_score(y,pred,labels=supported,average=\'macro\',zero_division=0)),\\n       top_k=k,top_k_accuracy=float(np.mean([label in indices for label,indices in zip(y,np.argsort(probs,axis=1)[:,-k:])])),\\n       ece=ece(y,probs),per_class=report,confusion_matrix=confusion_matrix(y,pred,labels=list(range(n))).tolist(),\\n       classes_absent_from_this_split=[classes[i] for i in range(n) if support[i]==0])\\n\\n\\ndef camera_bootstrap(y,probs,cameras,repeats=1000,seed=42,threshold=None):\\n    \\"\\"\\"Resample whole test cameras; preserve dependence among frames/crops within camera.\\"\\"\\"\\n    y=np.asarray(y);p=np.asarray(probs);c=np.asarray(cameras);sites=np.unique(c)\\n    if len(sites)<2:return {\'status\':\'not_estimable_fewer_than_two_cameras\'}\\n    rng=np.random.default_rng(seed);acc=[];f1=[];rec=[];empty=[]\\n    for _ in range(repeats):\\n        idx=np.concatenate([np.flatnonzero(c==site) for site in rng.choice(sites,len(sites),replace=True)])\\n        yy=y[idx];pp=p[idx]\\n        acc.append(float(np.mean(pp.argmax(1)==yy)))\\n        f1.append(float(f1_score(yy,pp.argmax(1),labels=range(p.shape[1]),average=\'macro\',zero_division=0)))\\n        if threshold is not None:\\n            m=triage_metrics(yy,pp[:,1],threshold)\\n            if m[\'animal_recall\'] is not None:rec.append(m[\'animal_recall\'])\\n            if m[\'empty_removal_rate\'] is not None:empty.append(m[\'empty_removal_rate\'])\\n    ci=lambda v:np.quantile(v,[.025,.975]).tolist() if v else [None,None]\\n    out=dict(unit=\'camera\',cameras=len(sites),repeats=repeats,accuracy95=ci(acc),macro_f1_95=ci(f1),\\n             caveat=\'Intervals are descriptive and unstable when there are few held-out cameras.\')\\n    if threshold is not None:out.update(animal_recall95=ci(rec),empty_removal95=ci(empty))\\n    return out\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 5. حلقة التدريب — forward → loss → backward → step", "metadata": {}}, {"cell_type": "code", "source": "from scipy.special import softmax\\nimport time,wandb\\n\\ndef loader(samples,train=False):\\n    gen=torch.Generator().manual_seed(CFG[\'seed\'])\\n    return DataLoader(FrameDataset(samples,CFG[\'image_size\'],train),batch_size=CFG[\'batch_size\'],\\n                      shuffle=train,num_workers=0,generator=gen)\\n\\ndef predict(model,data):\\n    model.eval();all_logits=[];all_y=[]\\n    with torch.no_grad():\\n        for images,labels,_ in data:\\n            all_logits.append(model(images.to(DEVICE)).cpu().numpy());all_y.extend(labels.tolist())\\n    assert all_logits, \'No known-class samples in this split\'\\n    return np.concatenate(all_logits),np.array(all_y,dtype=int)\\n\\ndef train_model(task,regime):\\n    seed_everything(CFG[\'seed\'])\\n    classes=[\'empty\',\'animal\'] if task==\'triage\' else read_json(DATA/\'species.json\')\\n    samples=samples_for_task(rows,task,classes)\\n    train=[r for r in samples if r[\'split\']==\'train\' and r[\'label\']>=0]\\n    val=[r for r in samples if r[\'split\']==\'val_select\' and r[\'label\']>=0]\\n    calibrate=[r for r in samples if r[\'split\']==\'val_calibrate\' and r[\'label\']>=0]\\n    assert train and val and calibrate\\n    folder=OUT/task/regime;folder.mkdir(parents=True,exist_ok=True)\\n    context=artifact_context(CFG,task)\\n    if (folder/\'summary.json\').exists():\\n        saved=read_json(folder/\'summary.json\');assert saved[\'context\']==context\\n        assert saved[\'checkpoint_sha256\']==digest(folder/\'best.pt\')\\n        print(\'Reusing completed run:\',task,regime);return saved\\n    assert not (folder/\'best.pt\').exists(), \'تدريب جزئي موجود؛ استخدم إصدار تجربة جديدًا لتجنب استبداله\'\\n    model=make_model(regime,len(classes),size=CFG[\'image_size\']).to(DEVICE)\\n    if regime!=\'scratch\':freeze_backbone(model,regime==\'frozen\')\\n    assert CFG[\'optimizer\']==\'Adam\', \'Only Adam is implemented\'\\n    weights=None\\n    if CFG[\'class_weighted_loss\']:\\n        counts=np.bincount([r[\'label\'] for r in train],minlength=len(classes))\\n        assert (counts>0).all(), \'Every training class must have samples\'\\n        weights=torch.tensor(len(train)/(len(classes)*counts),dtype=torch.float32,device=DEVICE)\\n    criterion=nn.CrossEntropyLoss(weight=weights,label_smoothing=CFG[\'label_smoothing\'])\\n    optimizer=torch.optim.Adam([p for p in model.parameters() if p.requires_grad],lr=CFG[\'lr_\'+regime],weight_decay=CFG[\'weight_decay\'])\\n    run=wandb.init(project=CFG[\'project\'],name=f\'{task}-{regime}\',group=DATA_HASH[:12],\\n                   mode=WANDB_MODE,dir=str(folder.resolve()),config=dict(CFG,task=task,regime=regime),reinit=True)\\n    best=-1;stale=0;history=[]\\n    for epoch in range(CFG[\'epochs\']):\\n        warmup=max(0,int(CFG[\'warmup_epochs\']))\\n        for group in optimizer.param_groups:\\n            group[\'lr\']=CFG[\'lr_\'+regime]*min(1.,(epoch+1)/max(1,warmup))\\n        model.train()\\n        if regime==\'frozen\':model.eval();model.classifier.train() # Freeze BatchNorm state too.\\n        total_loss=0;correct=0;total=0\\n        for images,labels,_ in loader(train,True):\\n            images,labels=images.to(DEVICE),labels.to(DEVICE)\\n            optimizer.zero_grad()\\n            logits=model(images)                     # forward\\n            loss=criterion(logits,labels)            # compare with true labels\\n            loss.backward()                         # gradients\\n            optimizer.step()                        # update weights\\n            total_loss+=loss.item()*len(labels)\\n            correct+=int((logits.argmax(1)==labels).sum());total+=len(labels)\\n        logits,y=predict(model,loader(val))\\n        m=classification_metrics(y,softmax(logits,axis=1),classes)\\n        val_loss=float(nn.functional.cross_entropy(torch.tensor(logits),torch.tensor(y)))\\n        row={\'epoch\':epoch+1,\'learning_rate\':optimizer.param_groups[0][\'lr\'],\'train_loss\':total_loss/total,\'train_accuracy\':correct/total,\\n             \'val_loss\':val_loss,\'val_accuracy\':m[\'accuracy\'],\'val_macro_f1\':m[\'macro_f1_all_training_classes\']}\\n        history.append(row);run.log(row);print(task,regime,row)\\n        if row[\'val_macro_f1\']>best:\\n            best=row[\'val_macro_f1\'];stale=0\\n            torch.save({\'state_dict\':model.state_dict(),\'regime\':regime,\'task\':task,\'classes\':classes,\\n              \'image_size\':CFG[\'image_size\'],\'epoch\':epoch+1,\'config\':CFG,\'context\':context},folder/\'best.pt\')\\n        else:stale+=1\\n        if stale>=CFG[\'patience\']:break\\n    pd.DataFrame(history).to_csv(folder/\'history.csv\',index=False)\\n    checkpoint=torch.load(folder/\'best.pt\',map_location=DEVICE,weights_only=False)\\n    model.load_state_dict(checkpoint[\'state_dict\'])\\n    z,y=predict(model,loader(calibrate));temperature=fit_temperature(z,y)\\n    before=softmax(z,axis=1);after=softmax(z/temperature,axis=1)\\n    policy={\'temperature\':temperature,\'calibration_split\':\'val_calibrate\',\\n            \'species_review_confidence\':CFG[\'species_review_confidence\']}\\n    if task==\'triage\':\\n        policy.update(choose_threshold(y,after[:,1],CFG[\'target_animal_recall\']))\\n        sweep=[triage_metrics(y,after[:,1],t) for t in np.linspace(0,1,101)]\\n        pd.DataFrame(sweep).to_csv(folder/\'validation_threshold_sweep.csv\',index=False)\\n    write_json(folder/\'policy.json\',policy)\\n    result={\'task\':task,\'regime\':regime,\'context\':context,\'selection_score\':best,\\n      \'best_epoch\':checkpoint[\'epoch\'],\'classes\':classes,\'checkpoint_sha256\':digest(folder/\'best.pt\'),\\n      \'wandb_run_id\':run.id,\'wandb_url\':run.url if WANDB_MODE==\'online\' else None,\\n      \'wandb_mode\':WANDB_MODE,\'calibration_ece_before\':ece(y,before),\'calibration_ece_after\':ece(y,after),\\n      \'train_n\':len(train),\'val_select_n\':len(val),\'val_calibrate_n\':len(calibrate)}\\n    run.summary.update({\'best_val_macro_f1\':best,\'calibration_ece_before\':ece(y,before),\'calibration_ece_after\':ece(y,after)})\\n    run.finish();write_json(folder/\'summary.json\',result)\\n    return result\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 6. تدريب فارغة/حيوان", "metadata": {}}, {"cell_type": "code", "source": "triage_runs=[train_model(\'triage\',regime) for regime in REGIMES]\\ndisplay(pd.DataFrame(triage_runs)[[\'task\',\'regime\',\'selection_score\',\'best_epoch\']])", "metadata": {}}, {"cell_type": "markdown", "source": "## 7. تدريب الأنواع على القصاصات البشرية", "metadata": {}}, {"cell_type": "code", "source": "species_runs=[train_model(\'species\',regime) for regime in REGIMES]\\ndisplay(pd.DataFrame(species_runs)[[\'task\',\'regime\',\'selection_score\',\'best_epoch\']])", "metadata": {}}, {"cell_type": "markdown", "source": "## 8. تثبيت الاختيار قبل Test", "metadata": {}}, {"cell_type": "code", "source": "selection={task:max([r for r in triage_runs+species_runs if r[\'task\']==task],key=lambda r:r[\'selection_score\'])[\'regime\'] for task in [\'triage\',\'species\']}\\nwrite_json(OUT/\'selection.json\',{\'selected\':selection,\'criterion\':\'val_select macro-F1\',\'dataset_hash\':DATA_HASH})\\nprint(selection)\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 9. التقييم النهائي وعرض النتائج", "metadata": {}}, {"cell_type": "code", "source": "import matplotlib.pyplot as plt\\nfrom sklearn.metrics import ConfusionMatrixDisplay\\n\\ndef evaluate_model(task,regime):\\n    folder=OUT/task/regime;summary=read_json(folder/\'summary.json\')\\n    signature={\'data\':DATA_HASH,\'checkpoint\':digest(folder/\'best.pt\'),\'policy\':digest(folder/\'policy.json\')}\\n    if (folder/\'test_metrics.json\').exists():\\n        saved=read_json(folder/\'test_metrics.json\');assert saved[\'signature\']==signature\\n        return saved\\n    checkpoint=torch.load(folder/\'best.pt\',map_location=DEVICE,weights_only=False)\\n    assert checkpoint[\'context\']==artifact_context(CFG,task)\\n    classes=checkpoint[\'classes\'];policy=read_json(folder/\'policy.json\')\\n    model=make_model(regime,len(classes),pretrained=False,size=CFG[\'image_size\']).to(DEVICE)\\n    model.load_state_dict(checkpoint[\'state_dict\'])\\n    all_test=[r for r in samples_for_task(rows,task,classes) if r[\'split\']==\'test\']\\n    known=[r for r in all_test if r[\'label\']>=0];unknown=[r for r in all_test if r[\'label\']<0]\\n    logits,y=predict(model,loader(known));raw=softmax(logits,axis=1);p=softmax(logits/policy[\'temperature\'],axis=1)\\n    m=classification_metrics(y,p,classes)\\n    m.update(task=task,regime=regime,split=\'test\',image_size=CFG[\'image_size\'],policy=policy,\\n       signature=signature,context=checkpoint[\'context\'],uncalibrated_ece=ece(y,raw),\\n       known_count=len(known),unknown_count=len(unknown),coverage=len(known)/len(all_test),\\n       unknown_species=sorted({r[\'species\'] for r in unknown}),\\n       accuracy_counting_unknown_as_errors=float((p.argmax(1)==y).sum()/len(all_test)),\\n       accuracy_wilson95_independence_assumption=wilson(int((p.argmax(1)==y).sum()),len(y)),\\n       camera_bootstrap=camera_bootstrap(y,p,[r[\'camera\'] for r in known],CFG[\'bootstrap_repeats\'],CFG[\'seed\'],\\n                                      policy[\'threshold\'] if task==\'triage\' else None))\\n    if task==\'triage\':m[\'operating_point\']=triage_metrics(y,p[:,1],policy[\'threshold\'])\\n    m[\'per_camera\']={camera:classification_metrics(y[idx],p[idx],classes)\\n      for camera in sorted({r[\'camera\'] for r in known})\\n      for idx in [[i for i,r in enumerate(known) if r[\'camera\']==camera]]}\\n    np.savez_compressed(folder/\'test_probabilities.npz\',y=y,probabilities=p,raw=raw)\\n    pred=p.argmax(1)\\n    pd.DataFrame([{\'sample_id\':r[\'sample_id\'],\'camera\':r[\'camera\'],\'true\':classes[y[i]],\\n      \'prediction\':classes[pred[i]],\'confidence\':float(p[i].max())} for i,r in enumerate(known)]).to_csv(folder/\'test_predictions.csv\',index=False)\\n    # Triage gallery uses the actual removal threshold, not argmax.\\n    decisions=(p[:,1]>=policy[\'threshold\']).astype(int) if task==\'triage\' else pred\\n    errors=np.flatnonzero(decisions!=y)\\n    chosen=sorted(errors,key=lambda i:float(p[i].max()),reverse=True)[:12]\\n    if chosen:\\n        fig,axes=plt.subplots((len(chosen)+3)//4,4,figsize=(12,3*((len(chosen)+3)//4)))\\n        for ax in np.asarray(axes).flat:ax.axis(\'off\')\\n        for ax,i in zip(np.asarray(axes).flat,chosen):\\n            rgb=cv2.cvtColor(load_rgb_sample(known[i]),cv2.COLOR_BGR2RGB)\\n            ax.imshow(rgb);ax.set_title(f\\"{known[i][\'sample_id\'][:12]}\\\\n{classes[y[i]]} -> {classes[decisions[i]]}\\",fontsize=8)\\n        fig.suptitle(f\'{task}/{regime}: test errors, input {CFG[\\"image_size\\"]}, threshold \'+str(policy.get(\'threshold\',\'argmax\')))\\n        fig.tight_layout();fig.savefig(folder/\'private_errors.jpg\');plt.close(fig)\\n    run=wandb.init(project=CFG[\'project\'],name=f\'{task}-{regime}-test\',group=DATA_HASH[:12],\\n      mode=WANDB_MODE,dir=str(folder.resolve()),config=dict(CFG,task=task,regime=regime,split=\'locked_test\'),reinit=True)\\n    run.log({\'test/accuracy\':m[\'accuracy\'],\'test/macro_f1\':m[\'macro_f1_all_training_classes\'],\\n             \'test/ece\':m[\'ece\'],\'test/top_k_accuracy\':m[\'top_k_accuracy\']})\\n    run.log({\'test/per_class\':wandb.Table(dataframe=pd.DataFrame(m[\'per_class\']).T.reset_index())})\\n    m[\'wandb_test_run_id\']=run.id;m[\'wandb_test_url\']=run.url if WANDB_MODE==\'online\' else None\\n    run.finish();write_json(folder/\'test_metrics.json\',m)\\n    return m\\n\\ndef show_results(task,regime):\\n    folder=OUT/task/regime;m=read_json(folder/\'test_metrics.json\')\\n    display(pd.DataFrame(m[\'per_class\']).T)\\n    print(\'Accuracy:\',m[\'accuracy\'],\'Top-k:\',m[\'top_k_accuracy\'],\'ECE:\',m[\'ece\'])\\n    print(\'Camera bootstrap:\',m[\'camera_bootstrap\'],\'Unknown crops:\',m[\'unknown_count\'])\\n    h=pd.read_csv(folder/\'history.csv\');h.plot(x=\'epoch\',y=[\'train_loss\',\'val_loss\'],title=f\'{task}/{regime}: train/val_select loss, {CFG[\\"image_size\\"]}px, threshold N/A\');plt.tight_layout();plt.savefig(folder/\'learning_loss.png\');plt.show()\\n    h.plot(x=\'epoch\',y=[\'train_accuracy\',\'val_accuracy\'],title=f\'{task}/{regime}: train/val_select accuracy, {CFG[\\"image_size\\"]}px, argmax\');plt.tight_layout();plt.savefig(folder/\'learning_accuracy.png\');plt.show()\\n    classes=read_json(folder/\'summary.json\')[\'classes\']\\n    ConfusionMatrixDisplay(np.array(m[\'confusion_matrix\']),display_labels=classes).plot(xticks_rotation=90)\\n    plt.title(f\'{task}/{regime}: test, {CFG[\\"image_size\\"]}px, argmax\');plt.tight_layout();plt.savefig(folder/\'confusion.png\');plt.show()\\n    z=np.load(folder/\'test_probabilities.npz\');y=z[\'y\']\\n    plt.plot([0,1],[0,1],\'--\',color=\'gray\')\\n    for key,label in [(\'raw\',\'Before calibration\'),(\'probabilities\',\'After calibration\')]:\\n        p=z[key];conf=p.max(1);correct=p.argmax(1)==y;bins=np.minimum((conf*10).astype(int),9)\\n        points=[(conf[bins==b].mean(),correct[bins==b].mean()) for b in range(10) if np.any(bins==b)]\\n        plt.plot(*zip(*points),\'o-\',label=label)\\n    plt.xlabel(\'Confidence\');plt.ylabel(\'Accuracy\');plt.legend();plt.title(f\'{task}/{regime}: test reliability, {CFG[\\"image_size\\"]}px, argmax\');plt.tight_layout();plt.savefig(folder/\'reliability.png\');plt.show()\\n    if (folder/\'private_errors.jpg\').exists():display(Image.open(folder/\'private_errors.jpg\'))\\n    if task==\'triage\':print(\'Operating point:\',m[\'operating_point\'])\\n", "metadata": {}}, {"cell_type": "code", "source": "for task in [\'triage\',\'species\']:\\n    for regime in REGIMES:\\n        evaluate_model(task,regime)\\n        show_results(task,regime)", "metadata": {}}, {"cell_type": "markdown", "source": "## 10. تحليل الأخطاء وتوثيق التجارب", "metadata": {}}, {"cell_type": "code", "source": "analysis=OUT/\'ERROR_ANALYSIS.md\'\\nif not analysis.exists():analysis.write_text(\'# Error analysis\\\\n\\\\nTODO: actual sample IDs, rare-class recall/support, false empty removals, calibration, plausible causes, and data improvement plan.\\\\n\')\\nSYNC_WANDB=False\\nif SYNC_WANDB:\\n    wandb.login()\\n    for p in OUT.glob(\'*/*/wandb/offline-run-*\'):\\n        subprocess.run([sys.executable,\'-m\',\'wandb\',\'sync\',str(p)],check=True)\\n", "metadata": {}}, {"cell_type": "code", "source": "required=[OUT/\'config.json\',OUT/\'selection.json\']\\nfor task in [\'triage\',\'species\']:\\n    for regime in REGIMES:\\n        required += [OUT/task/regime/n for n in [\'best.pt\',\'summary.json\',\'history.csv\',\'policy.json\',\'test_metrics.json\',\'test_probabilities.npz\',\'test_predictions.csv\']]\\nwrite_json(\'handoff/member3_release.json\',{\'schema\':2,\'dataset_hash\':DATA_HASH,\'output_dir\':str(OUT),\'member\':3,\'files\':{str(p):digest(p) for p in required}})\\nprint(\'Ready: member3_release.json\')\\n", "metadata": {}}, {"cell_type": "markdown", "source": "# 4. المقارنة والتحقق والتقرير", "metadata": {}}, {"cell_type": "code", "source": "from pathlib import Path\\nimport os,json,hashlib,sys,subprocess,platform,random\\nimport numpy as np,pandas as pd\\nfrom IPython.display import display,Markdown\\nROOT=Path.cwd() if Path.cwd().name==\'CameraTrap_Phase1_Team\' else Path.cwd()/\'CameraTrap_Phase1_Team\'\\nROOT.mkdir(exist_ok=True);os.chdir(ROOT)\\nfor name in [\'handoff\',\'runs\',\'review\',\'submission\',\'docs\']:Path(name).mkdir(exist_ok=True)\\ndef read_json(p):return json.loads(Path(p).read_text())\\ndef write_json(p,v):\\n    Path(p).parent.mkdir(parents=True,exist_ok=True)\\n    Path(p).write_text(json.dumps(v,indent=2,ensure_ascii=False,allow_nan=False))\\ndef digest(p):\\n    h=hashlib.sha256()\\n    with open(p,\'rb\') as f:\\n        for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\\n    return h.hexdigest()\\ndef environment(member):\\n    Path(f\'handoff/requirements_member{member}.txt\').write_text(subprocess.check_output([sys.executable,\'-m\',\'pip\',\'freeze\'],text=True))\\n    write_json(f\'handoff/runtime_member{member}.json\',{\'python\':sys.version,\'platform\':platform.platform()})\\nprint(\'Workspace:\',ROOT)\\n", "metadata": {}}, {"cell_type": "code", "source": "release=read_json(\'handoff/dataset_release.json\')\\nassert release[\'schema\']==2 and release[\'audit_complete\'], \'استخدم إصدار Member 1 الجديد بعد اكتمال التدقيق\'\\nfor p,h in release[\'files\'].items():assert Path(p).is_file() and digest(p)==h, f\'Changed/missing: {p}\'\\nDATA=Path(release[\'data_dir\']);rows=read_json(DATA/\'manifest.json\')\\nassert rows and all(Path(r[\'path\']).exists() for r in rows)\\nDATA_HASH=digest(DATA/\'manifest.json\')\\nCFG=read_json(\'handoff/data_config.json\')\\nassert CFG[\'preprocessing_sha256\']==\'829fef7a32c1157ffe4da29da42026105afc0266ed647e562127bcc416697937\'\\nprint(\'Frames:\',len(rows),\'| split sizes:\',pd.Series([r[\'split\'] for r in rows]).value_counts().to_dict())\\n", "metadata": {}}, {"cell_type": "code", "source": "environment(4)", "metadata": {}}, {"cell_type": "markdown", "source": "## 2. فحص مستقل ومقارنة التجارب الست", "metadata": {}}, {"cell_type": "code", "source": "from collections import defaultdict\\nfrom sklearn.metrics import accuracy_score,f1_score\\nchecks=[]\\ndef check(name,passed,details=\'\'):\\n    checks.append({\'check\':name,\'passed\':bool(passed),\'details\':str(details)})\\nfor key in [\'id\',\'camera\',\'event\',\'pixel_sha256\']:\\n    groups=defaultdict(set)\\n    for r in rows:groups[r[key]].add(r[\'split\'])\\n    bad=[k for k,v in groups.items() if len(v)>1]\\n    check(key+\'_disjoint\',not bad,len(bad))\\ncheck(\'unique_ids\',len(rows)==len({r[\'id\'] for r in rows}))\\nchanged=[]\\nfor r in rows:\\n    if hashlib.md5(Path(r[\'path\']).read_bytes()).hexdigest()!=r[\'processed_file_md5\']:changed.append(r[\'id\'])\\ncheck(\'image_integrity\',not changed,len(changed))\\naudit=read_json(DATA/\'audit_report.json\')\\ncheck(\'human_audit\',audit[\'status\']==\'complete\' and audit[\'unresolved_flagged_events\']==0)\\nfor split in [\'train\',\'val_select\',\'val_calibrate\',\'test\']:\\n    check(\'empty_animal_support_\'+split,{r[\'animal\'] for r in rows if r[\'split\']==split}=={0,1})\\nrecords=[];contexts=[]\\nfor member in [2,3]:\\n    rel=read_json(f\'handoff/member{member}_release.json\')\\n    assert rel[\'schema\']==2 and rel[\'dataset_hash\']==DATA_HASH\\n    for p,h in rel[\'files\'].items():assert Path(p).is_file() and digest(p)==h, f\'Changed run artifact: {p}\'\\n    out=Path(rel[\'output_dir\'])\\n    for task in [\'triage\',\'species\']:\\n        for regime in ([\'scratch\'] if member==2 else [\'frozen\',\'full\']):\\n            folder=out/task/regime;s=read_json(folder/\'summary.json\');m=read_json(folder/\'test_metrics.json\')\\n            z=np.load(folder/\'test_probabilities.npz\');y=z[\'y\'];p=z[\'probabilities\'];pred=p.argmax(1)\\n            confidence=p.max(1);correct=pred==y;bins=np.minimum((confidence*15).astype(int),14)\\n            ece_value=sum(np.mean(bins==i)*abs(correct[bins==i].mean()-confidence[bins==i].mean()) for i in range(15) if np.any(bins==i))\\n            f1=f1_score(y,pred,labels=list(range(p.shape[1])),average=\'macro\',zero_division=0)\\n            check(f\'metrics_{task}_{regime}\',np.isclose(m[\'accuracy\'],accuracy_score(y,pred)) and\\n                  np.isclose(m[\'macro_f1_all_training_classes\'],f1) and np.isclose(m[\'ece\'],ece_value))\\n            check(f\'provenance_{task}_{regime}\',m[\'context\']==s[\'context\'] and\\n                  m[\'signature\']=={\'data\':DATA_HASH,\'checkpoint\':digest(folder/\'best.pt\'),\'policy\':digest(folder/\'policy.json\')})\\n            contexts.append({k:v for k,v in s[\'context\'].items() if k!=\'task\'})\\n            records.append({\'member\':member,\'task\':task,\'regime\':regime,\'folder\':str(folder),\\n              \'val_macro_f1\':s[\'selection_score\'],\'test_accuracy\':m[\'accuracy\'],\\n              \'test_macro_f1\':m[\'macro_f1_all_training_classes\'],\'top_k\':m[\'top_k\'],\\n              \'top_k_accuracy\':m[\'top_k_accuracy\'],\'ece\':m[\'ece\'],\'raw_ece\':m[\'uncalibrated_ece\'],\\n              \'coverage\':m[\'coverage\'],\'unknown_crops\':m[\'unknown_count\'],\'input_size\':m[\'image_size\'],\\n              \'threshold\':m[\'policy\'].get(\'threshold\',\'argmax\'),\\n              \'animal_recall\':m.get(\'operating_point\',{}).get(\'animal_recall\'),\\n              \'empty_removal\':m.get(\'operating_point\',{}).get(\'empty_removal_rate\'),\\n              \'accuracy_ci_camera\':str(m[\'camera_bootstrap\'].get(\'accuracy95\')),\\n              \'train_run_id\':s[\'wandb_run_id\'],\'train_url\':s.get(\'wandb_url\') or \'\',\\n              \'test_run_id\':m[\'wandb_test_run_id\'],\'test_url\':m.get(\'wandb_test_url\') or \'\'})\\ncheck(\'common_training_configuration\',all(c==contexts[0] for c in contexts))\\ncomparison=pd.DataFrame(records)\\ncomparison[\'selected_on_validation\']=False\\nfor task in [\'triage\',\'species\']:\\n    idx=comparison.loc[comparison.task==task,\'val_macro_f1\'].idxmax()\\n    comparison.loc[idx,\'selected_on_validation\']=True\\n# Acceptance targets were fixed in Member1 before training; a failed target is a result, not a reason to retune on test.\\nacceptance=[]\\nfor row in comparison[comparison.selected_on_validation].to_dict(\'records\'):\\n    targets=([(\'animal_recall\',\'animal_recall_min\',\'min\'),(\'empty_removal\',\'empty_removal_min\',\'min\')]\\n             if row[\'task\']==\'triage\' else [(\'test_macro_f1\',\'species_macro_f1_min\',\'min\'),(\'ece\',\'species_ece_max\',\'max\')])\\n    for metric,key,direction in targets:\\n        value=row[metric];target=CFG[\'acceptance\'][key]\\n        acceptance.append({\'task\':row[\'task\'],\'regime\':row[\'regime\'],\'metric\':metric,\'value\':value,\'target\':target,\\n                           \'direction\':direction,\'met\':bool(value>=target if direction==\'min\' else value<=target)})\\npd.DataFrame(acceptance).to_csv(\'submission/acceptance.csv\',index=False)\\ndisplay(pd.DataFrame(acceptance))\\ncomparison.to_csv(\'submission/comparison.csv\',index=False)\\npd.DataFrame(checks).to_csv(\'review/automatic_checks.csv\',index=False)\\ndisplay(pd.DataFrame(checks));display(comparison)\\nprint(\'اختيار الأفضل مبني على val_macro_f1 فقط، وليس test.\')\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 3. عرض المقارنة وأدلة الأخطاء", "metadata": {}}, {"cell_type": "code", "source": "import matplotlib.pyplot as plt\\nfrom PIL import Image\\nfor task in [\'triage\',\'species\']:\\n    table=comparison[comparison.task==task]\\n    table.plot.bar(x=\'regime\',y=[\'test_accuracy\',\'test_macro_f1\',\'ece\'],title=f\'{task}: held-out test, {CFG[\\"image_size\\"]}px, argmax\')\\n    plt.tight_layout();plt.savefig(\'submission/\'+task+\'_comparison.png\');plt.show()\\nfor row in comparison[comparison.selected_on_validation].to_dict(\'records\'):\\n    folder=Path(row[\'folder\']);m=read_json(folder/\'test_metrics.json\')\\n    print(row[\'task\'],row[\'regime\'],\'| Split:\',m[\'split\'],\'Input:\',m[\'image_size\'],\'Policy:\',m[\'policy\'])\\n    display(pd.DataFrame(m[\'per_class\']).T)\\n    if (folder/\'private_errors.jpg\').exists():display(Image.open(folder/\'private_errors.jpg\'))\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 4. عينة مراجعة مستقلة وسجل المشاكل", "metadata": {}}, {"cell_type": "code", "source": "audit_path=Path(\'review/independent_audit.csv\')\\nif not audit_path.exists():\\n    events={}\\n    for r in rows:\\n        if r[\'split\']==\'train\':events.setdefault(r[\'event\'],[]).append(r)\\n    rng=random.Random(2026)\\n    sample=[rng.choice(events[e]) for e in rng.sample(sorted(events),min(20,len(events)))]\\n    pd.DataFrame([{\'id\':r[\'id\'],\'label_correct\':\'\',\'boxes_correct\':\'\',\'person_visible\':\'\',\'reviewer\':\'\',\'notes\':\'\'} for r in sample]).to_csv(audit_path,index=False)\\nby_id={r[\'id\']:r for r in rows}\\nfrom PIL import ImageDraw\\nfor ident in pd.read_csv(audit_path).id:\\n    r=by_id[ident];im=Image.open(r[\'path\']).convert(\'RGB\');draw=ImageDraw.Draw(im)\\n    for b in r[\'boxes\']:\\n        x1,y1,x2,y2=b[\'xyxy\'];draw.rectangle((x1*im.width,y1*im.height,x2*im.width,y2*im.height),outline=\'lime\',width=3)\\n    im.thumbnail((650,420));print(ident,r[\'labels\'] or [\'empty\']);display(im)\\nissues=Path(\'review/issues.csv\')\\nif not issues.exists():pd.DataFrame(columns=[\'issue_id\',\'owner\',\'problem\',\'status\',\'resolution\',\'evidence\']).to_csv(issues,index=False)\\nprint(\'CSV decisions: label_correct=yes/no, boxes_correct=yes/no/not applicable, person_visible=yes/no.\')\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 5. فحص إعادة تحميل الأوزان بعملية Python جديدة", "metadata": {}}, {"cell_type": "code", "source": "PROBE_SOURCE = \'import cv2, numpy as np, torch\\\\nfrom PIL import Image\\\\nfrom torchvision import transforms\\\\nfrom torch.utils.data import DataLoader\\\\nclass Preprocessor:\\\\n    \\"\\"\\"Lab 1 cell 72: prepare uint8 RGB, augment, finalize and CHW tensor.\\"\\"\\"\\\\n    def __init__(self,size):\\\\n        self.size=size\\\\n        self.mean=np.array([.485,.456,.406],np.float32)\\\\n        self.std=np.array([.229,.224,.225],np.float32)\\\\n    def prepare(self,bgr):\\\\n        rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB);h,w=rgb.shape[:2]\\\\n        interp=cv2.INTER_AREA if h>self.size or w>self.size else cv2.INTER_LINEAR\\\\n        return cv2.resize(rgb,(self.size,self.size),interpolation=interp)\\\\n    def finalize(self,rgb):return (rgb.astype(np.float32)/255.-self.mean)/self.std\\\\n\\\\ndef load_rgb_sample(row):\\\\n    im=cv2.imread(row[\\\\\'path\\\\\'])\\\\n    if im is None:raise ValueError(\\\\\'Unreadable image: \\\\\'+row[\\\\\'path\\\\\'])\\\\n    if row.get(\\\\\'crop\\\\\') is not None:\\\\n        x1,y1,x2,y2=row[\\\\\'crop\\\\\'];h,w=im.shape[:2];dx=(x2-x1)*.05;dy=(y2-y1)*.05\\\\n        left,top=max(0,int((x1-dx)*w)),max(0,int((y1-dy)*h))\\\\n        right,bottom=min(w,max(left+1,int((x2+dx)*w))),min(h,max(top+1,int((y2+dy)*h)))\\\\n        im=im[top:bottom,left:right].copy()\\\\n    return im\\\\n\\\\nclass FrameDataset(torch.utils.data.Dataset):\\\\n    \\"\\"\\"Lab 1 cell 104 adapted to frame and human-box crop tasks.\\"\\"\\"\\\\n    def __init__(self,samples,size,train=False):\\\\n        self.samples=samples;self.pre=Preprocessor(size)\\\\n        self.tf=transforms.Compose([transforms.RandomHorizontalFlip(),\\\\n                  transforms.ColorJitter(brightness=.15,contrast=.15)]) if train else None\\\\n    def __len__(self):return len(self.samples)\\\\n    def __getitem__(self,i):\\\\n        row=self.samples[i];img=self.pre.prepare(load_rgb_sample(row))\\\\n        if self.tf:img=np.array(self.tf(Image.fromarray(img)))\\\\n        x=torch.from_numpy(np.ascontiguousarray(self.pre.finalize(img))).permute(2,0,1).float()\\\\n        return x,int(row[\\\\\'label\\\\\']),i\\\\nfrom torch import nn\\\\n\\\\nclass ConvNN(nn.Module):\\\\n    \\"\\"\\"Lab 2 cell 39 architecture; debug helpers removed, dimensions explicit.\\"\\"\\"\\\\n    def __init__(self,num_classes,size=224,num_filters=16,num_neurons=100):\\\\n        super().__init__()\\\\n        def block(a,b):return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU())\\\\n        self.block_1=block(3,num_filters)\\\\n        self.block_2=block(num_filters,2*num_filters)\\\\n        self.block_3=block(2*num_filters,4*num_filters)\\\\n        self.maxpool=nn.MaxPool2d(kernel_size=3,stride=2)\\\\n        self.dropout=nn.Dropout(.2);self.relu=nn.ReLU();self.flat=nn.Flatten()\\\\n        h=size\\\\n        for _ in range(3):h=(h-3)//2+1\\\\n        self.fc1=nn.Linear(4*num_filters*h*h,num_neurons)\\\\n        self.fc2=nn.Linear(num_neurons,num_neurons)\\\\n        self.output=nn.Linear(num_neurons,num_classes)\\\\n    def forward(self,x):\\\\n        x=self.maxpool(self.block_1(x));x=self.maxpool(self.block_2(x));x=self.maxpool(self.block_3(x))\\\\n        x=self.flat(self.dropout(x));x=self.relu(self.fc1(x));x=self.relu(self.fc2(x))\\\\n        return self.output(x)\\\\n\\\\nclass TransferNet(nn.Module):\\\\n    \\"\\"\\"Lab 3 cells 25/34: the same HF ResNet-50 and replaced classifier.\\"\\"\\"\\\\n    def __init__(self,classes,pretrained=True):\\\\n        super().__init__()\\\\n        from transformers import AutoModelForImageClassification,ResNetConfig,ResNetForImageClassification\\\\n        self.net=(AutoModelForImageClassification.from_pretrained(\\\\\'microsoft/resnet-50\\\\\') if pretrained\\\\n                  else ResNetForImageClassification(ResNetConfig()))\\\\n        self.net.classifier=nn.Sequential(nn.Flatten(start_dim=1),nn.Linear(self.net.config.hidden_sizes[-1],classes))\\\\n        self.net.config.num_labels=classes\\\\n    def forward(self,x):return self.net(x).logits\\\\n    @property\\\\n    def classifier(self):return self.net.classifier\\\\n\\\\ndef make_model(regime,classes,pretrained=True,size=224):\\\\n    return ConvNN(classes,size) if regime==\\\\\'scratch\\\\\' else TransferNet(classes,pretrained)\\\\n\\\\ndef freeze_backbone(model,freeze):\\\\n    for p in model.net.base_model.parameters():p.requires_grad=not freeze\\\\n    for p in model.classifier.parameters():p.requires_grad=True\\\\nif __name__==\\\\\'__main__\\\\\':\\\\n    import sys,json\\\\n    checkpoint,image=sys.argv[1:3]\\\\n    torch.set_num_threads(2)\\\\n    ckpt=torch.load(checkpoint,map_location=\\\\\'cpu\\\\\',weights_only=False)\\\\n    model=make_model(ckpt[\\\\\'regime\\\\\'],len(ckpt[\\\\\'classes\\\\\']),pretrained=False,size=ckpt[\\\\\'image_size\\\\\']).eval()\\\\n    model.load_state_dict(ckpt[\\\\\'state_dict\\\\\'])\\\\n    x,_,_=FrameDataset([{\\\\\'path\\\\\':image,\\\\\'label\\\\\':0,\\\\\'crop\\\\\':None}],ckpt[\\\\\'image_size\\\\\'])[0]\\\\n    with torch.no_grad():y=model(x.unsqueeze(0))\\\\n    assert y.shape==(1,len(ckpt[\\\\\'classes\\\\\'])) and torch.isfinite(y).all()\\\\n    print(json.dumps({\\\\\'shape\\\\\':list(y.shape),\\\\\'finite\\\\\':True}))\\\\n\'\\nfresh=[]\\nsample=next(r[\'path\'] for r in rows if r[\'split\']==\'train\')\\nfor row in comparison.to_dict(\'records\'):\\n    checkpoint=Path(row[\'folder\'])/\'best.pt\'\\n    try:\\n        proc=subprocess.run([sys.executable,\'-c\',PROBE_SOURCE,str(checkpoint),sample],capture_output=True,text=True,timeout=180)\\n        fresh.append({\'checkpoint\':str(checkpoint),\'sha256\':digest(checkpoint),\'passed\':proc.returncode==0,\\n                      \'stdout\':proc.stdout[-1000:],\'stderr\':proc.stderr[-2000:]})\\n    except subprocess.TimeoutExpired:\\n        fresh.append({\'checkpoint\':str(checkpoint),\'sha256\':digest(checkpoint),\'passed\':False,\'stderr\':\'timeout\'})\\nwrite_json(\'review/fresh_process_results.json\',fresh);display(pd.DataFrame(fresh)[[\'checkpoint\',\'passed\']])\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 6. روابط W&B ومساهمات الفريق", "metadata": {}}, {"cell_type": "code", "source": "links_path=Path(\'submission/wandb_links.csv\')\\nif not links_path.exists():comparison[[\'task\',\'regime\',\'train_run_id\',\'train_url\',\'test_run_id\',\'test_url\']].to_csv(links_path,index=False)\\nTEAM={\'member1\':\'\',\'member2\':\'\',\'member3\':\'\',\'member4\':\'\'}\\nNOTES={\'motivation\':\'\',\'data_findings\':\'\',\'model_comparison\':\'\',\'error_analysis\':\'\',\'data_improvement_plan\':\'\',\'limitations\':\'\',\'conclusion\':\'\'}\\nREPLAY={\'reviewer\':\'\',\'date\':\'\',\'notebook\':\'\',\'cells_executed\':\'\',\'evidence\':\'\'}\\nREVIEWER=\'\'\\nVISUAL_REVIEW_COMPLETE=False\\nREPOSITORY_URL=\'https://github.com/salehdev3/Capstone_5\'\\nwrite_json(\'submission/report_inputs.json\',{\'team\':TEAM,\'notes\':NOTES,\'replay\':REPLAY,\'repository\':REPOSITORY_URL})\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 7. قائمة اكتمال المتطلبات", "metadata": {}}, {"cell_type": "code", "source": "gaps=[]\\nautomatic=pd.read_csv(\'review/automatic_checks.csv\')\\nif not automatic.passed.all():gaps.append(\'Automatic integrity/metrics checks failed\')\\nreview=pd.read_csv(\'review/independent_audit.csv\').fillna(\'\')\\nvalid=review.label_correct.eq(\'yes\') & review.boxes_correct.isin([\'yes\',\'not applicable\']) & review.person_visible.eq(\'no\') & review.reviewer.str.strip().ne(\'\')\\nif not valid.all():gaps.append(\'Independent audit pending or flagged\')\\nissues=pd.read_csv(\'review/issues.csv\').fillna(\'\')\\nif not issues.empty and not (issues.status.eq(\'resolved\') & issues.resolution.str.strip().ne(\'\') & issues.evidence.str.strip().ne(\'\')).all():gaps.append(\'Unresolved issues\')\\nif len(fresh)!=6 or not all(r[\'passed\'] for r in fresh):gaps.append(\'Fresh-process inference\')\\nif not REVIEWER.strip() or not VISUAL_REVIEW_COMPLETE:gaps.append(\'Named visual review signoff\')\\nfor k,v in TEAM.items():\\n    if not v.strip():gaps.append(\'Team name: \'+k)\\nfor k,v in NOTES.items():\\n    if not v.strip():gaps.append(\'Report section: \'+k)\\nif not all(v.strip() for v in REPLAY.values()):gaps.append(\'Clean notebook replay evidence\')\\nfor member in [2,3]:\\n    p=Path(f\'runs/member{member}/ERROR_ANALYSIS.md\')\\n    if not p.exists() or \'TODO\' in p.read_text():gaps.append(f\'Actual error analysis member {member}\')\\ncard=Path(release[\'dataset_card\']).read_text()\\nif \'Pending:\' in card:gaps.append(\'Dataset Card team observations\')\\nblur=pd.read_csv(DATA/\'blur_review.csv\').fillna(\'\')\\nif not (blur.reviewer.str.strip().ne(\'\') & blur.decision.isin([\'keep\',\'flag\']) & blur.notes.str.strip().ne(\'\')).all():gaps.append(\'Blur review notes\')\\nlinks=pd.read_csv(links_path).fillna(\'\')\\nexpected={(t,r) for t in [\'triage\',\'species\'] for r in [\'scratch\',\'frozen\',\'full\']}\\nif len(links)!=6 or set(zip(links.task,links.regime))!=expected:gaps.append(\'Six W&B experiment rows\')\\nfor field in [\'train_url\',\'test_url\']:\\n    if not links[field].str.match(r\'^https://wandb\\\\.ai/[^/]+/[^/]+/runs/[^/]+\').all():gaps.append(\'W&B \'+field)\\nfor i in range(1,5):\\n    if not Path(f\'handoff/requirements_member{i}.txt\').exists():gaps.append(f\'Environment member {i}\')\\nNAMES=[\'Phase1_Final_Presentation.ipynb\']\\nfor name in NAMES:\\n    if not (Path(\'notebooks\')/name).exists():gaps.append(\'Notebook \'+name)\\nif not REPOSITORY_URL.startswith(\'https://github.com/\'):gaps.append(\'GitHub repository URL\')\\nSTATUS=\'READY_FOR_TEAM_REVIEW\' if not gaps else \'INCOMPLETE\'\\nprint(STATUS)\\nfor gap in gaps:print(\'-\',gap)\\nwrite_json(\'handoff/qa_release.json\',{\'schema\':2,\'dataset_hash\':DATA_HASH,\'status\':STATUS,\'gaps\':gaps,\\n \'reviewer\':REVIEWER,\'input_hashes\':{f\'member{i}\':digest(f\'handoff/member{i}_release.json\') for i in [2,3]},\\n \'scope\':\'data/metrics checks and fresh-process inference; full replay scope documented separately\'})\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 8. التقرير من نتائج التشغيل الفعلية", "metadata": {}}, {"cell_type": "code", "source": "def table(df):\\n    df=df.fillna(\'\');clean=lambda v:str(v).replace(\'|\',\'/\').replace(\'\\\\n\',\' \')\\n    return \'\\\\n\'.join([\'| \'+\' | \'.join(df.columns)+\' |\',\'| \'+\' | \'.join([\'---\']*len(df.columns))+\' |\']+\\n                     [\'| \'+\' | \'.join(clean(v) for v in row)+\' |\' for row in df.itertuples(index=False,name=None)])\\nsections=[\'# Project 5 — Phase 1 report\',\'Status: \'+STATUS,\\n \'Scope: M1–M3 only. Frame empty/animal classification and species human-box crop classification.\',\\n \'## Contributions\',str(TEAM),\'## Motivation\',NOTES[\'motivation\'] or \'[pending]\',\\n \'## Dataset Card\',Path(release[\'dataset_card\']).read_text(),NOTES[\'data_findings\'] or \'[pending]\',\\n \'## Acceptance criteria\',table(pd.DataFrame(acceptance)),\\n \'## Model comparison\',table(comparison.drop(columns=[\'folder\',\'train_url\',\'test_url\'])),\\n \'Selections above use validation macro-F1 only. Test is held out. Per-class reports and confidence intervals are included in results/.\',\\n NOTES[\'model_comparison\'] or \'[pending]\',\'## Error analysis\',NOTES[\'error_analysis\'] or \'[pending]\']\\nfor member in [2,3]:\\n    p=Path(f\'runs/member{member}/ERROR_ANALYSIS.md\')\\n    sections.append(p.read_text() if p.exists() else \'[missing analysis]\')\\nsections+=[\'## Data improvement plan\',NOTES[\'data_improvement_plan\'] or \'[pending]\',\\n \'## Limitations\',NOTES[\'limitations\'] or \'[pending]\',\\n \'The species task uses human-box crops. This does not evaluate a detector-classifier cascade. Camera-based confidence intervals use few sites and are descriptive.\',\\n \'## Reproducibility\',str(REPLAY),\'Repository: \'+(REPOSITORY_URL or \'[pending]\'),\\n \'## W&B\',table(links),\'## Conclusion\',NOTES[\'conclusion\'] or \'[pending]\',\\n \'## Outstanding evidence\',\'\\\\n\'.join(\'- \'+g for g in gaps) or \'No automated completeness gaps.\']\\nPath(\'submission/REPORT_DRAFT.md\').write_text(\'\\\\n\\\\n\'.join(sections))\\ndisplay(Markdown(Path(\'submission/REPORT_DRAFT.md\').read_text()))\\n", "metadata": {}}, {"cell_type": "markdown", "source": "## 9. تجهيز المستودع دون الصور أو الأوزان", "metadata": {}}, {"cell_type": "code", "source": "import shutil\\nrepo=Path(\'submission/repository\');repo.mkdir(exist_ok=True)\\ndef collect(source,destination):\\n    source=Path(source)\\n    if source.exists():\\n        target=repo/destination;target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(source,target)\\nfor name in NAMES:\\n    p=Path(\'notebooks\')/name\\n    if p.exists():\\n        n=read_json(p)\\n        for cell in n[\'cells\']:\\n            if cell[\'cell_type\']==\'code\':cell[\'outputs\']=[];cell[\'execution_count\']=None\\n        write_json(repo/\'notebooks\'/name,n)\\nfor p in Path(\'handoff\').glob(\'requirements_member*.txt\'):collect(p,Path(\'environments\')/p.name)\\nfor p in Path(\'handoff\').glob(\'runtime_member*.json\'):collect(p,Path(\'environments\')/p.name)\\ncollect(\'handoff/data_config.json\',\'config.json\')\\ncollect(release[\'dataset_card\'],\'DATASET_CARD.md\')\\nfor name in [\'comparison.csv\',\'acceptance.csv\',\'REPORT_DRAFT.md\',\'wandb_links.csv\',\'report_inputs.json\',\'triage_comparison.png\',\'species_comparison.png\']:\\n    collect(Path(\'submission\')/name,name)\\nfor name in [\'automatic_checks.csv\',\'independent_audit.csv\',\'issues.csv\',\'fresh_process_results.json\']:\\n    collect(Path(\'review\')/name,Path(\'review\')/name)\\nfor name in [\'class_balance_proportions.csv\',\'species_crop_support.csv\',\'audit_report.json\',\'blur_review.csv\']:\\n    collect(DATA/name,Path(\'data_evidence\')/name)\\nweights=[]\\nfor row in comparison.to_dict(\'records\'):\\n    folder=Path(row[\'folder\']);destination=Path(\'results\')/row[\'task\']/row[\'regime\']\\n    for name in [\'summary.json\',\'test_metrics.json\',\'history.csv\',\'policy.json\',\'confusion.png\',\'learning_loss.png\',\'learning_accuracy.png\',\'reliability.png\',\'validation_threshold_sweep.csv\']:\\n        collect(folder/name,destination/name)\\n    weights.append({\'path\':str(folder/\'best.pt\'),\'sha256\':digest(folder/\'best.pt\'),\'access\':\'private team storage\'})\\nwrite_json(repo/\'weights_manifest.json\',weights)\\n(repo/\'.gitignore\').write_text(\'data/\\\\nruns/\\\\nwandb/\\\\n*.pt\\\\n*.pth\\\\n*.jpg\\\\n*.jpeg\\\\n.env\\\\n__pycache__/\\\\n.ipynb_checkpoints/\\\\n\')\\n(repo/\'README.md\').write_text(\\"# Project 5 — Phase 1\\\\n\\\\nRun Phase1_Final_Presentation.ipynb in section order. Use the same Member1 dataset release for both training notebooks. Upload private result archives to Colab and set ARCHIVES in the import cell; no Drive mount or external code modules are required. Internet and Colab\'s torch/torchvision are required; use GPU for training.\\\\n\\\\nMember1 uses CCT20 from https://lila.science/datasets/caltech-camera-traps. Follow dataset licence and course non-redistribution policy. Do not commit images or raw archives. The Dataset Card records filtering and limitations.\\\\n\\\\nAll three notebooks embed identical OpenCV preprocessing. Split is stratified by camera groups; bursts stay together. One shared config defines all six experiments. Validation selects models; calibration chooses temperature/triage threshold; test is evaluated once.\\\\n\\\\nSee environments/ for the actual resolved package versions and Python/platform of each member. Install one role\'s environment in a compatible runtime. No single cross-platform lock is claimed. The initial install cells describe required packages; actual resolved pins are captured after installation.\\\\n\\\\nReport and metrics are from actual execution, with pending sections visible. Test accuracy intervals include camera bootstrap and Wilson (the latter assumes independent samples). Human audit, error discussion, clean replay and W&B sync require team action. Fresh-process inference alone is not full training reproduction.\\\\n\\\\nWeights remain private; weights_manifest.json provides hashes and paths. Repository notebooks have outputs stripped; keep executed copies privately for assessment. This repository covers M1–M3 only, without detector training or deployment.\\\\n\\")\\nassert not list(repo.rglob(\'*.pt\')) and not list(repo.rglob(\'*.jpg\'))\\nwrite_json(\'handoff/integration_release.json\',{\'schema\':2,\'scope\':\'Phase 1\',\'status\':STATUS,\'gaps\':gaps,\\n \'dataset_hash\':DATA_HASH,\'report_sha256\':digest(repo/\'REPORT_DRAFT.md\'),\'repository\':REPOSITORY_URL})\\narchive=shutil.make_archive(str(ROOT.parent/\'Phase1_Submission\'),\'zip\',root_dir=repo)\\nprint(\'Prepared:\',repo.resolve(),\'|\',archive)\\nprint(\'Submission archive:\', archive)\\n", "metadata": {}}]')
for cell in presentation_cells:
    if cell['cell_type']=='code':
        cell.update(outputs=[], execution_count=None)
presentation_document = dict(nbformat=4, nbformat_minor=4, metadata={}, cells=presentation_cells)
presentation_path = presentation_sources/'Phase1_Final_Presentation.ipynb'
presentation_text = json.dumps(presentation_document, ensure_ascii=False, indent=1)
if presentation_path.exists():
    assert presentation_path.read_text() == presentation_text, 'Existing presentation source differs; use a fresh project directory.'
else:
    presentation_path.write_text(presentation_text)


# 1. البيانات والتدقيق — M1

In [ ]:
from pathlib import Path
import os, sys, subprocess
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
"""Download official CCT20 metadata; no predicted boxes are used as labels."""
import collections
import hashlib
import html
import io
import json
from pathlib import Path
import random
import tarfile
import time
import urllib.parse
import urllib.request
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import imagehash


META_URL = 'https://storage.googleapis.com/public-datasets-lila/caltechcameratraps/eccv_18_annotations.tar.gz'
IMAGE_BASES = [
    'https://storage.googleapis.com/public-datasets-lila/caltech-unzipped/cct_images/',
    'https://lilawildlife.blob.core.windows.net/lila-wildlife/caltech-unzipped/cct_images/']
NON_ANIMALS = {'empty', 'car', 'vehicle', 'person', 'human', 'people'}



SEED = 42
MAX_IMAGES = 6000
AUDIT_IMAGES = 100
PROJECT = Path.cwd() if Path.cwd().name == 'CameraTrap_Phase1_Team' else Path.cwd()/'CameraTrap_Phase1_Team'
PROJECT.mkdir(exist_ok=True)
os.chdir(PROJECT)
DATA = Path('data/cct20_v1')
DATA.mkdir(parents=True, exist_ok=True)
Path('runs/data').mkdir(parents=True, exist_ok=True)
Path('handoff').mkdir(exist_ok=True)

def read_json(path):
    return json.loads(Path(path).read_text())
def write_json(path, value):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False))
def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()

cfg = {'seed': SEED, 'max_images': MAX_IMAGES, 'max_image_side': 1024,
       'audit_events': AUDIT_IMAGES, 'data_dir': str(DATA), 'output_dir': 'runs/data'}
assert not Path('handoff/dataset_release.json').exists(), 'هذا الإصدار صدر بالفعل. استخدم نتائجه أو ابدأ مجلدًا جديدًا؛ لا تعدله بعد تدريب زملائك.'
print('Project folder:', PROJECT)


## 2. قراءة التسميات

In [ ]:
def fetch(url, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size:
        return
    tmp = path.with_suffix(path.suffix+'.part')
    for attempt in range(3):
        try:
            with urllib.request.urlopen(url, timeout=45) as r, tmp.open('wb') as f:
                while chunk := r.read(1024*1024):
                    f.write(chunk)
            tmp.replace(path)
            return
        except Exception:
            tmp.unlink(missing_ok=True)
            if attempt == 2:
                raise
            time.sleep(attempt+1)

def read_metadata(root):
    archive = root/'metadata'/'eccv_18_annotations.tar.gz'
    fetch(META_URL, archive)
    images, annotations, categories = {}, {}, {}
    with tarfile.open(archive) as tar:
        for member in tar.getmembers():
            if not member.name.endswith('.json') or not member.isfile():
                continue
            # Read members in memory: never extract untrusted paths.
            d = json.load(tar.extractfile(member))
            for c in d['categories']:
                if c['id'] in categories and categories[c['id']] != c['name']:
                    raise ValueError('Conflicting category maps')
                categories[c['id']] = c['name']
            for im in d['images']:
                if im['id'] in images and images[im['id']] != im:
                    raise ValueError('Conflicting image metadata')
                images[im['id']] = im
            for a in d['annotations']:
                key = (a['image_id'], a['category_id'], tuple(a.get('bbox', [])))
                annotations[key] = a
    by_image = collections.defaultdict(list)
    for a in annotations.values():
        by_image[a['image_id']].append(a)
    rows, excluded = [], []
    for ident, im in sorted(images.items()):
        aa = by_image[ident]
        labels = sorted({categories[a['category_id']] for a in aa})
        if not labels or any(c in NON_ANIMALS-{'empty'} for c in labels):
            excluded.append({'id': ident, 'reason': 'non-animal-category-or-unlabelled', 'labels': labels})
            continue
        animal_labels = [s for s in labels if s not in NON_ANIMALS]
        if 'empty' in labels and animal_labels:
            excluded.append({'id': ident, 'reason': 'conflicting-empty-label'})
            continue
        boxes = []
        incomplete = False
        for a in aa:
            name = categories[a['category_id']]
            if name in NON_ANIMALS:
                continue
            if 'bbox' not in a:
                incomplete = True
                continue
            x, y, w, h = map(float, a['bbox'])
            W, H = im['width'], im['height']
            x1, y1 = max(0.,x/W), max(0.,y/H)
            x2, y2 = min(1.,(x+w)/W), min(1.,(y+h)/H)
            if x2 <= x1 or y2 <= y1:
                incomplete = True
                continue
            boxes.append({'species': name, 'xyxy': [x1,y1,x2,y2], 'annotation_source': 'human_CCT20'})
        rows.append(dict(id=ident, file_name=Path(im['file_name']).name,
                         camera=str(im['location']), event=str(im.get('seq_id') or ident),
                         timestamp=im.get('date_captured',im.get('datetime')),
                         original_width=im['width'], original_height=im['height'],
                         seq_num_frames=im.get('seq_num_frames'), labels=animal_labels,
                         animal=int(bool(animal_labels)), boxes=boxes,
                         incomplete_boxes=incomplete))
    summary = dict(source=META_URL, sha256=digest(archive), total_images=len(images),
                   unique_annotations=len(annotations), human_boxes=sum('bbox' in a for a in annotations.values()),
                   cameras=len({str(i['location']) for i in images.values()}),
                   categories=categories, eligible_frames=len(rows), excluded_frames=len(excluded),
                   empty_frames=sum(r['animal']==0 for r in rows),
                   note='CCT20 is curated and is not the full 243k-image dataset. Percent empty is recomputed.')
    write_json(root/'metadata_summary.json', summary)
    write_json(root/'metadata_exclusions.json', excluded)
    return rows

rows = read_metadata(DATA)
print("Eligible frames:", len(rows))
display(pd.DataFrame(rows)[["id", "camera", "event", "animal", "labels"]].head())

## 3. Stratified Group Split

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

def assign_sites(rows, seed):
    # A multi-species frame gets one joint stratum, not multiple independently split copies.
    y=np.array(['|'.join(sorted(r['labels'])) or 'empty' for r in rows])
    groups=np.array([r['camera'] for r in rows])
    def first_fold(indices, folds):
        splitter=StratifiedGroupKFold(n_splits=folds,shuffle=True,random_state=seed)
        a,b=next(splitter.split(np.zeros(len(indices)),y[indices],groups[indices]))
        return indices[a],indices[b]
    remaining,test=first_fold(np.arange(len(rows)),5)
    train,validation=first_fold(remaining,4)
    select,calibrate=first_fold(validation,2)
    site_map={}
    for name,ids in [('train',train),('val_select',select),('val_calibrate',calibrate),('test',test)]:
        for camera in set(groups[ids]):
            assert camera not in site_map
            site_map[camera]=name
    return site_map

def assert_disjoint(rows):
    for field in ['id', 'camera', 'event', 'pixel_sha256']:
        seen = {}
        for r in rows:
            if field not in r:
                continue
            key = r[field]
            if key in seen and seen[key] != r['split']:
                raise AssertionError(f'{field} leakage across splits')
            seen[key] = r['split']

site_map=assign_sites(rows,SEED)
for r in rows:r['split']=site_map[r['camera']]
assert_disjoint(rows)
write_json(DATA/'site_split.json',site_map)
display(pd.DataFrame(rows).groupby('split').agg(images=('id','count'),cameras=('camera','nunique')))


## 4. اختيار العينة مع الحفاظ على bursts

In [ ]:
def sample_events(rows, limit, seed):
    """Sample whole bursts within each location, proportionally to eligible frame counts."""
    if not limit or limit >= len(rows):
        return rows
    rng = random.Random(seed)
    selected = []
    for camera in sorted({r['camera'] for r in rows}):
        rr = [r for r in rows if r['camera']==camera]
        groups = collections.defaultdict(list)
        for r in rr:
            groups[r['event']].append(r)
        events = sorted(groups)
        rng.shuffle(events)
        budget = max(3, round(limit*len(rr)/len(rows)))
        chosen = []
        for event in events:
            chosen.extend(groups[event])
            if len(chosen) >= budget:
                break
        selected.extend(chosen)
    return selected

chosen = sample_events(rows, MAX_IMAGES, SEED)
write_json(DATA/'selected_ids.json', [r['id'] for r in chosen])
print('Selected images:', len(chosen))


## 5. تنزيل الصور وفحص الملفات

In [ ]:
def download_one(row, root, max_side):
    path = root/'images'/(row['id']+'.jpg')
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        err = None
        for base in IMAGE_BASES:
            try:
                with urllib.request.urlopen(base+urllib.parse.quote(row['file_name']),timeout=40) as r:
                    raw = r.read()
                im = Image.open(io.BytesIO(raw)).convert('RGB')
                im.thumbnail((max_side,max_side), Image.Resampling.LANCZOS)
                tmp = path.with_suffix('.tmp')
                im.save(tmp, format='JPEG',quality=95)
                tmp.replace(path)
                break
            except Exception as e:
                err = str(e)
        else:
            return None, {'id':row['id'],'event':row['event'],'reason':err}
    try:
        with Image.open(path) as src:
            im = src.convert('RGB'); im.load()
            small = np.array(im.resize((64,64)),dtype=float)
            row = dict(row, path=str(path.resolve()),width=im.width,height=im.height,
                       processed_file_md5=hashlib.md5(path.read_bytes()).hexdigest(),
                       pixel_sha256=hashlib.sha256(im.tobytes()).hexdigest(),
                       phash=str(imagehash.phash(im)),
                       grayscale_proxy=bool(np.mean(np.max(small,2)-np.min(small,2))<6))
            return row, None
    except Exception as e:
        return None, {'id':row['id'],'event':row['event'],'reason':str(e)}

from tqdm.auto import tqdm
good, failures = [], []
for row in tqdm(chosen):
    image_row, error = download_one(row, DATA, cfg['max_image_side'])
    if image_row is not None:
        image_row['path'] = str(DATA/'images'/(image_row['id']+'.jpg'))  # portable relative path
        good.append(image_row)
    if error is not None: failures.append(error)
write_json(DATA/'download_failures.json', failures)
print('Readable:', len(good), '| Failed:', len(failures))
assert len(failures) <= 0.02*len(chosen), 'فشل أكثر من 2%؛ أعد خلية التنزيل قبل المتابعة'
failed_events = {e['event'] for e in failures}
good = sorted([r for r in good if r['event'] not in failed_events], key=lambda r:r['id'])


### فحص الصور والألوان والجودة

In [ ]:
import cv2
inventory=[]
for r in good:
    im=Image.open(r['path']);rgb=np.asarray(im.convert('RGB'))
    small=cv2.resize(rgb,(224,224),interpolation=cv2.INTER_AREA)
    blur=float(cv2.Laplacian(cv2.cvtColor(small,cv2.COLOR_RGB2GRAY),cv2.CV_64F).var())
    inventory.append({'id':r['id'],'split':r['split'],'mode':im.mode,'width':im.width,'height':im.height,
      'brightness':float(small.mean()),'blur_score':blur,'path':r['path']})
inventory=pd.DataFrame(inventory);inventory.to_csv(DATA/'inventory.csv',index=False)
display(inventory.describe(include='all'))
lowest=inventory.query("split == 'train'").nsmallest(8,'blur_score').copy()
lowest['reviewer']='';lowest['decision']='';lowest['notes']=''
if not (DATA/'blur_review.csv').exists():lowest.to_csv(DATA/'blur_review.csv',index=False)
for r in lowest.to_dict('records'):
    print(r['id'],'blur:',round(r['blur_score'],2));im=Image.open(r['path']);im.thumbnail((420,280));display(im)
print('أكمل blur_review.csv: reviewer = اسمك، decision = keep أو flag، notes = السبب. flag للتوثيق فقط؛ لا حذف تلقائي.')


## 6. إزالة التكرار وفحص التسرب

In [ ]:
def near_duplicate_quarantine(rows, distance=2):
    # 64-bit pHash: four chunks guarantee a candidate bucket if Hamming distance <=2.
    buckets = collections.defaultdict(list)
    bad, pairs = set(), []
    for r in rows:
        h = int(r['phash'],16)
        candidates = {}
        for part in range(4):
            for old in buckets[(part,(h>>(part*16)) & 65535)]:
                candidates[old['id']] = old
        for old in candidates.values():
            if old['split']==r['split']:
                continue
            dist = (h ^ int(old['phash'],16)).bit_count()
            if dist <= distance:
                bad.update([r['event'],old['event']])
                pairs.append({'id_a':old['id'],'id_b':r['id'],'distance':dist})
        for part in range(4):
            buckets[(part,(h>>(part*16)) & 65535)].append(r)
    return [r for r in rows if r['event'] not in bad], pairs, sorted(bad)

In [ ]:
hashes=collections.defaultdict(list)
for r in good:hashes[r['pixel_sha256']].append(r)
excluded_events=set(); duplicate_ids=[];conflicting_duplicate_groups=[]
for group in hashes.values():
    if len(group)>1:
        if len({tuple(r['labels']) for r in group})>1:
            conflicting_duplicate_groups.append([r['id'] for r in group])
            excluded_events.update(r['event'] for r in group)
        elif len({r['split'] for r in group})>1:
            excluded_events.update(r['event'] for r in group)
        else:duplicate_ids.extend(r['id'] for r in group[1:])
cleaned=[r for r in good if r['event'] not in excluded_events and r['id'] not in set(duplicate_ids)]
cleaned,pairs,near_events=near_duplicate_quarantine(cleaned)
assert_disjoint(cleaned)
for split in ['train','val_select','val_calibrate','test']:
    if {r['animal'] for r in cleaned if r['split']==split}!={0,1}:
        raise ValueError(f'{split} lacks empty/animal examples after sampling. Increase max_images; do not adjust test based on results.')
labels=sorted({b['species'] for r in cleaned if r['split']=='train' for b in r['boxes']})
if len(labels)<2:raise ValueError('Need at least two training species')
write_json(DATA/'species.json',labels)
write_json(DATA/'manifest.json',cleaned)
write_json(DATA/'preparation_config.json',cfg)
write_json(DATA/'cleaning_report.json',dict(selected=len(chosen),downloaded=len(good),kept=len(cleaned),
    failures=len(failures),excluded_failed_events=len(failed_events),duplicate_ids=duplicate_ids,
    exact_cross_split_events=sorted(excluded_events),near_duplicate_pairs=pairs,near_quarantined_events=near_events,
    conflicting_duplicate_label_groups=conflicting_duplicate_groups,
    near_policy='Conservative pHash <=2 across partitions: both events quarantined; not asserted to be true duplicates.',
    corrupted_or_failed_files=failures,grayscale_proxy='RGB channel difference <6; proxy only, not verified infrared ground truth'))

print('Clean frames:', len(cleaned), '| Training species:', labels)

## 7. عرض الصور وتوزيع الفئات

In [ ]:
def gallery(path, rows, draw_boxes=False, titles=None):
    path=Path(path);thumbs=path.parent/(path.stem+'_images');thumbs.mkdir(exist_ok=True)
    cards=[]
    for i,r in enumerate(rows):
        im=Image.open(r['path']).convert('RGB')
        if draw_boxes:
            draw=ImageDraw.Draw(im)
            for b in r['boxes']:
                x1,y1,x2,y2=b['xyxy'];box=[x1*im.width,y1*im.height,x2*im.width,y2*im.height]
                draw.rectangle(box,outline='lime',width=3);draw.text((box[0],box[1]),b['species'],fill='yellow')
        im.thumbnail((480,360));name=f'{i:04d}.jpg';im.save(thumbs/name)
        title=titles[i] if titles else f'{r["id"]} | {r["split"]} | {r["labels"] or ["empty"]}'
        cards.append(f'<article><img src="{thumbs.name}/{name}"><p>{html.escape(str(title))}</p></article>')
    path.write_text('<!doctype html><meta charset="utf-8"><title>Local review gallery</title><style>body{font:14px sans-serif;background:#132127;color:white}main{display:grid;grid-template-columns:repeat(auto-fit,minmax(340px,1fr));gap:16px}article{background:#22343b;padding:10px}img{max-width:100%}p{overflow-wrap:anywhere}</style><h1>Private local review gallery</h1><p>Review before sharing; exclude or blur any identifiable people. Do not redistribute dataset images.</p><main>'+''.join(cards)+'</main>',encoding='utf-8')

def audit_templates(cfg, rows):
    root=Path(cfg['data_dir'])
    # A random event sample with one random frame per event. Do not claim a random-frame error rate.
    rng=random.Random(cfg['seed'])
    events=collections.defaultdict(list)
    for r in rows:
        if r['split']=='train':events[r['event']].append(r)
    selected=rng.sample(sorted(events),min(cfg['audit_events'],len(events)))
    audit=[]
    for event in selected:
        r=rng.choice(events[event]);audit.append(dict(id=r['id'],event=event,split=r['split'],
          expected_species=';'.join(r['labels']) or 'empty',label_correct='',boxes_correct='',
          identifiable_person='',reviewer='',notes=''))
    pd.DataFrame(audit).to_csv(root/'label_audit.csv',index=False)
    gallery(root/'audit_gallery.html',[next(r for r in rows if r['id']==a['id']) for a in audit],draw_boxes=True)
    # Site-wide label balance and crop support, without exposing exact coordinates.
    count_rows=[dict(split=r['split'],label=s) for r in rows for s in (r['labels'] or ['empty'])]
    pd.crosstab(pd.DataFrame(count_rows).label,pd.DataFrame(count_rows).split).to_csv(root/'class_counts.csv')
    crop_rows=[dict(split=r['split'],label=b['species']) for r in rows for b in r['boxes']]
    pd.crosstab(pd.DataFrame(crop_rows).label,pd.DataFrame(crop_rows).split).to_csv(root/'crop_counts.csv')
    gallery(root/'annotation_gallery.html',rng.sample(rows,min(48,len(rows))),draw_boxes=True)

if not (DATA/'label_audit.csv').exists():
    audit_templates(cfg, cleaned)
counts = pd.read_csv(DATA/'class_counts.csv', index_col=0)
display(counts)
counts.plot.bar(stacked=True, figsize=(12,4))
plt.ylabel('Label occurrences'); plt.tight_layout(); plt.show()
for row in [r for r in cleaned if r['split']=='train'][:4]:
    print(row['id'], row['labels'] or ['empty'])
    im=Image.open(row['path']); im.thumbnail((600,400)); display(im)


## 8. تدقيق بشري

In [ ]:
def audit_widget(cfg):
    import ipywidgets as w
    from IPython.display import display,clear_output
    from PIL import Image,ImageDraw
    root=Path(cfg['data_dir']);path=root/'label_audit.csv'
    df=pd.read_csv(path).fillna('');rows={r['id']:r for r in read_json(root/'manifest.json')}
    index=w.IntSlider(value=0,min=0,max=len(df)-1,description='Sample')
    correct=w.Dropdown(options=['','yes','no'],description='Label correct')
    boxes=w.Dropdown(options=['','yes','no','not applicable'],description='Boxes correct')
    person=w.Dropdown(options=['','yes','no'],description='Person visible')
    reviewer=w.Text(description='Reviewer');notes=w.Textarea(description='Notes')
    save=w.Button(description='Save and next',button_style='success');output=w.Output();status=w.Output()
    def show(change=None):
        row=df.iloc[index.value];r=rows[row.id]
        correct.value=str(row.label_correct);boxes.value=str(row.boxes_correct);person.value=str(row.identifiable_person)
        if row.reviewer:reviewer.value=str(row.reviewer)
        notes.value=str(row.notes)
        with output:
            clear_output(wait=True);print('Expected:',row.expected_species,'| sample:',row.id)
            im=Image.open(r['path']).convert('RGB');draw=ImageDraw.Draw(im)
            for b in r['boxes']:
                x1,y1,x2,y2=b['xyxy'];draw.rectangle([x1*im.width,y1*im.height,x2*im.width,y2*im.height],outline='lime',width=3)
            im.thumbnail((900,600));display(im)
    def persist(_):
        with status:
            clear_output()
            if not reviewer.value.strip() or not correct.value or not person.value or not boxes.value:
                print('Fill every decision and the reviewer name before saving.');return
        i=index.value
        for key,value in [('label_correct',correct.value),('boxes_correct',boxes.value),('identifiable_person',person.value),
                          ('reviewer',reviewer.value.strip()),('notes',notes.value)]:df.loc[i,key]=value
        df.to_csv(path,index=False)
        with status:print('Saved. Corrections/privacy flags need resolution before final training/report.')
        if i<len(df)-1:index.value=i+1
    index.observe(show,names='value');save.on_click(persist)
    display(w.VBox([index,output,correct,boxes,person,reviewer,notes,save,status]));show()

audit_widget(cfg)

## 9. معالجة مشاكل التدقيق وحساب حد الخطأ

In [ ]:
from scipy.stats import beta

def audit_report(cfg):
    root=Path(cfg['data_dir']);df=pd.read_csv(root/'label_audit.csv').fillna('')
    valid=(df.label_correct.astype(str).str.lower().isin(['yes','no']) &
           df.boxes_correct.astype(str).str.lower().isin(['yes','no','not applicable']) &
           df.identifiable_person.astype(str).str.lower().isin(['yes','no']) &
           df.reviewer.astype(str).str.strip().ne(''))
    reviewed=df[valid];n=len(reviewed);errors=int((reviewed.label_correct.astype(str).str.lower()=='no').sum())
    rows=read_json(root/'manifest.json');present_events={r['event'] for r in rows}
    flagged=reviewed[(reviewed.label_correct.astype(str).str.lower()=='no') |
                     (reviewed.boxes_correct.astype(str).str.lower()=='no') |
                     (reviewed.identifiable_person.astype(str).str.lower()=='yes')]
    upper=float(beta.ppf(.95,errors+1,n-errors)) if n and errors<n else (1. if n else None)
    result=dict(status='complete' if n==len(df) else 'pending',requested=len(df),reviewed=n,label_errors=errors,
      estimated_error_rate=errors/n if n else None,one_sided_95_upper_bound=upper,
      sampling_unit='random train event, then one random eligible frame; not all dataset frames',
      limitations='Binomial interval assumes independent sampled events. Same-camera correlation may make it optimistic.',
      no_review_is_not_zero_errors=True,
      privacy_flags=int(((reviewed.identifiable_person.astype(str).str.lower()=='yes') & reviewed.event.isin(present_events)).sum()),
      unresolved_flagged_events=len(set(flagged.event)&present_events))
    write_json(root/'audit_report.json',result);return result

def apply_audit(cfg):
    """Conservatively quarantine train events with human-confirmed problems; no guessed relabelling."""
    root=Path(cfg['data_dir']);out=Path(cfg['output_dir'])
    df=pd.read_csv(root/'label_audit.csv').fillna('')
    reviewed=df[df.reviewer.astype(str).str.strip().ne('')]
    flagged=reviewed[(reviewed.label_correct.astype(str).str.lower()=='no') |
                     (reviewed.boxes_correct.astype(str).str.lower()=='no') |
                     (reviewed.identifiable_person.astype(str).str.lower()=='yes')]
    bad=set(flagged.event);rows=read_json(root/'manifest.json')
    if not (bad & {r['event'] for r in rows}):
        print('No unresolved human-flagged events to quarantine.');return
    if list(out.glob('*/*/best.pt')):
        raise RuntimeError('Checkpoints already exist. Set a NEW output_dir in config before applying training-data corrections.')
    if any(r['split']!='train' and r['event'] in bad for r in rows):
        raise ValueError('Only train audit changes are allowed; held-out protocol is immutable.')
    cleaned=[r for r in rows if r['event'] not in bad]
    labels=sorted({b['species'] for r in cleaned if r['split']=='train' for b in r['boxes']})
    if len(labels)<2 or {r['animal'] for r in cleaned if r['split']=='train'}!={0,1}:
        raise ValueError('Audit exclusions remove required training support; collect more training events.')
    write_json(root/'audit_quarantine.json',dict(events=sorted(bad),frames_removed=len(rows)-len(cleaned),
        policy='Human-flagged events excluded. No automatic corrected species labels.'))
    write_json(root/'manifest.json',cleaned);write_json(root/'species.json',labels);assert_disjoint(cleaned)
    for name,values in [('class_counts',[dict(split=r['split'],label=s) for r in cleaned for s in (r['labels'] or ['empty'])]),
                        ('crop_counts',[dict(split=r['split'],label=b['species']) for r in cleaned for b in r['boxes']])]:
        frame=pd.DataFrame(values);pd.crosstab(frame.label,frame.split).to_csv(root/(name+'.csv'))
    print('Quarantined',len(rows)-len(cleaned),'training frames across',len(bad),'flagged events.')

audit = audit_report(cfg)
assert audit['status']=='complete', 'أكمل نموذج التدقيق أولاً'
apply_audit(cfg)
audit = audit_report(cfg)
assert audit['unresolved_flagged_events']==0
rows = read_json(DATA/'manifest.json')
print(audit)


### التحقق النهائي من Class Stratification

In [ ]:
splits=['train','val_select','val_calibrate','test']
all_labels=sorted({label for r in rows for label in (r['labels'] or ['empty'])})
balance=[]
for label in all_labels:
    overall=sum(label in (r['labels'] or ['empty']) for r in rows)/len(rows)
    for split in splits:
        subset=[r for r in rows if r['split']==split]
        count=sum(label in (r['labels'] or ['empty']) for r in subset)
        rate=count/len(subset) if subset else 0
        balance.append({'label':label,'split':split,'count':count,'frames':len(subset),
                        'percent':100*rate,'overall_percent':100*overall,'difference_pp':100*(rate-overall)})
balance_report=pd.DataFrame(balance);balance_report.to_csv(DATA/'class_balance_proportions.csv',index=False)
display(balance_report)
balance_report.pivot(index='label',columns='split',values='percent').plot.bar(figsize=(12,5))
plt.ylabel('Frames containing label (%)');plt.tight_layout();plt.show()
counts=pd.DataFrame([{'species':b['species'],'split':r['split']} for r in rows for b in r['boxes']])
support=pd.crosstab(counts.species,counts.split).reindex(columns=splits,fill_value=0)
support.to_csv(DATA/'species_crop_support.csv');display(support)
print('Missing class/split combinations:');display(balance_report.query('count == 0')[['label','split']])
assert_disjoint(rows)
print('PASS: cameras, events and exact pixel duplicates are disjoint.')


## 10. Preprocessor + Dataset + DataLoader

In [ ]:
import cv2, numpy as np, torch
from PIL import Image
from torchvision import transforms
from torch.utils.data import DataLoader
class Preprocessor:
    """Lab 1 cell 72: prepare uint8 RGB, augment, finalize and CHW tensor."""
    def __init__(self,size):
        self.size=size
        self.mean=np.array([.485,.456,.406],np.float32)
        self.std=np.array([.229,.224,.225],np.float32)
    def prepare(self,bgr):
        rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB);h,w=rgb.shape[:2]
        interp=cv2.INTER_AREA if h>self.size or w>self.size else cv2.INTER_LINEAR
        return cv2.resize(rgb,(self.size,self.size),interpolation=interp)
    def finalize(self,rgb):return (rgb.astype(np.float32)/255.-self.mean)/self.std

def load_rgb_sample(row):
    im=cv2.imread(row['path'])
    if im is None:raise ValueError('Unreadable image: '+row['path'])
    if row.get('crop') is not None:
        x1,y1,x2,y2=row['crop'];h,w=im.shape[:2];dx=(x2-x1)*.05;dy=(y2-y1)*.05
        left,top=max(0,int((x1-dx)*w)),max(0,int((y1-dy)*h))
        right,bottom=min(w,max(left+1,int((x2+dx)*w))),min(h,max(top+1,int((y2+dy)*h)))
        im=im[top:bottom,left:right].copy()
    return im

class FrameDataset(torch.utils.data.Dataset):
    """Lab 1 cell 104 adapted to frame and human-box crop tasks."""
    def __init__(self,samples,size,train=False):
        self.samples=samples;self.pre=Preprocessor(size)
        self.tf=transforms.Compose([transforms.RandomHorizontalFlip(),
                  transforms.ColorJitter(brightness=.15,contrast=.15)]) if train else None
    def __len__(self):return len(self.samples)
    def __getitem__(self,i):
        row=self.samples[i];img=self.pre.prepare(load_rgb_sample(row))
        if self.tf:img=np.array(self.tf(Image.fromarray(img)))
        x=torch.from_numpy(np.ascontiguousarray(self.pre.finalize(img))).permute(2,0,1).float()
        return x,int(row['label']),i

train_rows=[dict(r,label=r['animal'],crop=None) for r in rows if r['split']=='train']
train_dataset=FrameDataset(train_rows,224,train=True)
eval_dataset=FrameDataset(train_rows,224,train=False)
train_loader=DataLoader(train_dataset,batch_size=8,shuffle=True,generator=torch.Generator().manual_seed(SEED))
images,targets,_=next(iter(train_loader))
print('N,C,H,W:',images.shape,'| labels:',targets.tolist())
fig,axes=plt.subplots(2,4,figsize=(12,6))
for x,y,ax in zip(images,targets,axes.flat):
    rgb=x.permute(1,2,0).numpy()*[.229,.224,.225]+[.485,.456,.406]
    ax.imshow(np.clip(rgb,0,1));ax.set_title(['empty','animal'][int(y)]);ax.axis('off')
plt.tight_layout();plt.show()


### قبل وبعد

In [ ]:
sample=next(r for r in rows if r['split']=='train' and r['animal'])
bgr=cv2.imread(sample['path']);pre=Preprocessor(224)
rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB);prepared=pre.prepare(bgr)
augmented=np.array(train_dataset.tf(Image.fromarray(prepared)))
fig,axes=plt.subplots(1,3,figsize=(12,4))
for ax,img,title in zip(axes,[rgb,prepared,augmented],['Downloaded RGB','Resize 224','Train augmentation']):
    ax.imshow(img);ax.set_title(title);ax.axis('off')
plt.show()
for ch,color in enumerate(['red','green','blue']):
    plt.hist(prepared[:,:,ch].ravel(),bins=32,alpha=.35,color=color,label=color)
plt.legend();plt.title('Train sample channel histogram');plt.show()


## 11. مثال قصاصة لتصنيف النوع

In [ ]:
row = next(r for r in rows if r['split']=='train' and r['boxes'])
box = row['boxes'][0]
im = Image.open(row['path']).convert('RGB')
x1,y1,x2,y2 = box['xyxy']
dx,dy=(x2-x1)*.05,(y2-y1)*.05
crop=im.crop((max(0,int((x1-dx)*im.width)),max(0,int((y1-dy)*im.height)),
              min(im.width,int((x2+dx)*im.width)),min(im.height,int((y2+dy)*im.height))))
print('Species:', box['species']); display(crop)


## 12. Dataset Card — توثيق البيانات

In [ ]:
TEAM_NOTES = ""  # اكتب ملاحظاتك الفعلية عن البيانات
counts = pd.read_csv(DATA/'class_counts.csv', index_col=0)
cleaning = read_json(DATA/'cleaning_report.json')
summary = pd.DataFrame(rows).groupby('split').agg(frames=('id','count'),cameras=('camera','nunique'))
card = f"""# Dataset Card — CCT20 / Phase 1
Source: https://lila.science/datasets/caltech-camera-traps
Citation: Beery, Van Horn and Perona, Recognition in Terra Incognita, ECCV 2018.
Licence: CDLA-Permissive; follow course policy: do not redistribute images or commit them to Git.

## Tasks
Full-frame empty/animal classification; species classification on human-box crops.
No predicted boxes are used as ground truth. Positive frames without boxes remain in triage only.
Human/vehicle-labelled, conflicting or unlabelled frames are excluded by metadata.

## Acquisition and cleaning
Selected: {cleaning['selected']}; retained after human audit: {len(rows)}.
Download failures: {cleaning['failures']}; exact duplicate IDs removed: {len(cleaning['duplicate_ids'])}.
Near-duplicate pairs across splits: {len(cleaning['near_duplicate_pairs'])}.
Image maximum side: 1024; model input: 224 x 224; seed: {SEED}.
Corrupt/failed-event and duplicate exclusions are recorded in cleaning_report.json.

## Split
StratifiedGroupKFold by camera, with joint image-label strata. Camera counts and class proportions below are measured, not assumed.
Whole bursts stay within one split; exact duplicate and event leakage checks pass.
Group constraints approximate class stratification; this is not the official CCT benchmark split.
{summary.to_string()}

## Long-tailed class distribution (frame label occurrences)
{counts.to_string()}
Species crop counts: crop_counts.csv. Rare species may have little or zero support in held-out locations.
Training vocabulary is built from train crops only; unseen held-out species are reported by Member 2.

## Final stratification check
{balance_report.to_string(index=False)}
Rates count label presence per frame; multi-label class percentages need not sum to 100.
See class_balance_proportions.csv and species_crop_support.csv. Camera constraints can prevent exact balance.

## Label audit
Reviewed: {audit['reviewed']} / {audit['requested']}; observed label errors: {audit['label_errors']}.
One-sided 95% upper error bound: {audit['one_sided_95_upper_bound']}.
Sampling: one frame per randomly sampled train event. Same-camera dependence may make this bound optimistic.
Flagged training events are quarantined; original audit decisions remain recorded.

## Preprocessing
RGB, resize 224, ImageNet normalization. Random flip and brightness/contrast changes on train only.
The identical OpenCV Preprocessor and augmentation code is embedded in Members 1, 2 and 3.

## Limitations and privacy
Curated US wildlife data; no claim of Saudi generalisation. The empty fraction must be measured on this subset.
Grayscale is an infrared proxy, not verified nighttime. Blur is a review flag; no image is discarded solely for low Laplacian variance. No GPS publication; keep photos and review galleries private.
Test accuracy confidence intervals belong to Member 2 after training, not this untrained data notebook.

## Team observations
{TEAM_NOTES or 'Pending: add actual observations before submission.'}
"""
Path('runs/data/DATASET_CARD.md').write_text(card)
display(Markdown(card))


## 13. تثبيت إصدار البيانات والإعداد المشترك

In [ ]:
blur_check=pd.read_csv(DATA/'blur_review.csv').fillna('')
assert len(blur_check) and blur_check['reviewer'].str.strip().ne('').all() and blur_check['decision'].isin(['keep','flag']).all() and blur_check['notes'].str.strip().ne('').all(), 'أكمل blur_review.csv: reviewer وdecision = keep أو flag؛ flag للتوثيق ولا يحذف الصورة'
assert TEAM_NOTES.strip(), 'أضف ملاحظاتك الفعلية في Dataset Card وأعد تشغيل خليتها'
training_defaults = {'project': 'camera-trap-phase1', 'backbone': 'microsoft/resnet-50', 'seed': 42, 'profile': 'full', 'data_dir': 'data/cct20_v1', 'output_dir': 'runs/data', 'max_images': 6000, 'download_workers': 12, 'max_image_side': 1024, 'image_size': 224, 'batch_size': 32, 'workers': 2, 'epochs': 12, 'patience': 4, 'warmup_epochs': 2, 'lr_scratch': 0.001, 'lr_frozen': 0.001, 'lr_full': 0.0001, 'weight_decay': 0.0001, 'label_smoothing': 0.05, 'class_weighted_loss': True, 'wandb_mode': 'offline', 'target_animal_recall': 0.99, 'species_review_confidence': 0.6, 'bootstrap_repeats': 1000, 'audit_events': 100, 'acceptance': {'animal_recall_min': 0.99, 'empty_removal_min': 0.2, 'species_macro_f1_min': 0.65, 'species_ece_max': 0.1}}
training_defaults.update(cfg)
training_defaults.update(contract_version=2,preprocessing_sha256='829fef7a32c1157ffe4da29da42026105afc0266ed647e562127bcc416697937',warmup_epochs=0,weight_decay=0.0,label_smoothing=0.0,class_weighted_loss=False,optimizer='Adam',wandb_mode='offline')
write_json('handoff/preprocessing.json',{'sha256':'829fef7a32c1157ffe4da29da42026105afc0266ed647e562127bcc416697937','input_size':224,'library':'OpenCV + torchvision','train_only':['horizontal_flip','brightness_contrast']})
write_json('handoff/data_config.json', training_defaults)
write_json(DATA/'preparation_config.json', training_defaults)
Path('handoff/requirements_member1.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
write_json('handoff/runtime_member1.json', {'python':sys.version, 'platform':sys.platform})
files=[DATA/name for name in ['manifest.json','species.json','site_split.json','audit_report.json',
 'label_audit.csv','preparation_config.json','cleaning_report.json','class_counts.csv','crop_counts.csv','metadata_summary.json','class_balance_proportions.csv','species_crop_support.csv','inventory.csv','blur_review.csv']]
files += [Path('handoff/data_config.json'),Path('runs/data/DATASET_CARD.md'),Path('handoff/preprocessing.json')]
write_json('handoff/dataset_release.json', {'schema':2,'scope':'Phase 1 M1-M3',
 'data_dir':str(DATA),'audit_complete':True,'files':{str(f):digest(f) for f in files},
 'owner':'member1','source':'CCT20 human annotations','dataset_card':'runs/data/DATASET_CARD.md'})
print('Member 1 outputs are ready in:', PROJECT)


# 2. الشبكة المدربة من الصفر — M2

In [ ]:
from pathlib import Path
import os,json,hashlib,sys,subprocess,platform,random
import numpy as np,pandas as pd
from IPython.display import display,Markdown
ROOT=Path.cwd() if Path.cwd().name=='CameraTrap_Phase1_Team' else Path.cwd()/'CameraTrap_Phase1_Team'
ROOT.mkdir(exist_ok=True);os.chdir(ROOT)
for name in ['handoff','runs','review','submission','docs']:Path(name).mkdir(exist_ok=True)
def read_json(p):return json.loads(Path(p).read_text())
def write_json(p,v):
    Path(p).parent.mkdir(parents=True,exist_ok=True)
    Path(p).write_text(json.dumps(v,indent=2,ensure_ascii=False,allow_nan=False))
def digest(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
def environment(member):
    Path(f'handoff/requirements_member{member}.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
    write_json(f'handoff/runtime_member{member}.json',{'python':sys.version,'platform':platform.platform()})
print('Workspace:',ROOT)


In [ ]:
release=read_json('handoff/dataset_release.json')
assert release['schema']==2 and release['audit_complete'], 'استخدم إصدار Member 1 الجديد بعد اكتمال التدقيق'
for p,h in release['files'].items():assert Path(p).is_file() and digest(p)==h, f'Changed/missing: {p}'
DATA=Path(release['data_dir']);rows=read_json(DATA/'manifest.json')
assert rows and all(Path(r['path']).exists() for r in rows)
DATA_HASH=digest(DATA/'manifest.json')
CFG=read_json('handoff/data_config.json')
assert CFG['preprocessing_sha256']=='829fef7a32c1157ffe4da29da42026105afc0266ed647e562127bcc416697937'
print('Frames:',len(rows),'| split sizes:',pd.Series([r['split'] for r in rows]).value_counts().to_dict())


In [ ]:
import cv2, numpy as np, torch
from PIL import Image
from torchvision import transforms
from torch.utils.data import DataLoader
class Preprocessor:
    """Lab 1 cell 72: prepare uint8 RGB, augment, finalize and CHW tensor."""
    def __init__(self,size):
        self.size=size
        self.mean=np.array([.485,.456,.406],np.float32)
        self.std=np.array([.229,.224,.225],np.float32)
    def prepare(self,bgr):
        rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB);h,w=rgb.shape[:2]
        interp=cv2.INTER_AREA if h>self.size or w>self.size else cv2.INTER_LINEAR
        return cv2.resize(rgb,(self.size,self.size),interpolation=interp)
    def finalize(self,rgb):return (rgb.astype(np.float32)/255.-self.mean)/self.std

def load_rgb_sample(row):
    im=cv2.imread(row['path'])
    if im is None:raise ValueError('Unreadable image: '+row['path'])
    if row.get('crop') is not None:
        x1,y1,x2,y2=row['crop'];h,w=im.shape[:2];dx=(x2-x1)*.05;dy=(y2-y1)*.05
        left,top=max(0,int((x1-dx)*w)),max(0,int((y1-dy)*h))
        right,bottom=min(w,max(left+1,int((x2+dx)*w))),min(h,max(top+1,int((y2+dy)*h)))
        im=im[top:bottom,left:right].copy()
    return im

class FrameDataset(torch.utils.data.Dataset):
    """Lab 1 cell 104 adapted to frame and human-box crop tasks."""
    def __init__(self,samples,size,train=False):
        self.samples=samples;self.pre=Preprocessor(size)
        self.tf=transforms.Compose([transforms.RandomHorizontalFlip(),
                  transforms.ColorJitter(brightness=.15,contrast=.15)]) if train else None
    def __len__(self):return len(self.samples)
    def __getitem__(self,i):
        row=self.samples[i];img=self.pre.prepare(load_rgb_sample(row))
        if self.tf:img=np.array(self.tf(Image.fromarray(img)))
        x=torch.from_numpy(np.ascontiguousarray(self.pre.finalize(img))).permute(2,0,1).float()
        return x,int(row['label']),i

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    import torch
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def artifact_context(cfg, task):
    return dict(training_implementation=2, task=task, profile=cfg['profile'], image_size=cfg['image_size'],
                seed=cfg['seed'], manifest_sha256=digest(Path(cfg['data_dir'])/'manifest.json'),
                split_policy='camera-disjoint train / val_select / val_calibrate / test',
                preprocessing_sha256=cfg['preprocessing_sha256'],
                training_config={k:cfg[k] for k in ['optimizer','backbone','batch_size','epochs','patience','warmup_epochs',
                    'lr_scratch','lr_frozen','lr_full','weight_decay','label_smoothing','class_weighted_loss']})

def samples_for_task(rows,task,classes):
    samples=[]
    for r in rows:
        if task=='triage':
            samples.append(dict(r,label=r['animal'],sample_id=r['id'],crop=None))
        else:
            for i,b in enumerate(r['boxes']):
                samples.append(dict(r,label=classes.index(b['species']) if b['species'] in classes else -1,
                                    sample_id=f'{r["id"]}_{i}',crop=b['xyxy'],species=b['species']))
    return samples

## 2. إعدادات مشتركة وتجارب W&B

In [ ]:
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Device:',DEVICE)
if DEVICE=='cpu':print('CPU بطيء للتدريب الكامل؛ اختر GPU في Colab')
WANDB_MODE='offline'  # online لحساب W&B؛ لا تكتب المفتاح في الكود
import wandb
if WANDB_MODE=='online':wandb.login()
REGIMES=['scratch']
OUT=Path('runs/member2');OUT.mkdir(parents=True,exist_ok=True)
CFG['output_dir']=str(OUT)
assert CFG['optimizer']=='Adam'
write_json(OUT/'config.json',CFG)
environment(2)


## 3. النموذج وforward

In [ ]:
from torch import nn

class ConvNN(nn.Module):
    """Lab 2 cell 39 architecture; debug helpers removed, dimensions explicit."""
    def __init__(self,num_classes,size=224,num_filters=16,num_neurons=100):
        super().__init__()
        def block(a,b):return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU())
        self.block_1=block(3,num_filters)
        self.block_2=block(num_filters,2*num_filters)
        self.block_3=block(2*num_filters,4*num_filters)
        self.maxpool=nn.MaxPool2d(kernel_size=3,stride=2)
        self.dropout=nn.Dropout(.2);self.relu=nn.ReLU();self.flat=nn.Flatten()
        h=size
        for _ in range(3):h=(h-3)//2+1
        self.fc1=nn.Linear(4*num_filters*h*h,num_neurons)
        self.fc2=nn.Linear(num_neurons,num_neurons)
        self.output=nn.Linear(num_neurons,num_classes)
    def forward(self,x):
        x=self.maxpool(self.block_1(x));x=self.maxpool(self.block_2(x));x=self.maxpool(self.block_3(x))
        x=self.flat(self.dropout(x));x=self.relu(self.fc1(x));x=self.relu(self.fc2(x))
        return self.output(x)

class TransferNet(nn.Module):
    """Lab 3 cells 25/34: the same HF ResNet-50 and replaced classifier."""
    def __init__(self,classes,pretrained=True):
        super().__init__()
        from transformers import AutoModelForImageClassification,ResNetConfig,ResNetForImageClassification
        self.net=(AutoModelForImageClassification.from_pretrained('microsoft/resnet-50') if pretrained
                  else ResNetForImageClassification(ResNetConfig()))
        self.net.classifier=nn.Sequential(nn.Flatten(start_dim=1),nn.Linear(self.net.config.hidden_sizes[-1],classes))
        self.net.config.num_labels=classes
    def forward(self,x):return self.net(x).logits
    @property
    def classifier(self):return self.net.classifier

def make_model(regime,classes,pretrained=True,size=224):
    return ConvNN(classes,size) if regime=='scratch' else TransferNet(classes,pretrained)

def freeze_backbone(model,freeze):
    for p in model.net.base_model.parameters():p.requires_grad=not freeze
    for p in model.classifier.parameters():p.requires_grad=True

In [ ]:
classes=['empty','animal']
example_model=make_model(REGIMES[0],len(classes),size=CFG['image_size'])
if REGIMES[0]!='scratch':freeze_backbone(example_model,True)
print(example_model)
print('Trainable parameters:',sum(p.numel() for p in example_model.parameters() if p.requires_grad))
example_model.eval()
with torch.no_grad():print('Forward shape:',example_model(torch.zeros(2,3,CFG['image_size'],CFG['image_size'])).shape)
del example_model


## 4. المقاييس والمعايرة

In [ ]:
import numpy as np
from scipy.special import softmax, logsumexp
from scipy.optimize import minimize_scalar
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score


def ece(y, probabilities, bins=15):
    y=np.asarray(y);p=np.asarray(probabilities)
    if len(y)==0:return None
    conf=p.max(1);correct=p.argmax(1)==y
    indices=np.minimum((conf*bins).astype(int),bins-1)
    return float(sum(np.mean(indices==i)*abs(correct[indices==i].mean()-conf[indices==i].mean())
                     for i in range(bins) if np.any(indices==i)))


def fit_temperature(logits,y):
    logits=np.asarray(logits);y=np.asarray(y,dtype=int)
    if not len(y):raise ValueError('No known-class calibration examples')
    def nll(log_t):
        z=logits/np.exp(log_t)
        return float(np.mean(logsumexp(z,axis=1)-z[np.arange(len(y)),y]))
    result=minimize_scalar(nll,bounds=(np.log(0.25),np.log(10)),method='bounded')
    return float(np.exp(result.x))


def wilson(k,n,z=1.959963984540054):
    if n==0:return [None,None]
    p=k/n;d=1+z*z/n
    centre=(p+z*z/(2*n))/d
    half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return [float(max(0,centre-half)),float(min(1,centre+half))]


def triage_metrics(y, animal_scores, threshold):
    y=np.asarray(y);pred=np.asarray(animal_scores)>=threshold
    pos=y==1;neg=y==0
    tp=int(np.sum(pred & pos));fn=int(np.sum(~pred & pos));tn=int(np.sum(~pred & neg));fp=int(np.sum(pred & neg))
    return dict(threshold=float(threshold),animal_recall=float(tp/pos.sum()) if pos.any() else None,
                empty_removal_rate=float(tn/neg.sum()) if neg.any() else None,
                animal_frames_lost=fn,animal_frames=int(pos.sum()),empty_frames=int(neg.sum()),
                removed_frames=tn+fn,tp=tp,tn=tn,fp=fp,fn=fn,
                animal_recall_wilson95_independence_assumption=wilson(tp,int(pos.sum())))


def choose_threshold(y, scores, target=0.99):
    if set(np.asarray(y).tolist())!={0,1}:raise ValueError('Calibration needs empty and animal labels')
    candidates=np.unique(np.r_[0.,scores,1.])
    feasible=[triage_metrics(y,scores,t) for t in candidates]
    feasible=[r for r in feasible if r['animal_recall']>=target]
    best=max(feasible,key=lambda r:(r['empty_removal_rate'],r['threshold']))
    best['selection_split']='val_calibrate'
    best['recall_target_is_empirical_not_guaranteed']=True
    return best


def classification_metrics(y,probs,classes):
    y=np.asarray(y);probs=np.asarray(probs);pred=probs.argmax(1);n=len(classes)
    support=np.bincount(y,minlength=n);supported=np.flatnonzero(support)
    report=classification_report(y,pred,labels=list(range(n)),target_names=classes,output_dict=True,zero_division=0)
    k=min(3,n)
    return dict(n=len(y),accuracy=float(accuracy_score(y,pred)),
       macro_f1_all_training_classes=float(f1_score(y,pred,labels=list(range(n)),average='macro',zero_division=0)),
       macro_f1_supported_classes=float(f1_score(y,pred,labels=supported,average='macro',zero_division=0)),
       top_k=k,top_k_accuracy=float(np.mean([label in indices for label,indices in zip(y,np.argsort(probs,axis=1)[:,-k:])])),
       ece=ece(y,probs),per_class=report,confusion_matrix=confusion_matrix(y,pred,labels=list(range(n))).tolist(),
       classes_absent_from_this_split=[classes[i] for i in range(n) if support[i]==0])


def camera_bootstrap(y,probs,cameras,repeats=1000,seed=42,threshold=None):
    """Resample whole test cameras; preserve dependence among frames/crops within camera."""
    y=np.asarray(y);p=np.asarray(probs);c=np.asarray(cameras);sites=np.unique(c)
    if len(sites)<2:return {'status':'not_estimable_fewer_than_two_cameras'}
    rng=np.random.default_rng(seed);acc=[];f1=[];rec=[];empty=[]
    for _ in range(repeats):
        idx=np.concatenate([np.flatnonzero(c==site) for site in rng.choice(sites,len(sites),replace=True)])
        yy=y[idx];pp=p[idx]
        acc.append(float(np.mean(pp.argmax(1)==yy)))
        f1.append(float(f1_score(yy,pp.argmax(1),labels=range(p.shape[1]),average='macro',zero_division=0)))
        if threshold is not None:
            m=triage_metrics(yy,pp[:,1],threshold)
            if m['animal_recall'] is not None:rec.append(m['animal_recall'])
            if m['empty_removal_rate'] is not None:empty.append(m['empty_removal_rate'])
    ci=lambda v:np.quantile(v,[.025,.975]).tolist() if v else [None,None]
    out=dict(unit='camera',cameras=len(sites),repeats=repeats,accuracy95=ci(acc),macro_f1_95=ci(f1),
             caveat='Intervals are descriptive and unstable when there are few held-out cameras.')
    if threshold is not None:out.update(animal_recall95=ci(rec),empty_removal95=ci(empty))
    return out


## 5. حلقة التدريب — forward → loss → backward → step

In [ ]:
from scipy.special import softmax
import time,wandb

def loader(samples,train=False):
    gen=torch.Generator().manual_seed(CFG['seed'])
    return DataLoader(FrameDataset(samples,CFG['image_size'],train),batch_size=CFG['batch_size'],
                      shuffle=train,num_workers=0,generator=gen)

def predict(model,data):
    model.eval();all_logits=[];all_y=[]
    with torch.no_grad():
        for images,labels,_ in data:
            all_logits.append(model(images.to(DEVICE)).cpu().numpy());all_y.extend(labels.tolist())
    assert all_logits, 'No known-class samples in this split'
    return np.concatenate(all_logits),np.array(all_y,dtype=int)

def train_model(task,regime):
    seed_everything(CFG['seed'])
    classes=['empty','animal'] if task=='triage' else read_json(DATA/'species.json')
    samples=samples_for_task(rows,task,classes)
    train=[r for r in samples if r['split']=='train' and r['label']>=0]
    val=[r for r in samples if r['split']=='val_select' and r['label']>=0]
    calibrate=[r for r in samples if r['split']=='val_calibrate' and r['label']>=0]
    assert train and val and calibrate
    folder=OUT/task/regime;folder.mkdir(parents=True,exist_ok=True)
    context=artifact_context(CFG,task)
    if (folder/'summary.json').exists():
        saved=read_json(folder/'summary.json');assert saved['context']==context
        assert saved['checkpoint_sha256']==digest(folder/'best.pt')
        print('Reusing completed run:',task,regime);return saved
    assert not (folder/'best.pt').exists(), 'تدريب جزئي موجود؛ استخدم إصدار تجربة جديدًا لتجنب استبداله'
    model=make_model(regime,len(classes),size=CFG['image_size']).to(DEVICE)
    if regime!='scratch':freeze_backbone(model,regime=='frozen')
    assert CFG['optimizer']=='Adam', 'Only Adam is implemented'
    weights=None
    if CFG['class_weighted_loss']:
        counts=np.bincount([r['label'] for r in train],minlength=len(classes))
        assert (counts>0).all(), 'Every training class must have samples'
        weights=torch.tensor(len(train)/(len(classes)*counts),dtype=torch.float32,device=DEVICE)
    criterion=nn.CrossEntropyLoss(weight=weights,label_smoothing=CFG['label_smoothing'])
    optimizer=torch.optim.Adam([p for p in model.parameters() if p.requires_grad],lr=CFG['lr_'+regime],weight_decay=CFG['weight_decay'])
    run=wandb.init(project=CFG['project'],name=f'{task}-{regime}',group=DATA_HASH[:12],
                   mode=WANDB_MODE,dir=str(folder.resolve()),config=dict(CFG,task=task,regime=regime),reinit=True)
    best=-1;stale=0;history=[]
    for epoch in range(CFG['epochs']):
        warmup=max(0,int(CFG['warmup_epochs']))
        for group in optimizer.param_groups:
            group['lr']=CFG['lr_'+regime]*min(1.,(epoch+1)/max(1,warmup))
        model.train()
        if regime=='frozen':model.eval();model.classifier.train() # Freeze BatchNorm state too.
        total_loss=0;correct=0;total=0
        for images,labels,_ in loader(train,True):
            images,labels=images.to(DEVICE),labels.to(DEVICE)
            optimizer.zero_grad()
            logits=model(images)                     # forward
            loss=criterion(logits,labels)            # compare with true labels
            loss.backward()                         # gradients
            optimizer.step()                        # update weights
            total_loss+=loss.item()*len(labels)
            correct+=int((logits.argmax(1)==labels).sum());total+=len(labels)
        logits,y=predict(model,loader(val))
        m=classification_metrics(y,softmax(logits,axis=1),classes)
        val_loss=float(nn.functional.cross_entropy(torch.tensor(logits),torch.tensor(y)))
        row={'epoch':epoch+1,'learning_rate':optimizer.param_groups[0]['lr'],'train_loss':total_loss/total,'train_accuracy':correct/total,
             'val_loss':val_loss,'val_accuracy':m['accuracy'],'val_macro_f1':m['macro_f1_all_training_classes']}
        history.append(row);run.log(row);print(task,regime,row)
        if row['val_macro_f1']>best:
            best=row['val_macro_f1'];stale=0
            torch.save({'state_dict':model.state_dict(),'regime':regime,'task':task,'classes':classes,
              'image_size':CFG['image_size'],'epoch':epoch+1,'config':CFG,'context':context},folder/'best.pt')
        else:stale+=1
        if stale>=CFG['patience']:break
    pd.DataFrame(history).to_csv(folder/'history.csv',index=False)
    checkpoint=torch.load(folder/'best.pt',map_location=DEVICE,weights_only=False)
    model.load_state_dict(checkpoint['state_dict'])
    z,y=predict(model,loader(calibrate));temperature=fit_temperature(z,y)
    before=softmax(z,axis=1);after=softmax(z/temperature,axis=1)
    policy={'temperature':temperature,'calibration_split':'val_calibrate',
            'species_review_confidence':CFG['species_review_confidence']}
    if task=='triage':
        policy.update(choose_threshold(y,after[:,1],CFG['target_animal_recall']))
        sweep=[triage_metrics(y,after[:,1],t) for t in np.linspace(0,1,101)]
        pd.DataFrame(sweep).to_csv(folder/'validation_threshold_sweep.csv',index=False)
    write_json(folder/'policy.json',policy)
    result={'task':task,'regime':regime,'context':context,'selection_score':best,
      'best_epoch':checkpoint['epoch'],'classes':classes,'checkpoint_sha256':digest(folder/'best.pt'),
      'wandb_run_id':run.id,'wandb_url':run.url if WANDB_MODE=='online' else None,
      'wandb_mode':WANDB_MODE,'calibration_ece_before':ece(y,before),'calibration_ece_after':ece(y,after),
      'train_n':len(train),'val_select_n':len(val),'val_calibrate_n':len(calibrate)}
    run.summary.update({'best_val_macro_f1':best,'calibration_ece_before':ece(y,before),'calibration_ece_after':ece(y,after)})
    run.finish();write_json(folder/'summary.json',result)
    return result


## 6. تدريب فارغة/حيوان

In [ ]:
triage_runs=[train_model('triage',regime) for regime in REGIMES]
display(pd.DataFrame(triage_runs)[['task','regime','selection_score','best_epoch']])

## 7. تدريب الأنواع على القصاصات البشرية

In [ ]:
species_runs=[train_model('species',regime) for regime in REGIMES]
display(pd.DataFrame(species_runs)[['task','regime','selection_score','best_epoch']])

## 8. تثبيت الاختيار قبل Test

In [ ]:
selection={task:max([r for r in triage_runs+species_runs if r['task']==task],key=lambda r:r['selection_score'])['regime'] for task in ['triage','species']}
write_json(OUT/'selection.json',{'selected':selection,'criterion':'val_select macro-F1','dataset_hash':DATA_HASH})
print(selection)


## 9. التقييم النهائي وعرض النتائج

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

def evaluate_model(task,regime):
    folder=OUT/task/regime;summary=read_json(folder/'summary.json')
    signature={'data':DATA_HASH,'checkpoint':digest(folder/'best.pt'),'policy':digest(folder/'policy.json')}
    if (folder/'test_metrics.json').exists():
        saved=read_json(folder/'test_metrics.json');assert saved['signature']==signature
        return saved
    checkpoint=torch.load(folder/'best.pt',map_location=DEVICE,weights_only=False)
    assert checkpoint['context']==artifact_context(CFG,task)
    classes=checkpoint['classes'];policy=read_json(folder/'policy.json')
    model=make_model(regime,len(classes),pretrained=False,size=CFG['image_size']).to(DEVICE)
    model.load_state_dict(checkpoint['state_dict'])
    all_test=[r for r in samples_for_task(rows,task,classes) if r['split']=='test']
    known=[r for r in all_test if r['label']>=0];unknown=[r for r in all_test if r['label']<0]
    logits,y=predict(model,loader(known));raw=softmax(logits,axis=1);p=softmax(logits/policy['temperature'],axis=1)
    m=classification_metrics(y,p,classes)
    m.update(task=task,regime=regime,split='test',image_size=CFG['image_size'],policy=policy,
       signature=signature,context=checkpoint['context'],uncalibrated_ece=ece(y,raw),
       known_count=len(known),unknown_count=len(unknown),coverage=len(known)/len(all_test),
       unknown_species=sorted({r['species'] for r in unknown}),
       accuracy_counting_unknown_as_errors=float((p.argmax(1)==y).sum()/len(all_test)),
       accuracy_wilson95_independence_assumption=wilson(int((p.argmax(1)==y).sum()),len(y)),
       camera_bootstrap=camera_bootstrap(y,p,[r['camera'] for r in known],CFG['bootstrap_repeats'],CFG['seed'],
                                      policy['threshold'] if task=='triage' else None))
    if task=='triage':m['operating_point']=triage_metrics(y,p[:,1],policy['threshold'])
    m['per_camera']={camera:classification_metrics(y[idx],p[idx],classes)
      for camera in sorted({r['camera'] for r in known})
      for idx in [[i for i,r in enumerate(known) if r['camera']==camera]]}
    np.savez_compressed(folder/'test_probabilities.npz',y=y,probabilities=p,raw=raw)
    pred=p.argmax(1)
    pd.DataFrame([{'sample_id':r['sample_id'],'camera':r['camera'],'true':classes[y[i]],
      'prediction':classes[pred[i]],'confidence':float(p[i].max())} for i,r in enumerate(known)]).to_csv(folder/'test_predictions.csv',index=False)
    # Triage gallery uses the actual removal threshold, not argmax.
    decisions=(p[:,1]>=policy['threshold']).astype(int) if task=='triage' else pred
    errors=np.flatnonzero(decisions!=y)
    chosen=sorted(errors,key=lambda i:float(p[i].max()),reverse=True)[:12]
    if chosen:
        fig,axes=plt.subplots((len(chosen)+3)//4,4,figsize=(12,3*((len(chosen)+3)//4)))
        for ax in np.asarray(axes).flat:ax.axis('off')
        for ax,i in zip(np.asarray(axes).flat,chosen):
            rgb=cv2.cvtColor(load_rgb_sample(known[i]),cv2.COLOR_BGR2RGB)
            ax.imshow(rgb);ax.set_title(f"{known[i]['sample_id'][:12]}\n{classes[y[i]]} -> {classes[decisions[i]]}",fontsize=8)
        fig.suptitle(f'{task}/{regime}: test errors, input {CFG["image_size"]}, threshold '+str(policy.get('threshold','argmax')))
        fig.tight_layout();fig.savefig(folder/'private_errors.jpg');plt.close(fig)
    run=wandb.init(project=CFG['project'],name=f'{task}-{regime}-test',group=DATA_HASH[:12],
      mode=WANDB_MODE,dir=str(folder.resolve()),config=dict(CFG,task=task,regime=regime,split='locked_test'),reinit=True)
    run.log({'test/accuracy':m['accuracy'],'test/macro_f1':m['macro_f1_all_training_classes'],
             'test/ece':m['ece'],'test/top_k_accuracy':m['top_k_accuracy']})
    run.log({'test/per_class':wandb.Table(dataframe=pd.DataFrame(m['per_class']).T.reset_index())})
    m['wandb_test_run_id']=run.id;m['wandb_test_url']=run.url if WANDB_MODE=='online' else None
    run.finish();write_json(folder/'test_metrics.json',m)
    return m

def show_results(task,regime):
    folder=OUT/task/regime;m=read_json(folder/'test_metrics.json')
    display(pd.DataFrame(m['per_class']).T)
    print('Accuracy:',m['accuracy'],'Top-k:',m['top_k_accuracy'],'ECE:',m['ece'])
    print('Camera bootstrap:',m['camera_bootstrap'],'Unknown crops:',m['unknown_count'])
    h=pd.read_csv(folder/'history.csv');h.plot(x='epoch',y=['train_loss','val_loss'],title=f'{task}/{regime}: train/val_select loss, {CFG["image_size"]}px, threshold N/A');plt.tight_layout();plt.savefig(folder/'learning_loss.png');plt.show()
    h.plot(x='epoch',y=['train_accuracy','val_accuracy'],title=f'{task}/{regime}: train/val_select accuracy, {CFG["image_size"]}px, argmax');plt.tight_layout();plt.savefig(folder/'learning_accuracy.png');plt.show()
    classes=read_json(folder/'summary.json')['classes']
    ConfusionMatrixDisplay(np.array(m['confusion_matrix']),display_labels=classes).plot(xticks_rotation=90)
    plt.title(f'{task}/{regime}: test, {CFG["image_size"]}px, argmax');plt.tight_layout();plt.savefig(folder/'confusion.png');plt.show()
    z=np.load(folder/'test_probabilities.npz');y=z['y']
    plt.plot([0,1],[0,1],'--',color='gray')
    for key,label in [('raw','Before calibration'),('probabilities','After calibration')]:
        p=z[key];conf=p.max(1);correct=p.argmax(1)==y;bins=np.minimum((conf*10).astype(int),9)
        points=[(conf[bins==b].mean(),correct[bins==b].mean()) for b in range(10) if np.any(bins==b)]
        plt.plot(*zip(*points),'o-',label=label)
    plt.xlabel('Confidence');plt.ylabel('Accuracy');plt.legend();plt.title(f'{task}/{regime}: test reliability, {CFG["image_size"]}px, argmax');plt.tight_layout();plt.savefig(folder/'reliability.png');plt.show()
    if (folder/'private_errors.jpg').exists():display(Image.open(folder/'private_errors.jpg'))
    if task=='triage':print('Operating point:',m['operating_point'])


In [ ]:
for task in ['triage','species']:
    for regime in REGIMES:
        evaluate_model(task,regime)
        show_results(task,regime)

## 10. تحليل الأخطاء وتوثيق التجارب

In [ ]:
analysis=OUT/'ERROR_ANALYSIS.md'
if not analysis.exists():analysis.write_text('# Error analysis\n\nTODO: actual sample IDs, rare-class recall/support, false empty removals, calibration, plausible causes, and data improvement plan.\n')
SYNC_WANDB=False
if SYNC_WANDB:
    wandb.login()
    for p in OUT.glob('*/*/wandb/offline-run-*'):
        subprocess.run([sys.executable,'-m','wandb','sync',str(p)],check=True)


In [ ]:
required=[OUT/'config.json',OUT/'selection.json']
for task in ['triage','species']:
    for regime in REGIMES:
        required += [OUT/task/regime/n for n in ['best.pt','summary.json','history.csv','policy.json','test_metrics.json','test_probabilities.npz','test_predictions.csv']]
write_json('handoff/member2_release.json',{'schema':2,'dataset_hash':DATA_HASH,'output_dir':str(OUT),'member':2,'files':{str(p):digest(p) for p in required}})
print('Ready: member2_release.json')


# 3. التعلم بالنقل — M3

In [ ]:
from pathlib import Path
import os,json,hashlib,sys,subprocess,platform,random
import numpy as np,pandas as pd
from IPython.display import display,Markdown
ROOT=Path.cwd() if Path.cwd().name=='CameraTrap_Phase1_Team' else Path.cwd()/'CameraTrap_Phase1_Team'
ROOT.mkdir(exist_ok=True);os.chdir(ROOT)
for name in ['handoff','runs','review','submission','docs']:Path(name).mkdir(exist_ok=True)
def read_json(p):return json.loads(Path(p).read_text())
def write_json(p,v):
    Path(p).parent.mkdir(parents=True,exist_ok=True)
    Path(p).write_text(json.dumps(v,indent=2,ensure_ascii=False,allow_nan=False))
def digest(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
def environment(member):
    Path(f'handoff/requirements_member{member}.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
    write_json(f'handoff/runtime_member{member}.json',{'python':sys.version,'platform':platform.platform()})
print('Workspace:',ROOT)


In [ ]:
release=read_json('handoff/dataset_release.json')
assert release['schema']==2 and release['audit_complete'], 'استخدم إصدار Member 1 الجديد بعد اكتمال التدقيق'
for p,h in release['files'].items():assert Path(p).is_file() and digest(p)==h, f'Changed/missing: {p}'
DATA=Path(release['data_dir']);rows=read_json(DATA/'manifest.json')
assert rows and all(Path(r['path']).exists() for r in rows)
DATA_HASH=digest(DATA/'manifest.json')
CFG=read_json('handoff/data_config.json')
assert CFG['preprocessing_sha256']=='829fef7a32c1157ffe4da29da42026105afc0266ed647e562127bcc416697937'
print('Frames:',len(rows),'| split sizes:',pd.Series([r['split'] for r in rows]).value_counts().to_dict())


In [ ]:
import cv2, numpy as np, torch
from PIL import Image
from torchvision import transforms
from torch.utils.data import DataLoader
class Preprocessor:
    """Lab 1 cell 72: prepare uint8 RGB, augment, finalize and CHW tensor."""
    def __init__(self,size):
        self.size=size
        self.mean=np.array([.485,.456,.406],np.float32)
        self.std=np.array([.229,.224,.225],np.float32)
    def prepare(self,bgr):
        rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB);h,w=rgb.shape[:2]
        interp=cv2.INTER_AREA if h>self.size or w>self.size else cv2.INTER_LINEAR
        return cv2.resize(rgb,(self.size,self.size),interpolation=interp)
    def finalize(self,rgb):return (rgb.astype(np.float32)/255.-self.mean)/self.std

def load_rgb_sample(row):
    im=cv2.imread(row['path'])
    if im is None:raise ValueError('Unreadable image: '+row['path'])
    if row.get('crop') is not None:
        x1,y1,x2,y2=row['crop'];h,w=im.shape[:2];dx=(x2-x1)*.05;dy=(y2-y1)*.05
        left,top=max(0,int((x1-dx)*w)),max(0,int((y1-dy)*h))
        right,bottom=min(w,max(left+1,int((x2+dx)*w))),min(h,max(top+1,int((y2+dy)*h)))
        im=im[top:bottom,left:right].copy()
    return im

class FrameDataset(torch.utils.data.Dataset):
    """Lab 1 cell 104 adapted to frame and human-box crop tasks."""
    def __init__(self,samples,size,train=False):
        self.samples=samples;self.pre=Preprocessor(size)
        self.tf=transforms.Compose([transforms.RandomHorizontalFlip(),
                  transforms.ColorJitter(brightness=.15,contrast=.15)]) if train else None
    def __len__(self):return len(self.samples)
    def __getitem__(self,i):
        row=self.samples[i];img=self.pre.prepare(load_rgb_sample(row))
        if self.tf:img=np.array(self.tf(Image.fromarray(img)))
        x=torch.from_numpy(np.ascontiguousarray(self.pre.finalize(img))).permute(2,0,1).float()
        return x,int(row['label']),i

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    import torch
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def artifact_context(cfg, task):
    return dict(training_implementation=2, task=task, profile=cfg['profile'], image_size=cfg['image_size'],
                seed=cfg['seed'], manifest_sha256=digest(Path(cfg['data_dir'])/'manifest.json'),
                split_policy='camera-disjoint train / val_select / val_calibrate / test',
                preprocessing_sha256=cfg['preprocessing_sha256'],
                training_config={k:cfg[k] for k in ['optimizer','backbone','batch_size','epochs','patience','warmup_epochs',
                    'lr_scratch','lr_frozen','lr_full','weight_decay','label_smoothing','class_weighted_loss']})

def samples_for_task(rows,task,classes):
    samples=[]
    for r in rows:
        if task=='triage':
            samples.append(dict(r,label=r['animal'],sample_id=r['id'],crop=None))
        else:
            for i,b in enumerate(r['boxes']):
                samples.append(dict(r,label=classes.index(b['species']) if b['species'] in classes else -1,
                                    sample_id=f'{r["id"]}_{i}',crop=b['xyxy'],species=b['species']))
    return samples

## 2. إعدادات مشتركة وتجارب W&B

In [ ]:
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Device:',DEVICE)
if DEVICE=='cpu':print('CPU بطيء للتدريب الكامل؛ اختر GPU في Colab')
WANDB_MODE='offline'  # online لحساب W&B؛ لا تكتب المفتاح في الكود
import wandb
if WANDB_MODE=='online':wandb.login()
REGIMES=['frozen', 'full']
OUT=Path('runs/member3');OUT.mkdir(parents=True,exist_ok=True)
CFG['output_dir']=str(OUT)
assert CFG['optimizer']=='Adam'
write_json(OUT/'config.json',CFG)
environment(3)


## 3. النموذج وforward

In [ ]:
from torch import nn

class ConvNN(nn.Module):
    """Lab 2 cell 39 architecture; debug helpers removed, dimensions explicit."""
    def __init__(self,num_classes,size=224,num_filters=16,num_neurons=100):
        super().__init__()
        def block(a,b):return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU())
        self.block_1=block(3,num_filters)
        self.block_2=block(num_filters,2*num_filters)
        self.block_3=block(2*num_filters,4*num_filters)
        self.maxpool=nn.MaxPool2d(kernel_size=3,stride=2)
        self.dropout=nn.Dropout(.2);self.relu=nn.ReLU();self.flat=nn.Flatten()
        h=size
        for _ in range(3):h=(h-3)//2+1
        self.fc1=nn.Linear(4*num_filters*h*h,num_neurons)
        self.fc2=nn.Linear(num_neurons,num_neurons)
        self.output=nn.Linear(num_neurons,num_classes)
    def forward(self,x):
        x=self.maxpool(self.block_1(x));x=self.maxpool(self.block_2(x));x=self.maxpool(self.block_3(x))
        x=self.flat(self.dropout(x));x=self.relu(self.fc1(x));x=self.relu(self.fc2(x))
        return self.output(x)

class TransferNet(nn.Module):
    """Lab 3 cells 25/34: the same HF ResNet-50 and replaced classifier."""
    def __init__(self,classes,pretrained=True):
        super().__init__()
        from transformers import AutoModelForImageClassification,ResNetConfig,ResNetForImageClassification
        self.net=(AutoModelForImageClassification.from_pretrained('microsoft/resnet-50') if pretrained
                  else ResNetForImageClassification(ResNetConfig()))
        self.net.classifier=nn.Sequential(nn.Flatten(start_dim=1),nn.Linear(self.net.config.hidden_sizes[-1],classes))
        self.net.config.num_labels=classes
    def forward(self,x):return self.net(x).logits
    @property
    def classifier(self):return self.net.classifier

def make_model(regime,classes,pretrained=True,size=224):
    return ConvNN(classes,size) if regime=='scratch' else TransferNet(classes,pretrained)

def freeze_backbone(model,freeze):
    for p in model.net.base_model.parameters():p.requires_grad=not freeze
    for p in model.classifier.parameters():p.requires_grad=True

In [ ]:
classes=['empty','animal']
example_model=make_model(REGIMES[0],len(classes),size=CFG['image_size'])
if REGIMES[0]!='scratch':freeze_backbone(example_model,True)
print(example_model)
print('Trainable parameters:',sum(p.numel() for p in example_model.parameters() if p.requires_grad))
example_model.eval()
with torch.no_grad():print('Forward shape:',example_model(torch.zeros(2,3,CFG['image_size'],CFG['image_size'])).shape)
del example_model


## 4. المقاييس والمعايرة

In [ ]:
import numpy as np
from scipy.special import softmax, logsumexp
from scipy.optimize import minimize_scalar
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score


def ece(y, probabilities, bins=15):
    y=np.asarray(y);p=np.asarray(probabilities)
    if len(y)==0:return None
    conf=p.max(1);correct=p.argmax(1)==y
    indices=np.minimum((conf*bins).astype(int),bins-1)
    return float(sum(np.mean(indices==i)*abs(correct[indices==i].mean()-conf[indices==i].mean())
                     for i in range(bins) if np.any(indices==i)))


def fit_temperature(logits,y):
    logits=np.asarray(logits);y=np.asarray(y,dtype=int)
    if not len(y):raise ValueError('No known-class calibration examples')
    def nll(log_t):
        z=logits/np.exp(log_t)
        return float(np.mean(logsumexp(z,axis=1)-z[np.arange(len(y)),y]))
    result=minimize_scalar(nll,bounds=(np.log(0.25),np.log(10)),method='bounded')
    return float(np.exp(result.x))


def wilson(k,n,z=1.959963984540054):
    if n==0:return [None,None]
    p=k/n;d=1+z*z/n
    centre=(p+z*z/(2*n))/d
    half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return [float(max(0,centre-half)),float(min(1,centre+half))]


def triage_metrics(y, animal_scores, threshold):
    y=np.asarray(y);pred=np.asarray(animal_scores)>=threshold
    pos=y==1;neg=y==0
    tp=int(np.sum(pred & pos));fn=int(np.sum(~pred & pos));tn=int(np.sum(~pred & neg));fp=int(np.sum(pred & neg))
    return dict(threshold=float(threshold),animal_recall=float(tp/pos.sum()) if pos.any() else None,
                empty_removal_rate=float(tn/neg.sum()) if neg.any() else None,
                animal_frames_lost=fn,animal_frames=int(pos.sum()),empty_frames=int(neg.sum()),
                removed_frames=tn+fn,tp=tp,tn=tn,fp=fp,fn=fn,
                animal_recall_wilson95_independence_assumption=wilson(tp,int(pos.sum())))


def choose_threshold(y, scores, target=0.99):
    if set(np.asarray(y).tolist())!={0,1}:raise ValueError('Calibration needs empty and animal labels')
    candidates=np.unique(np.r_[0.,scores,1.])
    feasible=[triage_metrics(y,scores,t) for t in candidates]
    feasible=[r for r in feasible if r['animal_recall']>=target]
    best=max(feasible,key=lambda r:(r['empty_removal_rate'],r['threshold']))
    best['selection_split']='val_calibrate'
    best['recall_target_is_empirical_not_guaranteed']=True
    return best


def classification_metrics(y,probs,classes):
    y=np.asarray(y);probs=np.asarray(probs);pred=probs.argmax(1);n=len(classes)
    support=np.bincount(y,minlength=n);supported=np.flatnonzero(support)
    report=classification_report(y,pred,labels=list(range(n)),target_names=classes,output_dict=True,zero_division=0)
    k=min(3,n)
    return dict(n=len(y),accuracy=float(accuracy_score(y,pred)),
       macro_f1_all_training_classes=float(f1_score(y,pred,labels=list(range(n)),average='macro',zero_division=0)),
       macro_f1_supported_classes=float(f1_score(y,pred,labels=supported,average='macro',zero_division=0)),
       top_k=k,top_k_accuracy=float(np.mean([label in indices for label,indices in zip(y,np.argsort(probs,axis=1)[:,-k:])])),
       ece=ece(y,probs),per_class=report,confusion_matrix=confusion_matrix(y,pred,labels=list(range(n))).tolist(),
       classes_absent_from_this_split=[classes[i] for i in range(n) if support[i]==0])


def camera_bootstrap(y,probs,cameras,repeats=1000,seed=42,threshold=None):
    """Resample whole test cameras; preserve dependence among frames/crops within camera."""
    y=np.asarray(y);p=np.asarray(probs);c=np.asarray(cameras);sites=np.unique(c)
    if len(sites)<2:return {'status':'not_estimable_fewer_than_two_cameras'}
    rng=np.random.default_rng(seed);acc=[];f1=[];rec=[];empty=[]
    for _ in range(repeats):
        idx=np.concatenate([np.flatnonzero(c==site) for site in rng.choice(sites,len(sites),replace=True)])
        yy=y[idx];pp=p[idx]
        acc.append(float(np.mean(pp.argmax(1)==yy)))
        f1.append(float(f1_score(yy,pp.argmax(1),labels=range(p.shape[1]),average='macro',zero_division=0)))
        if threshold is not None:
            m=triage_metrics(yy,pp[:,1],threshold)
            if m['animal_recall'] is not None:rec.append(m['animal_recall'])
            if m['empty_removal_rate'] is not None:empty.append(m['empty_removal_rate'])
    ci=lambda v:np.quantile(v,[.025,.975]).tolist() if v else [None,None]
    out=dict(unit='camera',cameras=len(sites),repeats=repeats,accuracy95=ci(acc),macro_f1_95=ci(f1),
             caveat='Intervals are descriptive and unstable when there are few held-out cameras.')
    if threshold is not None:out.update(animal_recall95=ci(rec),empty_removal95=ci(empty))
    return out


## 5. حلقة التدريب — forward → loss → backward → step

In [ ]:
from scipy.special import softmax
import time,wandb

def loader(samples,train=False):
    gen=torch.Generator().manual_seed(CFG['seed'])
    return DataLoader(FrameDataset(samples,CFG['image_size'],train),batch_size=CFG['batch_size'],
                      shuffle=train,num_workers=0,generator=gen)

def predict(model,data):
    model.eval();all_logits=[];all_y=[]
    with torch.no_grad():
        for images,labels,_ in data:
            all_logits.append(model(images.to(DEVICE)).cpu().numpy());all_y.extend(labels.tolist())
    assert all_logits, 'No known-class samples in this split'
    return np.concatenate(all_logits),np.array(all_y,dtype=int)

def train_model(task,regime):
    seed_everything(CFG['seed'])
    classes=['empty','animal'] if task=='triage' else read_json(DATA/'species.json')
    samples=samples_for_task(rows,task,classes)
    train=[r for r in samples if r['split']=='train' and r['label']>=0]
    val=[r for r in samples if r['split']=='val_select' and r['label']>=0]
    calibrate=[r for r in samples if r['split']=='val_calibrate' and r['label']>=0]
    assert train and val and calibrate
    folder=OUT/task/regime;folder.mkdir(parents=True,exist_ok=True)
    context=artifact_context(CFG,task)
    if (folder/'summary.json').exists():
        saved=read_json(folder/'summary.json');assert saved['context']==context
        assert saved['checkpoint_sha256']==digest(folder/'best.pt')
        print('Reusing completed run:',task,regime);return saved
    assert not (folder/'best.pt').exists(), 'تدريب جزئي موجود؛ استخدم إصدار تجربة جديدًا لتجنب استبداله'
    model=make_model(regime,len(classes),size=CFG['image_size']).to(DEVICE)
    if regime!='scratch':freeze_backbone(model,regime=='frozen')
    assert CFG['optimizer']=='Adam', 'Only Adam is implemented'
    weights=None
    if CFG['class_weighted_loss']:
        counts=np.bincount([r['label'] for r in train],minlength=len(classes))
        assert (counts>0).all(), 'Every training class must have samples'
        weights=torch.tensor(len(train)/(len(classes)*counts),dtype=torch.float32,device=DEVICE)
    criterion=nn.CrossEntropyLoss(weight=weights,label_smoothing=CFG['label_smoothing'])
    optimizer=torch.optim.Adam([p for p in model.parameters() if p.requires_grad],lr=CFG['lr_'+regime],weight_decay=CFG['weight_decay'])
    run=wandb.init(project=CFG['project'],name=f'{task}-{regime}',group=DATA_HASH[:12],
                   mode=WANDB_MODE,dir=str(folder.resolve()),config=dict(CFG,task=task,regime=regime),reinit=True)
    best=-1;stale=0;history=[]
    for epoch in range(CFG['epochs']):
        warmup=max(0,int(CFG['warmup_epochs']))
        for group in optimizer.param_groups:
            group['lr']=CFG['lr_'+regime]*min(1.,(epoch+1)/max(1,warmup))
        model.train()
        if regime=='frozen':model.eval();model.classifier.train() # Freeze BatchNorm state too.
        total_loss=0;correct=0;total=0
        for images,labels,_ in loader(train,True):
            images,labels=images.to(DEVICE),labels.to(DEVICE)
            optimizer.zero_grad()
            logits=model(images)                     # forward
            loss=criterion(logits,labels)            # compare with true labels
            loss.backward()                         # gradients
            optimizer.step()                        # update weights
            total_loss+=loss.item()*len(labels)
            correct+=int((logits.argmax(1)==labels).sum());total+=len(labels)
        logits,y=predict(model,loader(val))
        m=classification_metrics(y,softmax(logits,axis=1),classes)
        val_loss=float(nn.functional.cross_entropy(torch.tensor(logits),torch.tensor(y)))
        row={'epoch':epoch+1,'learning_rate':optimizer.param_groups[0]['lr'],'train_loss':total_loss/total,'train_accuracy':correct/total,
             'val_loss':val_loss,'val_accuracy':m['accuracy'],'val_macro_f1':m['macro_f1_all_training_classes']}
        history.append(row);run.log(row);print(task,regime,row)
        if row['val_macro_f1']>best:
            best=row['val_macro_f1'];stale=0
            torch.save({'state_dict':model.state_dict(),'regime':regime,'task':task,'classes':classes,
              'image_size':CFG['image_size'],'epoch':epoch+1,'config':CFG,'context':context},folder/'best.pt')
        else:stale+=1
        if stale>=CFG['patience']:break
    pd.DataFrame(history).to_csv(folder/'history.csv',index=False)
    checkpoint=torch.load(folder/'best.pt',map_location=DEVICE,weights_only=False)
    model.load_state_dict(checkpoint['state_dict'])
    z,y=predict(model,loader(calibrate));temperature=fit_temperature(z,y)
    before=softmax(z,axis=1);after=softmax(z/temperature,axis=1)
    policy={'temperature':temperature,'calibration_split':'val_calibrate',
            'species_review_confidence':CFG['species_review_confidence']}
    if task=='triage':
        policy.update(choose_threshold(y,after[:,1],CFG['target_animal_recall']))
        sweep=[triage_metrics(y,after[:,1],t) for t in np.linspace(0,1,101)]
        pd.DataFrame(sweep).to_csv(folder/'validation_threshold_sweep.csv',index=False)
    write_json(folder/'policy.json',policy)
    result={'task':task,'regime':regime,'context':context,'selection_score':best,
      'best_epoch':checkpoint['epoch'],'classes':classes,'checkpoint_sha256':digest(folder/'best.pt'),
      'wandb_run_id':run.id,'wandb_url':run.url if WANDB_MODE=='online' else None,
      'wandb_mode':WANDB_MODE,'calibration_ece_before':ece(y,before),'calibration_ece_after':ece(y,after),
      'train_n':len(train),'val_select_n':len(val),'val_calibrate_n':len(calibrate)}
    run.summary.update({'best_val_macro_f1':best,'calibration_ece_before':ece(y,before),'calibration_ece_after':ece(y,after)})
    run.finish();write_json(folder/'summary.json',result)
    return result


## 6. تدريب فارغة/حيوان

In [ ]:
triage_runs=[train_model('triage',regime) for regime in REGIMES]
display(pd.DataFrame(triage_runs)[['task','regime','selection_score','best_epoch']])

## 7. تدريب الأنواع على القصاصات البشرية

In [ ]:
species_runs=[train_model('species',regime) for regime in REGIMES]
display(pd.DataFrame(species_runs)[['task','regime','selection_score','best_epoch']])

## 8. تثبيت الاختيار قبل Test

In [ ]:
selection={task:max([r for r in triage_runs+species_runs if r['task']==task],key=lambda r:r['selection_score'])['regime'] for task in ['triage','species']}
write_json(OUT/'selection.json',{'selected':selection,'criterion':'val_select macro-F1','dataset_hash':DATA_HASH})
print(selection)


## 9. التقييم النهائي وعرض النتائج

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

def evaluate_model(task,regime):
    folder=OUT/task/regime;summary=read_json(folder/'summary.json')
    signature={'data':DATA_HASH,'checkpoint':digest(folder/'best.pt'),'policy':digest(folder/'policy.json')}
    if (folder/'test_metrics.json').exists():
        saved=read_json(folder/'test_metrics.json');assert saved['signature']==signature
        return saved
    checkpoint=torch.load(folder/'best.pt',map_location=DEVICE,weights_only=False)
    assert checkpoint['context']==artifact_context(CFG,task)
    classes=checkpoint['classes'];policy=read_json(folder/'policy.json')
    model=make_model(regime,len(classes),pretrained=False,size=CFG['image_size']).to(DEVICE)
    model.load_state_dict(checkpoint['state_dict'])
    all_test=[r for r in samples_for_task(rows,task,classes) if r['split']=='test']
    known=[r for r in all_test if r['label']>=0];unknown=[r for r in all_test if r['label']<0]
    logits,y=predict(model,loader(known));raw=softmax(logits,axis=1);p=softmax(logits/policy['temperature'],axis=1)
    m=classification_metrics(y,p,classes)
    m.update(task=task,regime=regime,split='test',image_size=CFG['image_size'],policy=policy,
       signature=signature,context=checkpoint['context'],uncalibrated_ece=ece(y,raw),
       known_count=len(known),unknown_count=len(unknown),coverage=len(known)/len(all_test),
       unknown_species=sorted({r['species'] for r in unknown}),
       accuracy_counting_unknown_as_errors=float((p.argmax(1)==y).sum()/len(all_test)),
       accuracy_wilson95_independence_assumption=wilson(int((p.argmax(1)==y).sum()),len(y)),
       camera_bootstrap=camera_bootstrap(y,p,[r['camera'] for r in known],CFG['bootstrap_repeats'],CFG['seed'],
                                      policy['threshold'] if task=='triage' else None))
    if task=='triage':m['operating_point']=triage_metrics(y,p[:,1],policy['threshold'])
    m['per_camera']={camera:classification_metrics(y[idx],p[idx],classes)
      for camera in sorted({r['camera'] for r in known})
      for idx in [[i for i,r in enumerate(known) if r['camera']==camera]]}
    np.savez_compressed(folder/'test_probabilities.npz',y=y,probabilities=p,raw=raw)
    pred=p.argmax(1)
    pd.DataFrame([{'sample_id':r['sample_id'],'camera':r['camera'],'true':classes[y[i]],
      'prediction':classes[pred[i]],'confidence':float(p[i].max())} for i,r in enumerate(known)]).to_csv(folder/'test_predictions.csv',index=False)
    # Triage gallery uses the actual removal threshold, not argmax.
    decisions=(p[:,1]>=policy['threshold']).astype(int) if task=='triage' else pred
    errors=np.flatnonzero(decisions!=y)
    chosen=sorted(errors,key=lambda i:float(p[i].max()),reverse=True)[:12]
    if chosen:
        fig,axes=plt.subplots((len(chosen)+3)//4,4,figsize=(12,3*((len(chosen)+3)//4)))
        for ax in np.asarray(axes).flat:ax.axis('off')
        for ax,i in zip(np.asarray(axes).flat,chosen):
            rgb=cv2.cvtColor(load_rgb_sample(known[i]),cv2.COLOR_BGR2RGB)
            ax.imshow(rgb);ax.set_title(f"{known[i]['sample_id'][:12]}\n{classes[y[i]]} -> {classes[decisions[i]]}",fontsize=8)
        fig.suptitle(f'{task}/{regime}: test errors, input {CFG["image_size"]}, threshold '+str(policy.get('threshold','argmax')))
        fig.tight_layout();fig.savefig(folder/'private_errors.jpg');plt.close(fig)
    run=wandb.init(project=CFG['project'],name=f'{task}-{regime}-test',group=DATA_HASH[:12],
      mode=WANDB_MODE,dir=str(folder.resolve()),config=dict(CFG,task=task,regime=regime,split='locked_test'),reinit=True)
    run.log({'test/accuracy':m['accuracy'],'test/macro_f1':m['macro_f1_all_training_classes'],
             'test/ece':m['ece'],'test/top_k_accuracy':m['top_k_accuracy']})
    run.log({'test/per_class':wandb.Table(dataframe=pd.DataFrame(m['per_class']).T.reset_index())})
    m['wandb_test_run_id']=run.id;m['wandb_test_url']=run.url if WANDB_MODE=='online' else None
    run.finish();write_json(folder/'test_metrics.json',m)
    return m

def show_results(task,regime):
    folder=OUT/task/regime;m=read_json(folder/'test_metrics.json')
    display(pd.DataFrame(m['per_class']).T)
    print('Accuracy:',m['accuracy'],'Top-k:',m['top_k_accuracy'],'ECE:',m['ece'])
    print('Camera bootstrap:',m['camera_bootstrap'],'Unknown crops:',m['unknown_count'])
    h=pd.read_csv(folder/'history.csv');h.plot(x='epoch',y=['train_loss','val_loss'],title=f'{task}/{regime}: train/val_select loss, {CFG["image_size"]}px, threshold N/A');plt.tight_layout();plt.savefig(folder/'learning_loss.png');plt.show()
    h.plot(x='epoch',y=['train_accuracy','val_accuracy'],title=f'{task}/{regime}: train/val_select accuracy, {CFG["image_size"]}px, argmax');plt.tight_layout();plt.savefig(folder/'learning_accuracy.png');plt.show()
    classes=read_json(folder/'summary.json')['classes']
    ConfusionMatrixDisplay(np.array(m['confusion_matrix']),display_labels=classes).plot(xticks_rotation=90)
    plt.title(f'{task}/{regime}: test, {CFG["image_size"]}px, argmax');plt.tight_layout();plt.savefig(folder/'confusion.png');plt.show()
    z=np.load(folder/'test_probabilities.npz');y=z['y']
    plt.plot([0,1],[0,1],'--',color='gray')
    for key,label in [('raw','Before calibration'),('probabilities','After calibration')]:
        p=z[key];conf=p.max(1);correct=p.argmax(1)==y;bins=np.minimum((conf*10).astype(int),9)
        points=[(conf[bins==b].mean(),correct[bins==b].mean()) for b in range(10) if np.any(bins==b)]
        plt.plot(*zip(*points),'o-',label=label)
    plt.xlabel('Confidence');plt.ylabel('Accuracy');plt.legend();plt.title(f'{task}/{regime}: test reliability, {CFG["image_size"]}px, argmax');plt.tight_layout();plt.savefig(folder/'reliability.png');plt.show()
    if (folder/'private_errors.jpg').exists():display(Image.open(folder/'private_errors.jpg'))
    if task=='triage':print('Operating point:',m['operating_point'])


In [ ]:
for task in ['triage','species']:
    for regime in REGIMES:
        evaluate_model(task,regime)
        show_results(task,regime)

## 10. تحليل الأخطاء وتوثيق التجارب

In [ ]:
analysis=OUT/'ERROR_ANALYSIS.md'
if not analysis.exists():analysis.write_text('# Error analysis\n\nTODO: actual sample IDs, rare-class recall/support, false empty removals, calibration, plausible causes, and data improvement plan.\n')
SYNC_WANDB=False
if SYNC_WANDB:
    wandb.login()
    for p in OUT.glob('*/*/wandb/offline-run-*'):
        subprocess.run([sys.executable,'-m','wandb','sync',str(p)],check=True)


In [ ]:
required=[OUT/'config.json',OUT/'selection.json']
for task in ['triage','species']:
    for regime in REGIMES:
        required += [OUT/task/regime/n for n in ['best.pt','summary.json','history.csv','policy.json','test_metrics.json','test_probabilities.npz','test_predictions.csv']]
write_json('handoff/member3_release.json',{'schema':2,'dataset_hash':DATA_HASH,'output_dir':str(OUT),'member':3,'files':{str(p):digest(p) for p in required}})
print('Ready: member3_release.json')


# 4. المقارنة والتحقق والتقرير

In [ ]:
from pathlib import Path
import os,json,hashlib,sys,subprocess,platform,random
import numpy as np,pandas as pd
from IPython.display import display,Markdown
ROOT=Path.cwd() if Path.cwd().name=='CameraTrap_Phase1_Team' else Path.cwd()/'CameraTrap_Phase1_Team'
ROOT.mkdir(exist_ok=True);os.chdir(ROOT)
for name in ['handoff','runs','review','submission','docs']:Path(name).mkdir(exist_ok=True)
def read_json(p):return json.loads(Path(p).read_text())
def write_json(p,v):
    Path(p).parent.mkdir(parents=True,exist_ok=True)
    Path(p).write_text(json.dumps(v,indent=2,ensure_ascii=False,allow_nan=False))
def digest(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
def environment(member):
    Path(f'handoff/requirements_member{member}.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
    write_json(f'handoff/runtime_member{member}.json',{'python':sys.version,'platform':platform.platform()})
print('Workspace:',ROOT)


In [ ]:
release=read_json('handoff/dataset_release.json')
assert release['schema']==2 and release['audit_complete'], 'استخدم إصدار Member 1 الجديد بعد اكتمال التدقيق'
for p,h in release['files'].items():assert Path(p).is_file() and digest(p)==h, f'Changed/missing: {p}'
DATA=Path(release['data_dir']);rows=read_json(DATA/'manifest.json')
assert rows and all(Path(r['path']).exists() for r in rows)
DATA_HASH=digest(DATA/'manifest.json')
CFG=read_json('handoff/data_config.json')
assert CFG['preprocessing_sha256']=='829fef7a32c1157ffe4da29da42026105afc0266ed647e562127bcc416697937'
print('Frames:',len(rows),'| split sizes:',pd.Series([r['split'] for r in rows]).value_counts().to_dict())


In [ ]:
environment(4)

## 2. فحص مستقل ومقارنة التجارب الست

In [ ]:
from collections import defaultdict
from sklearn.metrics import accuracy_score,f1_score
checks=[]
def check(name,passed,details=''):
    checks.append({'check':name,'passed':bool(passed),'details':str(details)})
for key in ['id','camera','event','pixel_sha256']:
    groups=defaultdict(set)
    for r in rows:groups[r[key]].add(r['split'])
    bad=[k for k,v in groups.items() if len(v)>1]
    check(key+'_disjoint',not bad,len(bad))
check('unique_ids',len(rows)==len({r['id'] for r in rows}))
changed=[]
for r in rows:
    if hashlib.md5(Path(r['path']).read_bytes()).hexdigest()!=r['processed_file_md5']:changed.append(r['id'])
check('image_integrity',not changed,len(changed))
audit=read_json(DATA/'audit_report.json')
check('human_audit',audit['status']=='complete' and audit['unresolved_flagged_events']==0)
for split in ['train','val_select','val_calibrate','test']:
    check('empty_animal_support_'+split,{r['animal'] for r in rows if r['split']==split}=={0,1})
records=[];contexts=[]
for member in [2,3]:
    rel=read_json(f'handoff/member{member}_release.json')
    assert rel['schema']==2 and rel['dataset_hash']==DATA_HASH
    for p,h in rel['files'].items():assert Path(p).is_file() and digest(p)==h, f'Changed run artifact: {p}'
    out=Path(rel['output_dir'])
    for task in ['triage','species']:
        for regime in (['scratch'] if member==2 else ['frozen','full']):
            folder=out/task/regime;s=read_json(folder/'summary.json');m=read_json(folder/'test_metrics.json')
            z=np.load(folder/'test_probabilities.npz');y=z['y'];p=z['probabilities'];pred=p.argmax(1)
            confidence=p.max(1);correct=pred==y;bins=np.minimum((confidence*15).astype(int),14)
            ece_value=sum(np.mean(bins==i)*abs(correct[bins==i].mean()-confidence[bins==i].mean()) for i in range(15) if np.any(bins==i))
            f1=f1_score(y,pred,labels=list(range(p.shape[1])),average='macro',zero_division=0)
            check(f'metrics_{task}_{regime}',np.isclose(m['accuracy'],accuracy_score(y,pred)) and
                  np.isclose(m['macro_f1_all_training_classes'],f1) and np.isclose(m['ece'],ece_value))
            check(f'provenance_{task}_{regime}',m['context']==s['context'] and
                  m['signature']=={'data':DATA_HASH,'checkpoint':digest(folder/'best.pt'),'policy':digest(folder/'policy.json')})
            contexts.append({k:v for k,v in s['context'].items() if k!='task'})
            records.append({'member':member,'task':task,'regime':regime,'folder':str(folder),
              'val_macro_f1':s['selection_score'],'test_accuracy':m['accuracy'],
              'test_macro_f1':m['macro_f1_all_training_classes'],'top_k':m['top_k'],
              'top_k_accuracy':m['top_k_accuracy'],'ece':m['ece'],'raw_ece':m['uncalibrated_ece'],
              'coverage':m['coverage'],'unknown_crops':m['unknown_count'],'input_size':m['image_size'],
              'threshold':m['policy'].get('threshold','argmax'),
              'animal_recall':m.get('operating_point',{}).get('animal_recall'),
              'empty_removal':m.get('operating_point',{}).get('empty_removal_rate'),
              'accuracy_ci_camera':str(m['camera_bootstrap'].get('accuracy95')),
              'train_run_id':s['wandb_run_id'],'train_url':s.get('wandb_url') or '',
              'test_run_id':m['wandb_test_run_id'],'test_url':m.get('wandb_test_url') or ''})
check('common_training_configuration',all(c==contexts[0] for c in contexts))
comparison=pd.DataFrame(records)
comparison['selected_on_validation']=False
for task in ['triage','species']:
    idx=comparison.loc[comparison.task==task,'val_macro_f1'].idxmax()
    comparison.loc[idx,'selected_on_validation']=True
# Acceptance targets were fixed in Member1 before training; a failed target is a result, not a reason to retune on test.
acceptance=[]
for row in comparison[comparison.selected_on_validation].to_dict('records'):
    targets=([('animal_recall','animal_recall_min','min'),('empty_removal','empty_removal_min','min')]
             if row['task']=='triage' else [('test_macro_f1','species_macro_f1_min','min'),('ece','species_ece_max','max')])
    for metric,key,direction in targets:
        value=row[metric];target=CFG['acceptance'][key]
        acceptance.append({'task':row['task'],'regime':row['regime'],'metric':metric,'value':value,'target':target,
                           'direction':direction,'met':bool(value>=target if direction=='min' else value<=target)})
pd.DataFrame(acceptance).to_csv('submission/acceptance.csv',index=False)
display(pd.DataFrame(acceptance))
comparison.to_csv('submission/comparison.csv',index=False)
pd.DataFrame(checks).to_csv('review/automatic_checks.csv',index=False)
display(pd.DataFrame(checks));display(comparison)
print('اختيار الأفضل مبني على val_macro_f1 فقط، وليس test.')


## 3. عرض المقارنة وأدلة الأخطاء

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
for task in ['triage','species']:
    table=comparison[comparison.task==task]
    table.plot.bar(x='regime',y=['test_accuracy','test_macro_f1','ece'],title=f'{task}: held-out test, {CFG["image_size"]}px, argmax')
    plt.tight_layout();plt.savefig('submission/'+task+'_comparison.png');plt.show()
for row in comparison[comparison.selected_on_validation].to_dict('records'):
    folder=Path(row['folder']);m=read_json(folder/'test_metrics.json')
    print(row['task'],row['regime'],'| Split:',m['split'],'Input:',m['image_size'],'Policy:',m['policy'])
    display(pd.DataFrame(m['per_class']).T)
    if (folder/'private_errors.jpg').exists():display(Image.open(folder/'private_errors.jpg'))


## 4. عينة مراجعة مستقلة وسجل المشاكل

In [ ]:
audit_path=Path('review/independent_audit.csv')
if not audit_path.exists():
    events={}
    for r in rows:
        if r['split']=='train':events.setdefault(r['event'],[]).append(r)
    rng=random.Random(2026)
    sample=[rng.choice(events[e]) for e in rng.sample(sorted(events),min(20,len(events)))]
    pd.DataFrame([{'id':r['id'],'label_correct':'','boxes_correct':'','person_visible':'','reviewer':'','notes':''} for r in sample]).to_csv(audit_path,index=False)
by_id={r['id']:r for r in rows}
from PIL import ImageDraw
for ident in pd.read_csv(audit_path).id:
    r=by_id[ident];im=Image.open(r['path']).convert('RGB');draw=ImageDraw.Draw(im)
    for b in r['boxes']:
        x1,y1,x2,y2=b['xyxy'];draw.rectangle((x1*im.width,y1*im.height,x2*im.width,y2*im.height),outline='lime',width=3)
    im.thumbnail((650,420));print(ident,r['labels'] or ['empty']);display(im)
issues=Path('review/issues.csv')
if not issues.exists():pd.DataFrame(columns=['issue_id','owner','problem','status','resolution','evidence']).to_csv(issues,index=False)
print('CSV decisions: label_correct=yes/no, boxes_correct=yes/no/not applicable, person_visible=yes/no.')


## 5. فحص إعادة تحميل الأوزان بعملية Python جديدة

In [ ]:
PROBE_SOURCE = 'import cv2, numpy as np, torch\nfrom PIL import Image\nfrom torchvision import transforms\nfrom torch.utils.data import DataLoader\nclass Preprocessor:\n    """Lab 1 cell 72: prepare uint8 RGB, augment, finalize and CHW tensor."""\n    def __init__(self,size):\n        self.size=size\n        self.mean=np.array([.485,.456,.406],np.float32)\n        self.std=np.array([.229,.224,.225],np.float32)\n    def prepare(self,bgr):\n        rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB);h,w=rgb.shape[:2]\n        interp=cv2.INTER_AREA if h>self.size or w>self.size else cv2.INTER_LINEAR\n        return cv2.resize(rgb,(self.size,self.size),interpolation=interp)\n    def finalize(self,rgb):return (rgb.astype(np.float32)/255.-self.mean)/self.std\n\ndef load_rgb_sample(row):\n    im=cv2.imread(row[\'path\'])\n    if im is None:raise ValueError(\'Unreadable image: \'+row[\'path\'])\n    if row.get(\'crop\') is not None:\n        x1,y1,x2,y2=row[\'crop\'];h,w=im.shape[:2];dx=(x2-x1)*.05;dy=(y2-y1)*.05\n        left,top=max(0,int((x1-dx)*w)),max(0,int((y1-dy)*h))\n        right,bottom=min(w,max(left+1,int((x2+dx)*w))),min(h,max(top+1,int((y2+dy)*h)))\n        im=im[top:bottom,left:right].copy()\n    return im\n\nclass FrameDataset(torch.utils.data.Dataset):\n    """Lab 1 cell 104 adapted to frame and human-box crop tasks."""\n    def __init__(self,samples,size,train=False):\n        self.samples=samples;self.pre=Preprocessor(size)\n        self.tf=transforms.Compose([transforms.RandomHorizontalFlip(),\n                  transforms.ColorJitter(brightness=.15,contrast=.15)]) if train else None\n    def __len__(self):return len(self.samples)\n    def __getitem__(self,i):\n        row=self.samples[i];img=self.pre.prepare(load_rgb_sample(row))\n        if self.tf:img=np.array(self.tf(Image.fromarray(img)))\n        x=torch.from_numpy(np.ascontiguousarray(self.pre.finalize(img))).permute(2,0,1).float()\n        return x,int(row[\'label\']),i\nfrom torch import nn\n\nclass ConvNN(nn.Module):\n    """Lab 2 cell 39 architecture; debug helpers removed, dimensions explicit."""\n    def __init__(self,num_classes,size=224,num_filters=16,num_neurons=100):\n        super().__init__()\n        def block(a,b):return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU())\n        self.block_1=block(3,num_filters)\n        self.block_2=block(num_filters,2*num_filters)\n        self.block_3=block(2*num_filters,4*num_filters)\n        self.maxpool=nn.MaxPool2d(kernel_size=3,stride=2)\n        self.dropout=nn.Dropout(.2);self.relu=nn.ReLU();self.flat=nn.Flatten()\n        h=size\n        for _ in range(3):h=(h-3)//2+1\n        self.fc1=nn.Linear(4*num_filters*h*h,num_neurons)\n        self.fc2=nn.Linear(num_neurons,num_neurons)\n        self.output=nn.Linear(num_neurons,num_classes)\n    def forward(self,x):\n        x=self.maxpool(self.block_1(x));x=self.maxpool(self.block_2(x));x=self.maxpool(self.block_3(x))\n        x=self.flat(self.dropout(x));x=self.relu(self.fc1(x));x=self.relu(self.fc2(x))\n        return self.output(x)\n\nclass TransferNet(nn.Module):\n    """Lab 3 cells 25/34: the same HF ResNet-50 and replaced classifier."""\n    def __init__(self,classes,pretrained=True):\n        super().__init__()\n        from transformers import AutoModelForImageClassification,ResNetConfig,ResNetForImageClassification\n        self.net=(AutoModelForImageClassification.from_pretrained(\'microsoft/resnet-50\') if pretrained\n                  else ResNetForImageClassification(ResNetConfig()))\n        self.net.classifier=nn.Sequential(nn.Flatten(start_dim=1),nn.Linear(self.net.config.hidden_sizes[-1],classes))\n        self.net.config.num_labels=classes\n    def forward(self,x):return self.net(x).logits\n    @property\n    def classifier(self):return self.net.classifier\n\ndef make_model(regime,classes,pretrained=True,size=224):\n    return ConvNN(classes,size) if regime==\'scratch\' else TransferNet(classes,pretrained)\n\ndef freeze_backbone(model,freeze):\n    for p in model.net.base_model.parameters():p.requires_grad=not freeze\n    for p in model.classifier.parameters():p.requires_grad=True\nif __name__==\'__main__\':\n    import sys,json\n    checkpoint,image=sys.argv[1:3]\n    torch.set_num_threads(2)\n    ckpt=torch.load(checkpoint,map_location=\'cpu\',weights_only=False)\n    model=make_model(ckpt[\'regime\'],len(ckpt[\'classes\']),pretrained=False,size=ckpt[\'image_size\']).eval()\n    model.load_state_dict(ckpt[\'state_dict\'])\n    x,_,_=FrameDataset([{\'path\':image,\'label\':0,\'crop\':None}],ckpt[\'image_size\'])[0]\n    with torch.no_grad():y=model(x.unsqueeze(0))\n    assert y.shape==(1,len(ckpt[\'classes\'])) and torch.isfinite(y).all()\n    print(json.dumps({\'shape\':list(y.shape),\'finite\':True}))\n'
fresh=[]
sample=next(r['path'] for r in rows if r['split']=='train')
for row in comparison.to_dict('records'):
    checkpoint=Path(row['folder'])/'best.pt'
    try:
        proc=subprocess.run([sys.executable,'-c',PROBE_SOURCE,str(checkpoint),sample],capture_output=True,text=True,timeout=180)
        fresh.append({'checkpoint':str(checkpoint),'sha256':digest(checkpoint),'passed':proc.returncode==0,
                      'stdout':proc.stdout[-1000:],'stderr':proc.stderr[-2000:]})
    except subprocess.TimeoutExpired:
        fresh.append({'checkpoint':str(checkpoint),'sha256':digest(checkpoint),'passed':False,'stderr':'timeout'})
write_json('review/fresh_process_results.json',fresh);display(pd.DataFrame(fresh)[['checkpoint','passed']])


## 6. روابط W&B ومساهمات الفريق

In [ ]:
links_path=Path('submission/wandb_links.csv')
if not links_path.exists():comparison[['task','regime','train_run_id','train_url','test_run_id','test_url']].to_csv(links_path,index=False)
TEAM={'member1':'','member2':'','member3':'','member4':''}
NOTES={'motivation':'','data_findings':'','model_comparison':'','error_analysis':'','data_improvement_plan':'','limitations':'','conclusion':''}
REPLAY={'reviewer':'','date':'','notebook':'','cells_executed':'','evidence':''}
REVIEWER=''
VISUAL_REVIEW_COMPLETE=False
REPOSITORY_URL='https://github.com/salehdev3/Capstone_5'
write_json('submission/report_inputs.json',{'team':TEAM,'notes':NOTES,'replay':REPLAY,'repository':REPOSITORY_URL})


## 7. قائمة اكتمال المتطلبات

In [ ]:
gaps=[]
automatic=pd.read_csv('review/automatic_checks.csv')
if not automatic.passed.all():gaps.append('Automatic integrity/metrics checks failed')
review=pd.read_csv('review/independent_audit.csv').fillna('')
valid=review.label_correct.eq('yes') & review.boxes_correct.isin(['yes','not applicable']) & review.person_visible.eq('no') & review.reviewer.str.strip().ne('')
if not valid.all():gaps.append('Independent audit pending or flagged')
issues=pd.read_csv('review/issues.csv').fillna('')
if not issues.empty and not (issues.status.eq('resolved') & issues.resolution.str.strip().ne('') & issues.evidence.str.strip().ne('')).all():gaps.append('Unresolved issues')
if len(fresh)!=6 or not all(r['passed'] for r in fresh):gaps.append('Fresh-process inference')
if not REVIEWER.strip() or not VISUAL_REVIEW_COMPLETE:gaps.append('Named visual review signoff')
for k,v in TEAM.items():
    if not v.strip():gaps.append('Team name: '+k)
for k,v in NOTES.items():
    if not v.strip():gaps.append('Report section: '+k)
if not all(v.strip() for v in REPLAY.values()):gaps.append('Clean notebook replay evidence')
for member in [2,3]:
    p=Path(f'runs/member{member}/ERROR_ANALYSIS.md')
    if not p.exists() or 'TODO' in p.read_text():gaps.append(f'Actual error analysis member {member}')
card=Path(release['dataset_card']).read_text()
if 'Pending:' in card:gaps.append('Dataset Card team observations')
blur=pd.read_csv(DATA/'blur_review.csv').fillna('')
if not (blur.reviewer.str.strip().ne('') & blur.decision.isin(['keep','flag']) & blur.notes.str.strip().ne('')).all():gaps.append('Blur review notes')
links=pd.read_csv(links_path).fillna('')
expected={(t,r) for t in ['triage','species'] for r in ['scratch','frozen','full']}
if len(links)!=6 or set(zip(links.task,links.regime))!=expected:gaps.append('Six W&B experiment rows')
for field in ['train_url','test_url']:
    if not links[field].str.match(r'^https://wandb\.ai/[^/]+/[^/]+/runs/[^/]+').all():gaps.append('W&B '+field)
for i in range(1,5):
    if not Path(f'handoff/requirements_member{i}.txt').exists():gaps.append(f'Environment member {i}')
NAMES=['Phase1_Final_Presentation.ipynb']
for name in NAMES:
    if not (Path('notebooks')/name).exists():gaps.append('Notebook '+name)
if not REPOSITORY_URL.startswith('https://github.com/'):gaps.append('GitHub repository URL')
STATUS='READY_FOR_TEAM_REVIEW' if not gaps else 'INCOMPLETE'
print(STATUS)
for gap in gaps:print('-',gap)
write_json('handoff/qa_release.json',{'schema':2,'dataset_hash':DATA_HASH,'status':STATUS,'gaps':gaps,
 'reviewer':REVIEWER,'input_hashes':{f'member{i}':digest(f'handoff/member{i}_release.json') for i in [2,3]},
 'scope':'data/metrics checks and fresh-process inference; full replay scope documented separately'})


## 8. التقرير من نتائج التشغيل الفعلية

In [ ]:
def table(df):
    df=df.fillna('');clean=lambda v:str(v).replace('|','/').replace('\n',' ')
    return '\n'.join(['| '+' | '.join(df.columns)+' |','| '+' | '.join(['---']*len(df.columns))+' |']+
                     ['| '+' | '.join(clean(v) for v in row)+' |' for row in df.itertuples(index=False,name=None)])
sections=['# Project 5 — Phase 1 report','Status: '+STATUS,
 'Scope: M1–M3 only. Frame empty/animal classification and species human-box crop classification.',
 '## Contributions',str(TEAM),'## Motivation',NOTES['motivation'] or '[pending]',
 '## Dataset Card',Path(release['dataset_card']).read_text(),NOTES['data_findings'] or '[pending]',
 '## Acceptance criteria',table(pd.DataFrame(acceptance)),
 '## Model comparison',table(comparison.drop(columns=['folder','train_url','test_url'])),
 'Selections above use validation macro-F1 only. Test is held out. Per-class reports and confidence intervals are included in results/.',
 NOTES['model_comparison'] or '[pending]','## Error analysis',NOTES['error_analysis'] or '[pending]']
for member in [2,3]:
    p=Path(f'runs/member{member}/ERROR_ANALYSIS.md')
    sections.append(p.read_text() if p.exists() else '[missing analysis]')
sections+=['## Data improvement plan',NOTES['data_improvement_plan'] or '[pending]',
 '## Limitations',NOTES['limitations'] or '[pending]',
 'The species task uses human-box crops. This does not evaluate a detector-classifier cascade. Camera-based confidence intervals use few sites and are descriptive.',
 '## Reproducibility',str(REPLAY),'Repository: '+(REPOSITORY_URL or '[pending]'),
 '## W&B',table(links),'## Conclusion',NOTES['conclusion'] or '[pending]',
 '## Outstanding evidence','\n'.join('- '+g for g in gaps) or 'No automated completeness gaps.']
Path('submission/REPORT_DRAFT.md').write_text('\n\n'.join(sections))
display(Markdown(Path('submission/REPORT_DRAFT.md').read_text()))


## 9. تجهيز المستودع دون الصور أو الأوزان

In [ ]:
import shutil
repo=Path('submission/repository');repo.mkdir(exist_ok=True)
def collect(source,destination):
    source=Path(source)
    if source.exists():
        target=repo/destination;target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(source,target)
for name in NAMES:
    p=Path('notebooks')/name
    if p.exists():
        n=read_json(p)
        for cell in n['cells']:
            if cell['cell_type']=='code':cell['outputs']=[];cell['execution_count']=None
        write_json(repo/'notebooks'/name,n)
for p in Path('handoff').glob('requirements_member*.txt'):collect(p,Path('environments')/p.name)
for p in Path('handoff').glob('runtime_member*.json'):collect(p,Path('environments')/p.name)
collect('handoff/data_config.json','config.json')
collect(release['dataset_card'],'DATASET_CARD.md')
for name in ['comparison.csv','acceptance.csv','REPORT_DRAFT.md','wandb_links.csv','report_inputs.json','triage_comparison.png','species_comparison.png']:
    collect(Path('submission')/name,name)
for name in ['automatic_checks.csv','independent_audit.csv','issues.csv','fresh_process_results.json']:
    collect(Path('review')/name,Path('review')/name)
for name in ['class_balance_proportions.csv','species_crop_support.csv','audit_report.json','blur_review.csv']:
    collect(DATA/name,Path('data_evidence')/name)
weights=[]
for row in comparison.to_dict('records'):
    folder=Path(row['folder']);destination=Path('results')/row['task']/row['regime']
    for name in ['summary.json','test_metrics.json','history.csv','policy.json','confusion.png','learning_loss.png','learning_accuracy.png','reliability.png','validation_threshold_sweep.csv']:
        collect(folder/name,destination/name)
    weights.append({'path':str(folder/'best.pt'),'sha256':digest(folder/'best.pt'),'access':'private team storage'})
write_json(repo/'weights_manifest.json',weights)
(repo/'.gitignore').write_text('data/\nruns/\nwandb/\n*.pt\n*.pth\n*.jpg\n*.jpeg\n.env\n__pycache__/\n.ipynb_checkpoints/\n')
(repo/'README.md').write_text("# Project 5 — Phase 1\n\nRun Phase1_Final_Presentation.ipynb in section order. Use the same Member1 dataset release for both training notebooks. Upload private result archives to Colab and set ARCHIVES in the import cell; no Drive mount or external code modules are required. Internet and Colab's torch/torchvision are required; use GPU for training.\n\nMember1 uses CCT20 from https://lila.science/datasets/caltech-camera-traps. Follow dataset licence and course non-redistribution policy. Do not commit images or raw archives. The Dataset Card records filtering and limitations.\n\nAll three notebooks embed identical OpenCV preprocessing. Split is stratified by camera groups; bursts stay together. One shared config defines all six experiments. Validation selects models; calibration chooses temperature/triage threshold; test is evaluated once.\n\nSee environments/ for the actual resolved package versions and Python/platform of each member. Install one role's environment in a compatible runtime. No single cross-platform lock is claimed. The initial install cells describe required packages; actual resolved pins are captured after installation.\n\nReport and metrics are from actual execution, with pending sections visible. Test accuracy intervals include camera bootstrap and Wilson (the latter assumes independent samples). Human audit, error discussion, clean replay and W&B sync require team action. Fresh-process inference alone is not full training reproduction.\n\nWeights remain private; weights_manifest.json provides hashes and paths. Repository notebooks have outputs stripped; keep executed copies privately for assessment. This repository covers M1–M3 only, without detector training or deployment.\n")
assert not list(repo.rglob('*.pt')) and not list(repo.rglob('*.jpg'))
write_json('handoff/integration_release.json',{'schema':2,'scope':'Phase 1','status':STATUS,'gaps':gaps,
 'dataset_hash':DATA_HASH,'report_sha256':digest(repo/'REPORT_DRAFT.md'),'repository':REPOSITORY_URL})
archive=shutil.make_archive(str(ROOT.parent/'Phase1_Submission'),'zip',root_dir=repo)
print('Prepared:',repo.resolve(),'|',archive)
print('Submission archive:', archive)
